# PlannerLLM implementation and experiment execution

Open this notebook with the submission folder as the working directory. Each experiment cell displays the actual source files and calls the repository's experiment functions. The original report runs are in `runs/`. The completed notebook runs are in `reproduced_runs/recorded_groq/`.

The saved cells were executed in two stages. The first used recorded Groq responses, and the second fetched missing responses through Groq using the same model IDs and recorded those responses in `data/cache/extended_groq.sqlite`. The cache was supplemented for the memory, cost-model and ablation experiments and the seed 2 independent judge. Original benchmark artifacts and the original response cache remain available for comparison.

The saved outputs document the original execution and its supplemented replay. On restart, the default automatic mode checks the bundled cache for each exact request. If a Groq key is available in a local `.env` or the environment, cache misses go to Groq automatically. Without a key, a cache miss is logged as incomplete. Timing and retraining can change optimizer decisions. New runs write to timestamped folders under `notebook_reruns/`. Automatic runs with a key use a private copy of the bundled cache in their timestamped folder. Set `PLANNER_NOTEBOOK_MODE=live` only if you want to start with an empty cache.

In [2]:
from pathlib import Path
import os
import sys
import json
import shutil
from collections import Counter
from IPython.display import Code, display

ROOT = Path.cwd()
assert (ROOT / "plannerllm").is_dir(), "Open the notebook in the submission folder"
RESPONSE_MODE = os.getenv("PLANNER_NOTEBOOK_MODE", "auto")
SEEDS = (0, 1, 2)
PROFILE = "quick"
WORKLOADS = ["msmarco", "cuad", "spider"]
os.environ["PLANNER_DATA_DIR"] = str(ROOT / "data")
from datetime import datetime

if RESPONSE_MODE == "complete_cache":
    run_dir = ROOT / "reproduced_runs" / "recorded_groq"
else:
    run_dir = ROOT / "notebook_reruns" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
os.environ["PLANNER_RUNS_DIR"] = str(run_dir)
sys.path.insert(0, str(ROOT))

os.environ["HF_HOME"] = str(ROOT / "model_cache")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ.setdefault("OMP_NUM_THREADS", "2")

import torch
torch.set_num_threads(2)
import faiss
faiss.omp_set_num_threads(2)
from plannerllm.settings import Settings
from plannerllm.env import load_env
from plannerllm.llm.client import GroqClient
from plannerllm.eval import experiments as E
from plannerllm.eval.aggregate import build
from plannerllm.eval.oracle import Oracle
from plannerllm.eval import metrics
from plannerllm.engine.sqlrun import results_match

class MissingRecordedResponse(RuntimeError):
    pass

load_env()
has_groq_key = any(
    key.strip() and key.strip() not in {"recorded-mode-unused", "your_own_key_here", "gsk_..."}
    for name in ("GROQ_API_KEY", "GROQ_API_KEYS")
    for key in os.getenv(name, "").split(",")
)
allow_live = RESPONSE_MODE in ("live", "complete_cache") or (RESPONSE_MODE == "auto" and has_groq_key)

if RESPONSE_MODE not in ("auto", "recorded_groq", "live", "complete_cache"):
    raise ValueError(RESPONSE_MODE)
if allow_live and not has_groq_key:
    raise RuntimeError("Set GROQ_API_KEY in the local .env or environment")

cache_path = ROOT / "data/cache/extended_groq.sqlite"
if RESPONSE_MODE == "auto" and allow_live:
    run_dir.mkdir(parents=True, exist_ok=True)
    run_cache = run_dir / "groq_responses.sqlite"
    shutil.copy2(cache_path, run_cache)
    cache_path = run_cache
elif RESPONSE_MODE == "live":
    cache_path = run_dir / "groq_responses.sqlite"

if not allow_live:
    def require_recorded_response(self, prompt, system, max_tokens, json_mode, task, payload):
        from plannerllm.llm.cache import ResponseCache
        request_hash = ResponseCache.make_key(self.model, system, prompt, max_tokens, json_mode, self.temperature)
        raise MissingRecordedResponse(f"Missing original Groq response: {self.model}, {task}, {request_hash}")
    GroqClient._raw = require_recorded_response
    os.environ["GROQ_API_KEY"] = "recorded-mode-unused"

def settings_for(seed):
    settings = Settings.load(ROOT / "configs/default.yaml")
    settings.llm.backend = "groq"
    settings.llm.cache_path = str(cache_path)
    settings.sampling.task_seed = seed
    settings.sampling.seed = 13 + seed
    return settings

print("Response mode:", RESPONSE_MODE, "| seeds:", SEEDS, "| datasets:", WORKLOADS)
print("Groq on cache miss:", "enabled" if allow_live else "disabled (no live requests)")
print("Generator:", settings_for(0).llm.model, "| independent judge:", settings_for(0).llm.judge_model)

ModuleNotFoundError: No module named 'torch'

In [2]:
def show_implementation(snapshots):
    for path, source in snapshots.items():
        assert (ROOT / "plannerllm" / path).read_text(encoding="utf-8") == source, path
        print("plannerllm/" + path)
        display(Code(source, language="python"))

def run_selected(names):
    for name in names:
        for seed in SEEDS:
            settings = settings_for(seed)
            expected = E.out_dir(settings) / f"{name}.json"
            if RESPONSE_MODE == "complete_cache" and expected.exists():
                print(f"{name} | seed {seed} | already executed | {expected.relative_to(ROOT)}")
                continue
            log = []
            try:
                path = E.run_experiment(name, settings, PROFILE, WORKLOADS, progress=log.append)
            except MissingRecordedResponse as exc:
                path = E.out_dir(settings) / f"{name}.incomplete.log"
                path.write_text("\n".join(log + [str(exc)]) + "\n", encoding="utf-8")
                print(f"{name} | seed {seed} | INCOMPLETE | {exc}")
                continue
            path.with_suffix(".log").write_text("\n".join(log) + "\n", encoding="utf-8")
            result = json.loads(path.read_text(encoding="utf-8"))
            print(f"{name} | seed {seed} | {len(log)} progress entries | {path.relative_to(ROOT)}")
            for message in log[-3:]:
                print("  ", message)
            if name == "baselines":
                for row in result["summary"]:
                    print(row["workload"], row["system"], round(row["accuracy"], 4), "opt tokens:", round(row["opt_tokens"]))


## Pipeline execution and scoring

In [3]:
# Source snapshot. The experiment below executes the imported copy.
show_implementation({
    'core/pipeline.py': (
        '"""Declarative pipeline model (Section 5.1).\n'
        '\n'
        'A pipeline is a DAG of typed operators written in a DocETL-like YAML.  Every\n'
        'operator names its inputs; when ``inputs`` is omitted the previous operator\n'
        'is used, so the common retrieve -> rerank -> generate chain stays short.\n'
        '\n'
        'Operator kinds\n'
        '--------------\n'
        'retrieve       ANN probe (``top_k``, ``index`` hnsw|ivf|flat, optional\n'
        '               ``where`` => filtered vector probe)\n'
        'filter         SQL predicate over corpus metadata, evaluated in DuckDB\n'
        'rerank         cross-encoder rescoring, keeps ``top_k``\n'
        'map            LLM call per record (extract / summarise), drops "NONE"\n'
        'generate       one LLM call producing the final answer from the records\n'
        'sql_generate   text-to-SQL over the schema in context (``fewshot_k``)\n'
        'sql_repair     execute, and on error/empty result ask the LLM to fix\n'
        'sql_execute    run the SQL and attach result rows\n'
        '"""\n'
        'from __future__ import annotations\n'
        '\n'
        'import copy\n'
        'import hashlib\n'
        'import json\n'
        'from dataclasses import dataclass, field\n'
        'from typing import Any\n'
        '\n'
        'import yaml\n'
        '\n'
        'KINDS = {"retrieve", "filter", "rerank", "map", "generate",\n'
        '         "sql_generate", "sql_repair", "sql_execute"}\n'
        'LLM_KINDS = {"map", "generate", "sql_generate", "sql_repair"}\n'
        'CHEAP_KINDS = KINDS - LLM_KINDS\n'
        '\n'
        '\n'
        'class PipelineError(ValueError):\n'
        '    pass\n'
        '\n'
        '\n'
        '@dataclass\n'
        'class Operator:\n'
        '    name: str\n'
        '    kind: str\n'
        '    params: dict[str, Any] = field(default_factory=dict)\n'
        '    prompt: str = ""\n'
        '    inputs: list[str] = field(default_factory=list)\n'
        '\n'
        '    def to_dict(self) -> dict:\n'
        '        d: dict[str, Any] = {"name": self.name, "kind": self.kind}\n'
        '        if self.params:\n'
        '            d["params"] = copy.deepcopy(self.params)\n'
        '        if self.prompt:\n'
        '            d["prompt"] = self.prompt\n'
        '        if self.inputs:\n'
        '            d["inputs"] = list(self.inputs)\n'
        '        return d\n'
        '\n'
        '    @classmethod\n'
        '    def from_dict(cls, d: dict) -> "Operator":\n'
        '        if d.get("kind") not in KINDS:\n'
        '            raise PipelineError(f"operator \'{d.get(\'name\')}\' has unknown kind \'{d.get(\'kind\')}\'")\n'
        '        return cls(name=str(d["name"]), kind=d["kind"], params=dict(d.get("params") or {}),\n'
        '                   prompt=str(d.get("prompt") or ""), inputs=list(d.get("inputs") or []))\n'
        '\n'
        '    @property\n'
        '    def uses_llm(self) -> bool:\n'
        '        return self.kind in LLM_KINDS\n'
        '\n'
        '\n'
        '@dataclass\n'
        'class Pipeline:\n'
        '    name: str\n'
        '    workload: str\n'
        '    operators: list[Operator] = field(default_factory=list)\n'
        '    lineage: list[str] = field(default_factory=list)   # directives applied so far\n'
        '\n'
        '    # ---------------------------------------------------------- structure\n'
        '    def op(self, name: str) -> Operator:\n'
        '        for o in self.operators:\n'
        '            if o.name == name:\n'
        '                return o\n'
        '        raise KeyError(name)\n'
        '\n'
        '    def find(self, kind: str) -> Operator | None:\n'
        '        return next((o for o in self.operators if o.kind == kind), None)\n'
        '\n'
        '    def index_of(self, name: str) -> int:\n'
        '        return next(i for i, o in enumerate(self.operators) if o.name == name)\n'
        '\n'
        '    def kinds(self) -> list[str]:\n'
        '        return [o.kind for o in self.topological()]\n'
        '\n'
        '    def resolved_inputs(self, op: Operator) -> list[str]:\n'
        '        if op.inputs:\n'
        '            return op.inputs\n'
        '        i = self.index_of(op.name)\n'
        '        return [self.operators[i - 1].name] if i > 0 else []\n'
        '\n'
        '    def topological(self) -> list[Operator]:\n'
        '        """Kahn\'s algorithm; raises on cycles or dangling inputs."""\n'
        '        names = {o.name for o in self.operators}\n'
        '        deps = {o.name: set(self.resolved_inputs(o)) for o in self.operators}\n'
        '        for n, ds in deps.items():\n'
        '            missing = ds - names\n'
        '            if missing:\n'
        '                raise PipelineError(f"operator \'{n}\' reads unknown input(s) {sorted(missing)}")\n'
        '        order, ready = [], [o.name for o in self.operators if not deps[o.name]]\n'
        '        while ready:\n'
        '            n = ready.pop(0)\n'
        '            order.append(n)\n'
        '            for m, ds in deps.items():\n'
        '                if n in ds:\n'
        '                    ds.discard(n)\n'
        '                    if not ds and m not in order and m not in ready:\n'
        '                        ready.append(m)\n'
        '        if len(order) != len(self.operators):\n'
        '            raise PipelineError("pipeline graph has a cycle")\n'
        '        by_name = {o.name: o for o in self.operators}\n'
        '        return [by_name[n] for n in order]\n'
        '\n'
        '    def validate(self) -> None:\n'
        '        seen = set()\n'
        '        for o in self.operators:\n'
        '            if o.name in seen:\n'
        '                raise PipelineError(f"duplicate operator name \'{o.name}\'")\n'
        '            seen.add(o.name)\n'
        '        self.topological()\n'
        '        terminal = self.topological()[-1].kind\n'
        '        if terminal not in {"generate", "sql_execute", "map"}:\n'
        '            raise PipelineError(f"pipeline must end in an answer-producing operator, got \'{terminal}\'")\n'
        '\n'
        '    # ---------------------------------------------------------- editing\n'
        '    def clone(self) -> "Pipeline":\n'
        '        return Pipeline(self.name, self.workload,\n'
        '                        [Operator.from_dict(o.to_dict()) for o in self.operators],\n'
        '                        list(self.lineage))\n'
        '\n'
        '    def insert_after(self, anchor: str, op: Operator) -> None:\n'
        '        """Insert ``op`` downstream of ``anchor`` and re-point anchor\'s readers."""\n'
        '        i = self.index_of(anchor)\n'
        '        repointed = False\n'
        '        for other in self.operators[i + 1:]:\n'
        '            if anchor in other.inputs:\n'
        '                other.inputs = [op.name if x == anchor else x for x in other.inputs]\n'
        '                repointed = True\n'
        '        if repointed:\n'
        '            op.inputs = [anchor]\n'
        '        self.operators.insert(i + 1, op)\n'
        '\n'
        '    def insert_before(self, anchor: str, op: Operator) -> None:\n'
        '        i = self.index_of(anchor)\n'
        '        anchor_op = self.operators[i]\n'
        '        if anchor_op.inputs:\n'
        '            op.inputs = list(anchor_op.inputs)\n'
        '            anchor_op.inputs = [op.name]\n'
        '        self.operators.insert(i, op)\n'
        '\n'
        '    def remove(self, name: str) -> None:\n'
        '        i = self.index_of(name)\n'
        '        gone = self.operators.pop(i)\n'
        '        upstream = gone.inputs or ([self.operators[i - 1].name] if i > 0 else [])\n'
        '        for other in self.operators:\n'
        '            if name in other.inputs:\n'
        '                new_inputs: list[str] = []\n'
        '                for x in other.inputs:\n'
        '                    new_inputs.extend(upstream if x == name else [x])\n'
        '                other.inputs = new_inputs\n'
        '\n'
        '    def unique_name(self, base: str) -> str:\n'
        '        names = {o.name for o in self.operators}\n'
        '        if base not in names:\n'
        '            return base\n'
        '        k = 2\n'
        '        while f"{base}_{k}" in names:\n'
        '            k += 1\n'
        '        return f"{base}_{k}"\n'
        '\n'
        '    # ---------------------------------------------------------- identity\n'
        '    def structural_dict(self) -> dict:\n'
        '        return {"ops": [o.to_dict() for o in self.topological()]}\n'
        '\n'
        '    def fingerprint(self) -> str:\n'
        '        blob = json.dumps(self.structural_dict(), sort_keys=True)\n'
        '        return hashlib.sha1(blob.encode()).hexdigest()[:12]\n'
        '\n'
        '    def describe(self) -> str:\n'
        '        parts = []\n'
        '        for o in self.topological():\n'
        '            p = o.params\n'
        '            if o.kind == "retrieve":\n'
        '                extra = f"k={p.get(\'top_k\')},{p.get(\'index\', \'hnsw\')}" + (",where" if p.get("where") else "")\n'
        '            elif o.kind == "rerank":\n'
        '                extra = f"k={p.get(\'top_k\')}"\n'
        '            elif o.kind == "sql_generate":\n'
        '                extra = f"shots={p.get(\'fewshot_k\', 0)}"\n'
        '            else:\n'
        '                extra = ""\n'
        '            parts.append(f"{o.kind}({extra})" if extra else o.kind)\n'
        '        return " -> ".join(parts)\n'
        '\n'
        '    # ---------------------------------------------------------- (de)serialise\n'
        '    def to_dict(self) -> dict:\n'
        '        return {"pipeline": self.name, "workload": self.workload,\n'
        '                "lineage": list(self.lineage),\n'
        '                "operators": [o.to_dict() for o in self.operators]}\n'
        '\n'
        '    def to_yaml(self) -> str:\n'
        '        return yaml.safe_dump(self.to_dict(), sort_keys=False, width=100)\n'
        '\n'
        '    @classmethod\n'
        '    def from_dict(cls, d: dict) -> "Pipeline":\n'
        '        p = cls(name=d.get("pipeline", "pipeline"), workload=d.get("workload", ""),\n'
        '                operators=[Operator.from_dict(o) for o in d.get("operators", [])],\n'
        '                lineage=list(d.get("lineage") or []))\n'
        '        p.validate()\n'
        '        return p\n'
        '\n'
        '    @classmethod\n'
        '    def from_yaml(cls, text: str) -> "Pipeline":\n'
        '        return cls.from_dict(yaml.safe_load(text))\n'
    ),
    'engine/executor.py': (
        '"""Execution engine: runs one pipeline for one query and records telemetry.\n'
        '\n'
        'Telemetry is split the way Section 10.6 decomposes wall-clock time:\n'
        'DB operator time (DuckDB / SQLite), retrieval time (ANN probe), and LLM time,\n'
        'plus the tokens and dollars each operator consumed.  The same numbers feed\n'
        'the observed-cost log that the neural residual is trained on.\n'
        '"""\n'
        'from __future__ import annotations\n'
        '\n'
        'import time\n'
        'from dataclasses import asdict, dataclass, field\n'
        'from typing import Any\n'
        '\n'
        'from ..core.pipeline import Operator, Pipeline\n'
        'from . import prompts\n'
        'from .sqlrun import clean_sql, schema_text\n'
        'from .store import render_where\n'
        '\n'
        'MAX_DOC_CHARS = 600\n'
        '\n'
        '\n'
        '@dataclass\n'
        'class OpTelemetry:\n'
        '    op: str\n'
        '    kind: str\n'
        '    rows_in: int\n'
        '    rows_out: int\n'
        '    ms: float\n'
        '    db_ms: float = 0.0\n'
        '    ann_ms: float = 0.0\n'
        '    llm_ms: float = 0.0\n'
        '    tokens: int = 0\n'
        '    usd: float = 0.0\n'
        '    llm_wall_ms: float = 0.0\n'
        '\n'
        '\n'
        '@dataclass\n'
        'class ExecResult:\n'
        '    qid: str\n'
        '    answer: str\n'
        '    context: list[str] = field(default_factory=list)\n'
        '    sql: str | None = None\n'
        '    rows: list | None = None\n'
        '    error: str | None = None\n'
        '    retrieved_ids: list[str] = field(default_factory=list)\n'
        '    telemetry: list[OpTelemetry] = field(default_factory=list)\n'
        '\n'
        '    # aggregate views ---------------------------------------------------\n'
        '    @property\n'
        '    def tokens(self) -> int:\n'
        '        return sum(t.tokens for t in self.telemetry)\n'
        '\n'
        '    @property\n'
        '    def llm_usd(self) -> float:\n'
        '        return sum(t.usd for t in self.telemetry)\n'
        '\n'
        '    @property\n'
        '    def latency_ms(self) -> float:\n'
        '        return sum(t.ms for t in self.telemetry)\n'
        '\n'
        '    def breakdown(self) -> dict:\n'
        '        return {"db_ms": sum(t.db_ms for t in self.telemetry),\n'
        '                "ann_ms": sum(t.ann_ms for t in self.telemetry),\n'
        '                "llm_ms": sum(t.llm_ms for t in self.telemetry),\n'
        '                "other_ms": sum(max(0.0, t.ms - t.db_ms - t.ann_ms - t.llm_ms) for t in self.telemetry)}\n'
        '\n'
        '    def to_dict(self) -> dict:\n'
        '        d = asdict(self)\n'
        '        d["tokens"], d["latency_ms"] = self.tokens, self.latency_ms\n'
        '        d["rows"] = [list(map(str, r)) for r in (self.rows or [])[:10]]\n'
        '        return d\n'
        '\n'
        '\n'
        'class _State:\n'
        '    """Data flowing between operators for one query."""\n'
        '\n'
        '    def __init__(self) -> None:\n'
        '        self.positions: list[int] | None = None   # None == whole corpus\n'
        '        self.extracts: dict[int, str] = {}\n'
        '        self.answer: str = ""\n'
        '        self.sql: str | None = None\n'
        '        self.rows: list | None = None\n'
        '        self.error: str | None = None\n'
        '        self.context: list[str] = []\n'
        '\n'
        '\n'
        'class Executor:\n'
        '    def __init__(self, workload, llm, cpu_usd_per_hour: float = 0.04) -> None:\n'
        '        self.wl = workload\n'
        '        self.llm = llm\n'
        '        self.cpu_usd_per_ms = cpu_usd_per_hour / 3.6e6\n'
        '\n'
        '    # ------------------------------------------------------------------\n'
        '    def run(self, plan: Pipeline, query: dict, purpose: str = "execute") -> ExecResult:\n'
        '        st = _State()\n'
        '        res = ExecResult(qid=query["qid"], answer="")\n'
        '        for op in plan.topological():\n'
        '            t0 = time.perf_counter()\n'
        '            rows_in = len(st.positions) if st.positions is not None else self.wl.store.n_rows\n'
        '            tel = OpTelemetry(op=op.name, kind=op.kind, rows_in=rows_in, rows_out=0, ms=0.0)\n'
        '            try:\n'
        '                getattr(self, f"_op_{op.kind}")(op, query, st, tel, purpose)\n'
        '            except KeyError as exc:  # missing query field for a predicate\n'
        '                st.error = f"{op.name}: {exc}"\n'
        "            # wall time of the operator with the LLM's wall time (which on a cache hit is ~0 and on a\n"
        "            # fresh call includes rate-limit waits) replaced by the API's service time\n"
        '            tel.ms = (time.perf_counter() - t0) * 1000 - tel.llm_wall_ms + tel.llm_ms\n'
        '            tel.rows_out = len(st.positions) if st.positions is not None else tel.rows_in\n'
        '            res.telemetry.append(tel)\n'
        '        res.answer, res.sql, res.rows, res.error = st.answer, st.sql, st.rows, st.error\n'
        '        res.context = st.context\n'
        '        if st.positions is not None:\n'
        '            res.retrieved_ids = [self.wl.records[p]["id"] for p in st.positions]\n'
        '        return res\n'
        '\n'
        '    def observed_usd(self, res: ExecResult) -> float:\n'
        '        cpu_ms = sum(t.db_ms + t.ann_ms for t in res.telemetry)\n'
        '        return res.llm_usd + cpu_ms * self.cpu_usd_per_ms\n'
        '\n'
        '    # ------------------------------------------------------------ helpers\n'
        '    def _llm(self, tel: OpTelemetry, prompt: str, *, purpose: str, task: str,\n'
        '             payload: dict, max_tokens: int, system: str = prompts.SYSTEM_EXEC) -> str:\n'
        '        t0 = time.perf_counter()\n'
        '        r = self.llm.complete(prompt, purpose=purpose, task=task, payload=payload,\n'
        '                              system=system, max_tokens=max_tokens)\n'
        '        tel.llm_wall_ms += (time.perf_counter() - t0) * 1000\n'
        '        tel.llm_ms += r.latency_s * 1000\n'
        '        tel.tokens += r.total_tokens\n'
        '        tel.usd += self.llm.ledger.usd_for(r.prompt_tokens, r.completion_tokens)\n'
        '        return r.text.strip()\n'
        '\n'
        '    def _texts(self, st: _State, limit: int) -> list[str]:\n'
        '        pos = st.positions if st.positions is not None else []\n'
        '        out = []\n'
        '        for p in pos[:limit]:\n'
        '            out.append(st.extracts.get(p) or self.wl.records[p]["text"][:MAX_DOC_CHARS])\n'
        '        return out\n'
        '\n'
        '    # ------------------------------------------------------------ operators\n'
        '    def _op_filter(self, op: Operator, q: dict, st: _State, tel: OpTelemetry, purpose: str) -> None:\n'
        '        where = render_where(op.params["where"], q.get("fields", {}))\n'
        '        keep, ms = self.wl.store.positions(where, subset=st.positions)\n'
        '        tel.db_ms += ms\n'
        '        if st.positions is None:\n'
        '            st.positions = keep\n'
        '        else:\n'
        '            allowed = set(keep)\n'
        '            st.positions = [p for p in st.positions if p in allowed]\n'
        '\n'
        '    def _op_retrieve(self, op: Operator, q: dict, st: _State, tel: OpTelemetry, purpose: str) -> None:\n'
        '        p = op.params\n'
        '        allowed = st.positions\n'
        '        if p.get("where"):\n'
        '            where = render_where(p["where"], q.get("fields", {}))\n'
        '            allowed, ms = self.wl.store.positions(where, subset=allowed)\n'
        '            tel.db_ms += ms\n'
        '        qvec = self.wl.embedder.encode_query(q["question"])\n'
        '        pos, _, ms = self.wl.index.search(qvec, int(p.get("top_k", 10)), p.get("index", "hnsw"),\n'
        '                                          allowed=allowed, ef_search=int(p.get("ef_search", 64)),\n'
        '                                          nprobe=int(p.get("nprobe", 8)))\n'
        '        tel.ann_ms += ms\n'
        '        st.positions = pos\n'
        '\n'
        '    def _op_rerank(self, op: Operator, q: dict, st: _State, tel: OpTelemetry, purpose: str) -> None:\n'
        '        pos = st.positions or []\n'
        '        if not pos:\n'
        '            return\n'
        '        t0 = time.perf_counter()\n'
        '        scores = self.wl.reranker.score(q["question"], [self.wl.records[i]["text"] for i in pos])\n'
        '        tel.ann_ms += (time.perf_counter() - t0) * 1000   # local model time, CPU\n'
        '        order = sorted(range(len(pos)), key=lambda i: -scores[i])\n'
        '        st.positions = [pos[i] for i in order[: int(op.params.get("top_k", 5))]]\n'
        '\n'
        '    def _op_map(self, op: Operator, q: dict, st: _State, tel: OpTelemetry, purpose: str) -> None:\n'
        '        kept = []\n'
        '        cap = op.params.get("max_records")\n'
        '        for i in (st.positions or [])[: int(cap) if cap else None]:\n'
        '            text = self.wl.records[i]["text"][:1200]\n'
        '            prompt = prompts.MAP.format(instruction=op.prompt, question=q["question"], text=text)\n'
        '            out = self._llm(tel, prompt, purpose=purpose, task="extract", max_tokens=160,\n'
        '                            payload={"question": q["question"], "instruction": op.prompt, "text": text})\n'
        '            if out and not out.strip().upper().startswith("NONE"):\n'
        '                st.extracts[i] = out[:MAX_DOC_CHARS]\n'
        '                kept.append(i)\n'
        '        st.positions = kept\n'
        '\n'
        '    def _op_generate(self, op: Operator, q: dict, st: _State, tel: OpTelemetry, purpose: str) -> None:\n'
        '        docs = self._texts(st, int(op.params.get("max_docs", 6)))\n'
        '        st.context = docs\n'
        '        ctx = "\\n\\n".join(f"[{k + 1}] {d}" for k, d in enumerate(docs)) or "(no context retrieved)"\n'
        '        prompt = prompts.GENERATE.format(instruction=op.prompt, question=q["question"], context=ctx)\n'
        '        st.answer = self._llm(tel, prompt, purpose=purpose, task="answer", max_tokens=160,\n'
        '                              payload={"question": q["question"], "context": docs})\n'
        '\n'
        '    def _schema_for(self, q: dict, st: _State) -> dict[str, list[str]]:\n'
        '        full = self.wl.sql.schema(q["db_id"])\n'
        '        if st.positions is None:\n'
        '            return full\n'
        '        tables = [self.wl.records[p].get("table_name") for p in st.positions]\n'
        '        picked = {t: full[t] for t in tables if t in full}\n'
        '        return picked or full\n'
        '\n'
        '    def _op_sql_generate(self, op: Operator, q: dict, st: _State, tel: OpTelemetry, purpose: str) -> None:\n'
        '        schema = self._schema_for(q, st)\n'
        '        st.context = [schema_text(schema)]\n'
        '        shots = ""\n'
        '        k = int(op.params.get("fewshot_k", 0))\n'
        '        if k and self.wl.examples is not None:\n'
        '            ex = self.wl.nearest_examples(q["question"], k)\n'
        '            shots = "\\nExamples:\\n" + "\\n".join(f"Q: {e[\'text\']}\\nSQL: {e[\'sql\']}" for e in ex) + "\\n"\n'
        '        prompt = prompts.SQL_GENERATE.format(instruction=op.prompt, schema=schema_text(schema),\n'
        '                                             examples=shots, question=q["question"])\n'
        '        out = self._llm(tel, prompt, purpose=purpose, task="sql", max_tokens=220,\n'
        '                        payload={"question": q["question"], "schema": schema})\n'
        '        st.sql = clean_sql(out)\n'
        '\n'
        '    def _op_sql_repair(self, op: Operator, q: dict, st: _State, tel: OpTelemetry, purpose: str) -> None:\n'
        '        rows, err, ms = self.wl.sql.run(q["db_id"], st.sql or "")\n'
        '        tel.db_ms += ms\n'
        '        if err is None and rows:\n'
        '            return\n'
        '        schema = self._schema_for(q, st)\n'
        '        prompt = prompts.SQL_REPAIR.format(schema=schema_text(schema), question=q["question"],\n'
        '                                           sql=st.sql or "", problem=err or "query returned no rows")\n'
        '        out = self._llm(tel, prompt, purpose=purpose, task="sql_repair", max_tokens=220,\n'
        '                        payload={"question": q["question"], "schema": schema})\n'
        '        fixed = clean_sql(out)\n'
        '        if fixed:\n'
        '            st.sql = fixed\n'
        '\n'
        '    def _op_sql_execute(self, op: Operator, q: dict, st: _State, tel: OpTelemetry, purpose: str) -> None:\n'
        '        rows, err, ms = self.wl.sql.run(q["db_id"], st.sql or "")\n'
        '        tel.db_ms += ms\n'
        '        st.rows, st.error = rows, err\n'
        '        st.answer = st.sql or ""\n'
        '\n'
        '\n'
        'def run_many(executor: Executor, plan: Pipeline, queries: list[dict], purpose: str = "execute") -> list[ExecResult]:\n'
        '    return [executor.run(plan, q, purpose) for q in queries]\n'
        '\n'
        '\n'
        'def as_jsonable(x: Any) -> Any:\n'
        '    if hasattr(x, "to_dict"):\n'
        '        return x.to_dict()\n'
        '    return x\n'
    ),
    'eval/metrics.py': (
        '"""Accuracy metrics (Section 10.1 primary criteria).\n'
        '\n'
        '* token-level precision / recall / F1 (SQuAD normalisation, max over refs)\n'
        '* Rouge-L F-measure from the longest common *subsequence*\n'
        '* hallucination rate: share of answers whose content words are mostly absent\n'
        '  from the source the pipeline was allowed to read\n'
        '* execution accuracy for SQL (result-set equality)\n'
        '* retrieval recall@k for MS MARCO gold passages\n'
        '"""\n'
        'from __future__ import annotations\n'
        '\n'
        'import re\n'
        'import string\n'
        'import unicodedata\n'
        'from collections import Counter\n'
        '\n'
        '_ARTICLES = re.compile(r"\\b(a|an|the)\\b")\n'
        '_PUNCT = set(string.punctuation)\n'
        '_STOP = set("a an the of to in and or is are was were be been for on with by as at from that this "\n'
        '            "it its into than then there their they he she his her which who what when where how".split())\n'
        '\n'
        '\n'
        '_CITE = re.compile(r"【[^】]*】|\\[\\d+(?:[,\\s]*\\d+)*\\]")   # citation markers such as [2]\n'
        '\n'
        '\n'
        'def normalize(text: str) -> str:\n'
        '    text = _CITE.sub(" ", str(text)).lower()\n'
        '    # SQuAD normalisation drops punctuation; Unicode punctuation (curly quotes, CJK brackets) is dropped too\n'
        '    text = "".join(ch for ch in text if ch not in _PUNCT and not unicodedata.category(ch).startswith("P"))\n'
        '    text = _ARTICLES.sub(" ", text)\n'
        '    return " ".join(text.split())\n'
        '\n'
        '\n'
        'def tokens(text: str) -> list[str]:\n'
        '    return normalize(text).split()\n'
        '\n'
        '\n'
        'def is_none_answer(text: str) -> bool:\n'
        '    t = normalize(text)\n'
        '    return t in {"", "none", "not found", "no answer", "n a"} or t.startswith("none ")\n'
        '\n'
        '\n'
        'def prf(pred: str, ref: str) -> tuple[float, float, float]:\n'
        '    if is_none_answer(ref) or is_none_answer(pred):\n'
        '        hit = float(is_none_answer(ref) and is_none_answer(pred))\n'
        '        return hit, hit, hit\n'
        '    p, r = tokens(pred), tokens(ref)\n'
        '    common = Counter(p) & Counter(r)\n'
        '    same = sum(common.values())\n'
        '    if same == 0:\n'
        '        return 0.0, 0.0, 0.0\n'
        '    precision, recall = same / len(p), same / len(r)\n'
        '    return precision, recall, 2 * precision * recall / (precision + recall)\n'
        '\n'
        '\n'
        'def best_prf(pred: str, refs: list[str]) -> tuple[float, float, float]:\n'
        '    """For multi-span references (CUAD) the spans are also joined into one ref."""\n'
        '    cands = list(refs) + ([" ".join(refs)] if len(refs) > 1 else [])\n'
        '    return max((prf(pred, r) for r in cands), key=lambda t: t[2], default=(0.0, 0.0, 0.0))\n'
        '\n'
        '\n'
        'def _lcs(a: list[str], b: list[str]) -> int:\n'
        '    if not a or not b:\n'
        '        return 0\n'
        '    prev = [0] * (len(b) + 1)\n'
        '    for x in a:\n'
        '        cur = [0]\n'
        '        for j, y in enumerate(b, 1):\n'
        '            cur.append(prev[j - 1] + 1 if x == y else max(prev[j], cur[j - 1]))\n'
        '        prev = cur\n'
        '    return prev[-1]\n'
        '\n'
        '\n'
        'def rouge_l(pred: str, ref: str, beta: float = 1.2) -> float:\n'
        '    p, r = tokens(pred), tokens(ref)\n'
        '    lcs = _lcs(p, r)\n'
        '    if lcs == 0:\n'
        '        return float(is_none_answer(pred) and is_none_answer(ref))\n'
        '    prec, rec = lcs / len(p), lcs / len(r)\n'
        '    return (1 + beta ** 2) * prec * rec / (rec + beta ** 2 * prec)\n'
        '\n'
        '\n'
        'def best_rouge(pred: str, refs: list[str]) -> float:\n'
        '    return max((rouge_l(pred, r) for r in refs), default=0.0)\n'
        '\n'
        '\n'
        'def hallucinated(pred: str, source: str, threshold: float = 0.6) -> bool:\n'
        '    """True when fewer than ``threshold`` of the answer\'s content words occur in the source."""\n'
        '    if is_none_answer(pred):\n'
        '        return False\n'
        '    words = [w for w in tokens(pred) if w not in _STOP and len(w) > 2]\n'
        '    if not words:\n'
        '        return False\n'
        '    src = set(tokens(source))\n'
        '    grounded = sum(1 for w in words if w in src)\n'
        '    return grounded / len(words) < threshold\n'
        '\n'
        '\n'
        'def recall_at_k(retrieved: list[str], gold: list[str]) -> float:\n'
        '    if not gold:\n'
        '        return 0.0\n'
        '    return float(any(g in retrieved for g in gold))\n'
    ),
    'eval/oracle.py': (
        '"""Ground-truth scoring of executed plans.\n'
        '\n'
        'This module is the *only* place that reads reference answers.  The optimizer\n'
        'and its validation agent never call it; experiments use it to measure what\n'
        'the optimizer actually achieved (and to build the full-validation oracle that\n'
        'Section 10.2 compares decision quality against).\n'
        '"""\n'
        'from __future__ import annotations\n'
        '\n'
        'from statistics import mean\n'
        '\n'
        'from ..engine.executor import ExecResult\n'
        'from ..engine.sqlrun import results_match\n'
        'from . import metrics as M\n'
        '\n'
        '\n'
        'class Oracle:\n'
        '    def __init__(self, workload) -> None:\n'
        '        self.wl = workload\n'
        '        self._gold_rows: dict[str, list | None] = {}\n'
        '\n'
        '    def score_one(self, query: dict, res: ExecResult) -> dict:\n'
        '        out = {"qid": query["qid"], "tokens": res.tokens, "latency_ms": res.latency_ms}\n'
        '        if self.wl.metric == "execution_accuracy":\n'
        '            gold_sql = query["gold_sql"]\n'
        '            if query["qid"] not in self._gold_rows:\n'
        '                rows, err, _ = self.wl.sql.run(query["db_id"], gold_sql)\n'
        '                self._gold_rows[query["qid"]] = rows if err is None else None\n'
        '            ok = results_match(res.rows, self._gold_rows[query["qid"]], "order by" in gold_sql.lower())\n'
        '            out.update(accuracy=float(ok), exec_acc=float(ok), exec_error=res.error is not None,\n'
        '                       metric="execution_accuracy", answer=(res.sql or "")[:400])\n'
        '            return out\n'
        '        refs = query["answers"]\n'
        '        p, r, f = M.best_prf(res.answer, refs)\n'
        '        source = " ".join(res.context) + " " + query.get("contract_text", "")\n'
        '        rl = M.best_rouge(res.answer, refs)\n'
        '        # primary metric: Rouge-L for MS MARCO (its official QA metric), token F1 for CUAD spans\n'
        '        primary, name = (rl, "rouge_l") if self.wl.name == "msmarco" else (f, "token_f1")\n'
        '        out.update(accuracy=primary, metric=name, precision=p, recall=r, f1=f, rouge_l=rl,\n'
        '                   answer=res.answer[:400],\n'
        '                   hallucinated=float(M.hallucinated(res.answer, source)),\n'
        '                   answer_chars=len(res.answer))\n'
        '        if query.get("gold_ids"):\n'
        '            out["recall_at_k"] = M.recall_at_k(res.retrieved_ids, query["gold_ids"])\n'
        '        return out\n'
        '\n'
        '    def score(self, queries: list[dict], results: list[ExecResult]) -> dict:\n'
        '        rows = [self.score_one(q, r) for q, r in zip(queries, results)]\n'
        '        agg = {"n": len(rows)}\n'
        '        for key in ("accuracy", "precision", "recall", "f1", "rouge_l", "hallucinated", "exec_acc",\n'
        '                    "recall_at_k", "exec_error", "tokens", "latency_ms", "answer_chars"):\n'
        '            vals = [r[key] for r in rows if key in r]\n'
        '            if vals:\n'
        '                agg[key] = float(mean(vals))\n'
        '        agg["per_query"] = rows\n'
        '        return agg\n'
    ),
    'engine/sqlrun.py': (
        '"""SQLite execution for the SQL-heavy workloads (Spider, BIRD)."""\n'
        'from __future__ import annotations\n'
        '\n'
        'import re\n'
        'import sqlite3\n'
        'import time\n'
        'from collections import Counter\n'
        'from functools import lru_cache\n'
        'from pathlib import Path\n'
        '\n'
        '\n'
        'class SQLRunner:\n'
        '    def __init__(self, db_root: Path, timeout_s: float = 4.0, max_rows: int = 2000) -> None:\n'
        '        self.db_root = Path(db_root)\n'
        '        self.timeout_s = timeout_s\n'
        '        self.max_rows = max_rows\n'
        '\n'
        '    def db_path(self, db_id: str) -> Path:\n'
        '        return self.db_root / db_id / f"{db_id}.sqlite"\n'
        '\n'
        '    def run(self, db_id: str, sql: str) -> tuple[list[tuple] | None, str | None, float]:\n'
        '        """Execute read-only; returns (rows, error, ms).  Aborts slow queries."""\n'
        '        sql = clean_sql(sql)\n'
        '        if not sql:\n'
        '            return None, "empty SQL", 0.0\n'
        '        if re.match(r"^\\s*(insert|update|delete|drop|alter|create|attach|pragma)\\b", sql, re.I):\n'
        '            return None, "only SELECT statements are allowed", 0.0\n'
        '        uri = f"file:{self.db_path(db_id).as_posix()}?mode=ro"\n'
        '        t0 = time.perf_counter()\n'
        '        con = sqlite3.connect(uri, uri=True, check_same_thread=False)\n'
        '        con.text_factory = lambda b: b.decode("utf-8", errors="replace")\n'
        '        deadline = t0 + self.timeout_s\n'
        '        con.set_progress_handler(lambda: 1 if time.perf_counter() > deadline else 0, 10000)\n'
        '        try:\n'
        '            rows = con.execute(sql).fetchmany(self.max_rows)\n'
        '            return rows, None, (time.perf_counter() - t0) * 1000\n'
        '        except Exception as exc:  # sqlite3.Error, OperationalError on interrupt, ...\n'
        '            return None, str(exc)[:200], (time.perf_counter() - t0) * 1000\n'
        '        finally:\n'
        '            con.close()\n'
        '\n'
        '    @lru_cache(maxsize=64)\n'
        '    def schema(self, db_id: str) -> dict[str, list[str]]:\n'
        '        con = sqlite3.connect(f"file:{self.db_path(db_id).as_posix()}?mode=ro", uri=True)\n'
        '        try:\n'
        '            tables = [r[0] for r in con.execute(\n'
        '                "SELECT name FROM sqlite_master WHERE type=\'table\' AND name NOT LIKE \'sqlite_%\'")]\n'
        '            return {t: [r[1] for r in con.execute(f\'PRAGMA table_info("{t}")\')] for t in tables}\n'
        '        finally:\n'
        '            con.close()\n'
        '\n'
        '\n'
        'def schema_text(schema: dict[str, list[str]], tables: list[str] | None = None) -> str:\n'
        '    keep = tables or list(schema)\n'
        '    return "\\n".join(f"{t}({\', \'.join(schema[t])})" for t in keep if t in schema)\n'
        '\n'
        '\n'
        'def clean_sql(text: str) -> str:\n'
        '    """Pull a single statement out of an LLM reply (fences, prose, trailing ;)."""\n'
        '    if not text:\n'
        '        return ""\n'
        '    m = re.search(r"```(?:sql)?\\s*(.*?)```", text, re.S | re.I)\n'
        '    if m:\n'
        '        text = m.group(1)\n'
        '    m = re.search(r"\\b(SELECT|WITH)\\b.*", text, re.S | re.I)\n'
        '    if m:\n'
        '        text = m.group(0)\n'
        '    text = text.strip().split(";")[0]\n'
        '    return " ".join(text.split())\n'
        '\n'
        '\n'
        'def results_match(pred: list[tuple] | None, gold: list[tuple] | None, ordered: bool) -> bool:\n'
        '    """Execution accuracy: same multiset of rows (same order if ORDER BY)."""\n'
        '    if pred is None or gold is None:\n'
        '        return False\n'
        '    norm = lambda rows: [tuple(_norm_cell(c) for c in r) for r in rows]\n'
        '    p, g = norm(pred), norm(gold)\n'
        '    if ordered:\n'
        '        return p == g\n'
        '    return Counter(p) == Counter(g)\n'
        '\n'
        '\n'
        'def _norm_cell(c):\n'
        '    if isinstance(c, float):\n'
        '        return round(c, 4)\n'
        '    if isinstance(c, str):\n'
        '        return c.strip().lower()\n'
        '    return c\n'
    ),
})
run_selected(['baselines'])

plannerllm/core/pipeline.py
plannerllm/engine/executor.py
plannerllm/eval/metrics.py
plannerllm/eval/oracle.py
plannerllm/engine/sqlrun.py
baselines | seed 0 | 30 progress entries | reproduced_runs\recorded_groq\openai_gpt-oss-20b_seed0\baselines.json
   docetl | spider:world_1 | pass 1
   plannerllm | spider:orchestra | pass 1
   plannerllm | spider:world_1 | pass 1
msmarco no_opt 0.1683 opt tokens: 0
msmarco random 0.1864 opt tokens: 0
msmarco cost_only 0.1683 opt tokens: 1540
msmarco docetl 0.1541 opt tokens: 111644
msmarco plannerllm 0.1683 opt tokens: 32636
cuad no_opt 0.0 opt tokens: 0
cuad random 0.0 opt tokens: 0
cuad cost_only 0.0 opt tokens: 1558
cuad docetl 0.1399 opt tokens: 52890
cuad plannerllm 0.1517 opt tokens: 31113
spider no_opt 0.75 opt tokens: 0
spider random 0.75 opt tokens: 0
spider cost_only 0.75 opt tokens: 695
spider docetl 0.6667 opt tokens: 34785
spider plannerllm 0.75 opt tokens: 18318
baselines | seed 1 | 30 progress entries | reproduced_runs\recorded_groq\

"""Declarative pipeline model (Section 5.1).

A pipeline is a DAG of typed operators written in a DocETL-like YAML.  Every
operator names its inputs; when ``inputs`` is omitted the previous operator
is used, so the common retrieve -> rerank -> generate chain stays short.

Operator kinds
--------------
retrieve       ANN probe (``top_k``, ``index`` hnsw|ivf|flat, optional
               ``where`` => filtered vector probe)
filter         SQL predicate over corpus metadata, evaluated in DuckDB
rerank         cross-encoder rescoring, keeps ``top_k``
map            LLM call per record (extract / summarise), drops "NONE"
generate       one LLM call producing the final answer from the records
sql_generate   text-to-SQL over the schema in context (``fewshot_k``)
sql_repair     execute, and on error/empty result ask the LLM to fix
sql_execute    run the SQL and attach result rows
"""
from __future__ import annotations

import copy
import hashlib
import json
from dataclasses import dataclass, field
from typing import Any

import yaml

KINDS = {"retrieve", "filter", "rerank", "map", "generate",
         "sql_generate", "sql_repair", "sql_execute"}
LLM_KINDS = {"map", "generate", "sql_generate", "sql_repair"}
CHEAP_KINDS = KINDS - LLM_KINDS


class PipelineError(ValueError):
    pass


@dataclass
class Operator:
    name: str
    kind: str
    params: dict[str, Any] = field(default_factory=dict)
    prompt: str = ""
    inputs: list[str] = field(default_factory=list)

    def to_dict(self) -> dict:
        d: dict[str, Any] = {"name": self.name, "kind": self.kind}
        if self.params:
            d["params"] = copy.deepcopy(self.params)
        if self.prompt:
            d["prompt"] = self.prompt
        if self.inputs:
            d["inputs"] = list(self.inputs)
        return d

    @classmethod
    def from_dict(cls, d: dict) -> "Operator":
        if d.get("kind") not in KINDS:
            raise PipelineError(f"operator '{d.get('name')}' has unknown kind '{d.get('kind')}'")
        return cls(name=str(d["name"]), kind=d["kind"], params=dict(d.get("params") or {}),
                   prompt=str(d.get("prompt") or ""), inputs=list(d.get("inputs") or []))

    @property
    def uses_llm(self) -> bool:
        return self.kind in LLM_KINDS


@dataclass
class Pipeline:
    name: str
    workload: str
    operators: list[Operator] = field(default_factory=list)
    lineage: list[str] = field(default_factory=list)   # directives applied so far

    # ---------------------------------------------------------- structure
    def op(self, name: str) -> Operator:
        for o in self.operators:
            if o.name == name:
                return o
        raise KeyError(name)

    def find(self, kind: str) -> Operator | None:
        return next((o for o in self.operators if o.kind == kind), None)

    def index_of(self, name: str) -> int:
        return next(i for i, o in enumerate(self.operators) if o.name == name)

    def kinds(self) -> list[str]:
        return [o.kind for o in self.topological()]

    def resolved_inputs(self, op: Operator) -> list[str]:
        if op.inputs:
            return op.inputs
        i = self.index_of(op.name)
        return [self.operators[i - 1].name] if i > 0 else []

    def topological(self) -> list[Operator]:
        """Kahn's algorithm; raises on cycles or dangling inputs."""
        names = {o.name for o in self.operators}
        deps = {o.name: set(self.resolved_inputs(o)) for o in self.operators}
        for n, ds in deps.items():
            missing = ds - names
            if missing:
                raise PipelineError(f"operator '{n}' reads unknown input(s) {sorted(missing)}")
        order, ready = [], [o.name for o in self.operators if not deps[o.name]]
        while ready:
            n = ready.pop(0)
            order.append(n)
            for m, ds in deps.items():
                if n in ds:
                    ds.discard(n)
                    if not ds and m not in order and m not in

"""Execution engine: runs one pipeline for one query and records telemetry.

Telemetry is split the way Section 10.6 decomposes wall-clock time:
DB operator time (DuckDB / SQLite), retrieval time (ANN probe), and LLM time,
plus the tokens and dollars each operator consumed.  The same numbers feed
the observed-cost log that the neural residual is trained on.
"""
from __future__ import annotations

import time
from dataclasses import asdict, dataclass, field
from typing import Any

from ..core.pipeline import Operator, Pipeline
from . import prompts
from .sqlrun import clean_sql, schema_text
from .store import render_where

MAX_DOC_CHARS = 600


@dataclass
class OpTelemetry:
    op: str
    kind: str
    rows_in: int
    rows_out: int
    ms: float
    db_ms: float = 0.0
    ann_ms: float = 0.0
    llm_ms: float = 0.0
    tokens: int = 0
    usd: float = 0.0
    llm_wall_ms: float = 0.0


@dataclass
class ExecResult:
    qid: str
    answer: str
    context: list[str] = field(default_factory=list)
    sql: str | None = None
    rows: list | None = None
    error: str | None = None
    retrieved_ids: list[str] = field(default_factory=list)
    telemetry: list[OpTelemetry] = field(default_factory=list)

    # aggregate views ---------------------------------------------------
    @property
    def tokens(self) -> int:
        return sum(t.tokens for t in self.telemetry)

    @property
    def llm_usd(self) -> float:
        return sum(t.usd for t in self.telemetry)

    @property
    def latency_ms(self) -> float:
        return sum(t.ms for t in self.telemetry)

    def breakdown(self) -> dict:
        return {"db_ms": sum(t.db_ms for t in self.telemetry),
                "ann_ms": sum(t.ann_ms for t in self.telemetry),
                "llm_ms": sum(t.llm_ms for t in self.telemetry),
                "other_ms": sum(max(0.0, t.ms - t.db_ms - t.ann_ms - t.llm_ms) for t in self.telemetry)}

    def to_dict(self) -> dict:
        d = asdict(self)
        d["tokens"], d["latency_ms"] = self.tokens, self.latency_ms
        d["rows"] = [list(map(str, r)) for r in (self.rows or [])[:10]]
        return d


class _State:
    """Data flowing between operators for one query."""

    def __init__(self) -> None:
        self.positions: list[int] | None = None   # None == whole corpus
        self.extracts: dict[int, str] = {}
        self.answer: str = ""
        self.sql: str | None = None
        self.rows: list | None = None
        self.error: str | None = None
        self.context: list[str] = []


class Executor:
    def __init__(self, workload, llm, cpu_usd_per_hour: float = 0.04) -> None:
        self.wl = workload
        self.llm = llm
        self.cpu_usd_per_ms = cpu_usd_per_hour / 3.6e6

    # ------------------------------------------------------------------
    def run(self, plan: Pipeline, query: dict, purpose: str = "execute") -> ExecResult:
        st = _State()
        res = ExecResult(qid=query["qid"], answer="")
        for op in plan.topological():
            t0 = time.perf_counter()
            rows_in = len(st.positions) if st.positions is not None else self.wl.store.n_rows
            tel = OpTelemetry(op=op.name, kind=op.kind, rows_in=rows_in, rows_out=0, ms=0.0)
            try:
                getattr(self, f"_op_{op.kind}")(op, query, st, tel, purpose)
            except KeyError as exc:  # missing query field for a predicate
                st.error = f"{op.name}: {exc}"
            # wall time of the operator with the LLM's wall time (which on a cache hit is ~0 and on a
            # fresh call includes rate-limit waits) replaced by the API's service time
            tel.ms = (time.perf_counter() - t0) * 1000 - tel.llm_wall_ms + tel.llm_ms
            tel.rows_out = len(st.positions) if st.positions is not None else tel.rows_in
            res.telemetry.append(tel)
        res.answer, res.sql, res.rows, res.error = st.answer, st.sql, st.rows, st.error
        res.context = st.context
        if st.positions is 

"""Accuracy metrics (Section 10.1 primary criteria).

* token-level precision / recall / F1 (SQuAD normalisation, max over refs)
* Rouge-L F-measure from the longest common *subsequence*
* hallucination rate: share of answers whose content words are mostly absent
  from the source the pipeline was allowed to read
* execution accuracy for SQL (result-set equality)
* retrieval recall@k for MS MARCO gold passages
"""
from __future__ import annotations

import re
import string
import unicodedata
from collections import Counter

_ARTICLES = re.compile(r"\b(a|an|the)\b")
_PUNCT = set(string.punctuation)
_STOP = set("a an the of to in and or is are was were be been for on with by as at from that this "
            "it its into than then there their they he she his her which who what when where how".split())


_CITE = re.compile(r"【[^】]*】|\[\d+(?:[,\s]*\d+)*\]")   # citation markers such as [2]


def normalize(text: str) -> str:
    text = _CITE.sub(" ", str(text)).lower()
    # SQuAD normalisation drops punctuation; Unicode punctuation (curly quotes, CJK brackets) is dropped too
    text = "".join(ch for ch in text if ch not in _PUNCT and not unicodedata.category(ch).startswith("P"))
    text = _ARTICLES.sub(" ", text)
    return " ".join(text.split())


def tokens(text: str) -> list[str]:
    return normalize(text).split()


def is_none_answer(text: str) -> bool:
    t = normalize(text)
    return t in {"", "none", "not found", "no answer", "n a"} or t.startswith("none ")


def prf(pred: str, ref: str) -> tuple[float, float, float]:
    if is_none_answer(ref) or is_none_answer(pred):
        hit = float(is_none_answer(ref) and is_none_answer(pred))
        return hit, hit, hit
    p, r = tokens(pred), tokens(ref)
    common = Counter(p) & Counter(r)
    same = sum(common.values())
    if same == 0:
        return 0.0, 0.0, 0.0
    precision, recall = same / len(p), same / len(r)
    return precision, recall, 2 * precision * recall / (precision + recall)


def best_prf(pred: str, refs: list[str]) -> tuple[float, float, float]:
    """For multi-span references (CUAD) the spans are also joined into one ref."""
    cands = list(refs) + ([" ".join(refs)] if len(refs) > 1 else [])
    return max((prf(pred, r) for r in cands), key=lambda t: t[2], default=(0.0, 0.0, 0.0))


def _lcs(a: list[str], b: list[str]) -> int:
    if not a or not b:
        return 0
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b, 1):
            cur.append(prev[j - 1] + 1 if x == y else max(prev[j], cur[j - 1]))
        prev = cur
    return prev[-1]


def rouge_l(pred: str, ref: str, beta: float = 1.2) -> float:
    p, r = tokens(pred), tokens(ref)
    lcs = _lcs(p, r)
    if lcs == 0:
        return float(is_none_answer(pred) and is_none_answer(ref))
    prec, rec = lcs / len(p), lcs / len(r)
    return (1 + beta ** 2) * prec * rec / (rec + beta ** 2 * prec)


def best_rouge(pred: str, refs: list[str]) -> float:
    return max((rouge_l(pred, r) for r in refs), default=0.0)


def hallucinated(pred: str, source: str, threshold: float = 0.6) -> bool:
    """True when fewer than ``threshold`` of the answer's content words occur in the source."""
    if is_none_answer(pred):
        return False
    words = [w for w in tokens(pred) if w not in _STOP and len(w) > 2]
    if not words:
        return False
    src = set(tokens(source))
    grounded = sum(1 for w in words if w in src)
    return grounded / len(words) < threshold


def recall_at_k(retrieved: list[str], gold: list[str]) -> float:
    if not gold:
        return 0.0
    return float(any(g in retrieved for g in gold))

"""Ground-truth scoring of executed plans.

This module is the *only* place that reads reference answers.  The optimizer
and its validation agent never call it; experiments use it to measure what
the optimizer actually achieved (and to build the full-validation oracle that
Section 10.2 compares decision quality against).
"""
from __future__ import annotations

from statistics import mean

from ..engine.executor import ExecResult
from ..engine.sqlrun import results_match
from . import metrics as M


class Oracle:
    def __init__(self, workload) -> None:
        self.wl = workload
        self._gold_rows: dict[str, list | None] = {}

    def score_one(self, query: dict, res: ExecResult) -> dict:
        out = {"qid": query["qid"], "tokens": res.tokens, "latency_ms": res.latency_ms}
        if self.wl.metric == "execution_accuracy":
            gold_sql = query["gold_sql"]
            if query["qid"] not in self._gold_rows:
                rows, err, _ = self.wl.sql.run(query["db_id"], gold_sql)
                self._gold_rows[query["qid"]] = rows if err is None else None
            ok = results_match(res.rows, self._gold_rows[query["qid"]], "order by" in gold_sql.lower())
            out.update(accuracy=float(ok), exec_acc=float(ok), exec_error=res.error is not None,
                       metric="execution_accuracy", answer=(res.sql or "")[:400])
            return out
        refs = query["answers"]
        p, r, f = M.best_prf(res.answer, refs)
        source = " ".join(res.context) + " " + query.get("contract_text", "")
        rl = M.best_rouge(res.answer, refs)
        # primary metric: Rouge-L for MS MARCO (its official QA metric), token F1 for CUAD spans
        primary, name = (rl, "rouge_l") if self.wl.name == "msmarco" else (f, "token_f1")
        out.update(accuracy=primary, metric=name, precision=p, recall=r, f1=f, rouge_l=rl,
                   answer=res.answer[:400],
                   hallucinated=float(M.hallucinated(res.answer, source)),
                   answer_chars=len(res.answer))
        if query.get("gold_ids"):
            out["recall_at_k"] = M.recall_at_k(res.retrieved_ids, query["gold_ids"])
        return out

    def score(self, queries: list[dict], results: list[ExecResult]) -> dict:
        rows = [self.score_one(q, r) for q, r in zip(queries, results)]
        agg = {"n": len(rows)}
        for key in ("accuracy", "precision", "recall", "f1", "rouge_l", "hallucinated", "exec_acc",
                    "recall_at_k", "exec_error", "tokens", "latency_ms", "answer_chars"):
            vals = [r[key] for r in rows if key in r]
            if vals:
                agg[key] = float(mean(vals))
        agg["per_query"] = rows
        return agg

"""SQLite execution for the SQL-heavy workloads (Spider, BIRD)."""
from __future__ import annotations

import re
import sqlite3
import time
from collections import Counter
from functools import lru_cache
from pathlib import Path


class SQLRunner:
    def __init__(self, db_root: Path, timeout_s: float = 4.0, max_rows: int = 2000) -> None:
        self.db_root = Path(db_root)
        self.timeout_s = timeout_s
        self.max_rows = max_rows

    def db_path(self, db_id: str) -> Path:
        return self.db_root / db_id / f"{db_id}.sqlite"

    def run(self, db_id: str, sql: str) -> tuple[list[tuple] | None, str | None, float]:
        """Execute read-only; returns (rows, error, ms).  Aborts slow queries."""
        sql = clean_sql(sql)
        if not sql:
            return None, "empty SQL", 0.0
        if re.match(r"^\s*(insert|update|delete|drop|alter|create|attach|pragma)\b", sql, re.I):
            return None, "only SELECT statements are allowed", 0.0
        uri = f"file:{self.db_path(db_id).as_posix()}?mode=ro"
        t0 = time.perf_counter()
        con = sqlite3.connect(uri, uri=True, check_same_thread=False)
        con.text_factory = lambda b: b.decode("utf-8", errors="replace")
        deadline = t0 + self.timeout_s
        con.set_progress_handler(lambda: 1 if time.perf_counter() > deadline else 0, 10000)
        try:
            rows = con.execute(sql).fetchmany(self.max_rows)
            return rows, None, (time.perf_counter() - t0) * 1000
        except Exception as exc:  # sqlite3.Error, OperationalError on interrupt, ...
            return None, str(exc)[:200], (time.perf_counter() - t0) * 1000
        finally:
            con.close()

    @lru_cache(maxsize=64)
    def schema(self, db_id: str) -> dict[str, list[str]]:
        con = sqlite3.connect(f"file:{self.db_path(db_id).as_posix()}?mode=ro", uri=True)
        try:
            tables = [r[0] for r in con.execute(
                "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")]
            return {t: [r[1] for r in con.execute(f'PRAGMA table_info("{t}")')] for t in tables}
        finally:
            con.close()


def schema_text(schema: dict[str, list[str]], tables: list[str] | None = None) -> str:
    keep = tables or list(schema)
    return "\n".join(f"{t}({', '.join(schema[t])})" for t in keep if t in schema)


def clean_sql(text: str) -> str:
    """Pull a single statement out of an LLM reply (fences, prose, trailing ;)."""
    if not text:
        return ""
    m = re.search(r"```(?:sql)?\s*(.*?)```", text, re.S | re.I)
    if m:
        text = m.group(1)
    m = re.search(r"\b(SELECT|WITH)\b.*", text, re.S | re.I)
    if m:
        text = m.group(0)
    text = text.strip().split(";")[0]
    return " ".join(text.split())


def results_match(pred: list[tuple] | None, gold: list[tuple] | None, ordered: bool) -> bool:
    """Execution accuracy: same multiset of rows (same order if ORDER BY)."""
    if pred is None or gold is None:
        return False
    norm = lambda rows: [tuple(_norm_cell(c) for c in r) for r in rows]
    p, g = norm(pred), norm(gold)
    if ordered:
        return p == g
    return Counter(p) == Counter(g)


def _norm_cell(c):
    if isinstance(c, float):
        return round(c, 4)
    if isinstance(c, str):
        return c.strip().lower()
    return c

In [4]:
# Execute the exact scorer on real outputs produced by this notebook.
for seed in SEEDS:
    settings = settings_for(seed)
    result = E.load(settings, "baselines")
    for workload in WORKLOADS:
        session = E._session(workload, settings)
        row = next(r for r in result["rows"] if r["workload"] == workload and r["system"] == "no_opt")
        query_id = row["per_query"][0]["qid"]
        query = next(q for q in session.workload.queries if q["qid"] == query_id)
        execution = session.executor.run(session.workload.base_pipeline(), query, purpose="execute")
        score = session.oracle.score_one(query, execution)
        print(f"Seed {seed}, {workload}, {query_id}: {score['metric']} = {score['accuracy']:.4f}")
        print("  Prediction:", score["answer"][:160])
        if workload == "spider":
            gold_rows, error, _ = session.workload.sql.run(query["db_id"], query["gold_sql"])
            assert error is None
            print("  Predicted / reference row counts:", len(execution.rows or []), len(gold_rows))
        else:
            print("  Reference:", query["answers"][0][:160])

Seed 0, msmarco, m16343: rouge_l = 0.1961
  Prediction: She is currently in London, where she is promoting her autobiography.
  Reference: London
Seed 0, cuad, c51_governing_law: token_f1 = 0.0000
  Prediction: NONE
  Reference: This Agreement shall be governed by, and its provisions enforced in accordance with, the laws of California without regard to its principles of conflicts of law
Seed 0, spider, s827: execution_accuracy = 1.0000
  Prediction: SELECT Name FROM conductor WHERE Nationality <> 'USA'
  Predicted / reference row counts: 6 6
Seed 1, msmarco, m16844: rouge_l = 0.0000
  Prediction: Sang Nila Utama was a prince from Palembang, the capital of the Sriwijaya Empire【1】【2】.
  Reference: Sumatra
Seed 1, cuad, c26_expiration_date: token_f1 = 0.0000
  Prediction: NONE
  Reference: This Agreement shall be valid for an indefinite period.
Seed 1, spider, s62: execution_accuracy = 0.0000
  Prediction: SELECT DISTINCT s.Major, s.Age FROM Student s WHERE NOT EXISTS ( SELECT 1 FROM Has_

## Rewrite search, token budgets and VoI gate

In [5]:
# Source snapshot. The experiment below executes the imported copy.
show_implementation({
    'core/directives.py': (
        '"""Rewrite directives (Section 5.1).\n'
        '\n'
        'Each directive is a small, named plan transformation.  Logical directives\n'
        'change *what* the LLM is asked to do (decompose, merge, prompt rewrite);\n'
        'physical ones change *how* data is accessed (index family, top-k); the\n'
        'retrieval-subplan family added by PlannerLLM moves predicates in and out of\n'
        'the ANN probe and adds/removes a rerank stage ("and vice versa").  SQL\n'
        'directives cover the text-to-SQL workloads.\n'
        '\n'
        'A directive never decides whether its rewrite is good - that is the job of\n'
        'the cost model, the plan memory and the gated validation agent.\n'
        '"""\n'
        'from __future__ import annotations\n'
        '\n'
        'from dataclasses import dataclass, field\n'
        'from typing import Callable\n'
        '\n'
        'from . import pipeline as P\n'
        'from .pipeline import Operator, Pipeline\n'
        '\n'
        'MAP_INSTRUCTION = ("Copy the sentence(s) from the text that help answer the question, word for word. "\n'
        '                   "If none do, reply NONE.")\n'
        '\n'
        '\n'
        '@dataclass\n'
        'class DirectiveContext:\n'
        '    workload: str\n'
        '    family: str\n'
        '    task_text: str\n'
        '    filter_template: str\n'
        '    stats: dict\n'
        '    llm: object | None = None\n'
        '\n'
        '\n'
        '@dataclass\n'
        'class Candidate:\n'
        '    plan: Pipeline\n'
        '    directive: str\n'
        '    detail: str = ""\n'
        '    proposed_by: str = "rules"   # rules | agent\n'
        '\n'
        '\n'
        '@dataclass\n'
        'class Directive:\n'
        '    name: str\n'
        '    family: str          # retrieval | physical | logical | sql\n'
        '    description: str\n'
        '    fn: Callable[[Pipeline, DirectiveContext], list[tuple[Pipeline, str]]] = field(repr=False)\n'
        '\n'
        '    def apply(self, plan: Pipeline, ctx: DirectiveContext) -> list[Candidate]:\n'
        '        out = []\n'
        '        for new, detail in self.fn(plan, ctx):\n'
        '            new.lineage = plan.lineage + [self.name]\n'
        '            try:\n'
        '                new.validate()\n'
        '            except P.PipelineError:\n'
        '                continue\n'
        '            out.append(Candidate(new, self.name, detail))\n'
        '        return out\n'
        '\n'
        '\n'
        '# ------------------------------------------------------------------ helpers\n'
        'def _first_llm_op(plan: Pipeline) -> Operator | None:\n'
        '    return next((o for o in plan.topological() if o.uses_llm), None)\n'
        '\n'
        '\n'
        'def _terminal_llm(plan: Pipeline) -> Operator | None:\n'
        '    llm_ops = [o for o in plan.topological() if o.kind in ("generate", "sql_generate")]\n'
        '    return llm_ops[-1] if llm_ops else None\n'
        '\n'
        '\n'
        '# ------------------------------------------------------------------ retrieval family\n'
        'def add_rerank(plan: Pipeline, ctx: DirectiveContext):\n'
        '    r = plan.find("retrieve")\n'
        '    if r is None or plan.find("rerank") is not None or ctx.family == "sql-heavy":\n'
        '        return []\n'
        '    p = plan.clone()\n'
        '    r2 = p.op(r.name)\n'
        '    keep = int(r2.params.get("top_k", 10))\n'
        '    r2.params["top_k"] = min(max(keep * 2, 20), 40)\n'
        '    anchor = _first_llm_op(p)\n'
        '    p.insert_before(anchor.name, Operator(p.unique_name("rerank"), "rerank", {"top_k": keep}))\n'
        '    return [(p, f"retrieve top_k {keep}->{r2.params[\'top_k\']}, cross-encoder keeps {keep}")]\n'
        '\n'
        '\n'
        'def drop_rerank(plan: Pipeline, ctx: DirectiveContext):\n'
        '    rr, r = plan.find("rerank"), plan.find("retrieve")\n'
        '    if rr is None or r is None:\n'
        '        return []\n'
        '    p = plan.clone()\n'
        '    p.op(r.name).params["top_k"] = int(rr.params.get("top_k", 5))\n'
        '    p.remove(rr.name)\n'
        '    return [(p, "single-stage probe, top_k = former rerank k")]\n'
        '\n'
        '\n'
        'def push_filter_into_probe(plan: Pipeline, ctx: DirectiveContext):\n'
        '    """filter (before or after) + retrieve  ==>  retrieve[where]  (filtered vector probe)."""\n'
        '    r, f = plan.find("retrieve"), plan.find("filter")\n'
        '    if r is None or f is None or r.params.get("where"):\n'
        '        return []\n'
        '    p = plan.clone()\n'
        '    where = p.op(f.name).params["where"]\n'
        '    p.remove(f.name)\n'
        '    p.op(r.name).params["where"] = where\n'
        '    return [(p, f"predicate \'{where}\' evaluated inside the ANN probe")]\n'
        '\n'
        '\n'
        'def pull_filter_out_of_probe(plan: Pipeline, ctx: DirectiveContext):\n'
        '    """retrieve[where]  ==>  retrieve(over-fetch) -> filter   (post-filtering)."""\n'
        '    r = plan.find("retrieve")\n'
        '    if r is None or not r.params.get("where"):\n'
        '        return []\n'
        '    p = plan.clone()\n'
        '    r2 = p.op(r.name)\n'
        '    where = r2.params.pop("where")\n'
        '    r2.params["top_k"] = min(int(r2.params.get("top_k", 10)) * 3, 60)\n'
        '    p.insert_after(r2.name, Operator(p.unique_name("post_filter"), "filter", {"where": where}))\n'
        '    return [(p, f"probe without predicate (k x3), then filter \'{where}\'")]\n'
        '\n'
        '\n'
        'def add_filter(plan: Pipeline, ctx: DirectiveContext):\n'
        '    """Insert an early metadata predicate taken from the query\'s structured fields."""\n'
        '    r = plan.find("retrieve")\n'
        '    if r is None or plan.find("filter") is not None or r.params.get("where"):\n'
        '        return []\n'
        '    p = plan.clone()\n'
        '    p.insert_before(r.name, Operator(p.unique_name("pre_filter"), "filter", {"where": ctx.filter_template}))\n'
        '    return [(p, f"pre-filter rows with \'{ctx.filter_template}\' before the probe")]\n'
        '\n'
        '\n'
        '# ------------------------------------------------------------------ physical family\n'
        'def swap_index(plan: Pipeline, ctx: DirectiveContext):\n'
        '    r = plan.find("retrieve")\n'
        '    if r is None:\n'
        '        return []\n'
        '    cur = r.params.get("index", "hnsw")\n'
        '    out = []\n'
        '    for target in {"hnsw": ("ivf", "flat"), "ivf": ("hnsw",), "flat": ("hnsw",)}.get(cur, ("hnsw",)):\n'
        '        p = plan.clone()\n'
        '        p.op(r.name).params["index"] = target\n'
        '        out.append((p, f"index {cur} -> {target}"))\n'
        '    return out\n'
        '\n'
        '\n'
        'def tune_topk(plan: Pipeline, ctx: DirectiveContext):\n'
        '    r = plan.find("rerank") or plan.find("retrieve")\n'
        '    if r is None:\n'
        '        return []\n'
        '    k = int(r.params.get("top_k", 10))\n'
        '    out = []\n'
        '    for new_k in sorted({max(3, k // 2), min(40, k * 2)} - {k}):\n'
        '        p = plan.clone()\n'
        '        p.op(r.name).params["top_k"] = new_k\n'
        '        if r.kind == "rerank":\n'
        '            rr = p.find("retrieve")\n'
        '            if rr is not None and int(rr.params.get("top_k", 10)) < new_k:\n'
        '                rr.params["top_k"] = new_k * 2\n'
        '        out.append((p, f"{r.kind} top_k {k} -> {new_k}"))\n'
        '    return out\n'
        '\n'
        '\n'
        '# ------------------------------------------------------------------ logical family\n'
        'def decompose_generate(plan: Pipeline, ctx: DirectiveContext):\n'
        '    """generate  ==>  map(extract per record) -> generate   (DocETL chaining)."""\n'
        '    g = plan.find("generate")\n'
        '    if g is None or plan.find("map") is not None:\n'
        '        return []\n'
        '    p = plan.clone()\n'
        '    # extract only from the records the final call will read; the rest would be wasted calls\n'
        '    cap = int(g.params.get("max_docs", 6))\n'
        '    p.insert_before(g.name, Operator(p.unique_name("extract"), "map", {"max_records": cap}, prompt=MAP_INSTRUCTION))\n'
        '    return [(p, "per-record extraction before the final answer")]\n'
        '\n'
        '\n'
        'def merge_map(plan: Pipeline, ctx: DirectiveContext):\n'
        '    """map -> generate  ==>  generate   (fusion: fewer LLM round-trips)."""\n'
        '    m, g = plan.find("map"), plan.find("generate")\n'
        '    if m is None or g is None:\n'
        '        return []\n'
        '    p = plan.clone()\n'
        '    p.remove(m.name)\n'
        '    return [(p, "fold extraction into the final generate call")]\n'
        '\n'
        '\n'
        'def prompt_rewrite(plan: Pipeline, ctx: DirectiveContext):\n'
        '    target = _terminal_llm(plan)\n'
        '    if target is None or ctx.llm is None or "PROMPT_REWRITE" in plan.lineage:\n'
        '        return []\n'
        '    from ..engine import prompts\n'
        '    from ..llm.base import BudgetExhausted\n'
        '    msg = prompts.PROMPT_REWRITE.format(kind=target.kind, task=ctx.task_text, prompt=target.prompt)\n'
        '    try:\n'
        '        r = ctx.llm.complete(msg, purpose="prompt_rewrite", task="prompt_rewrite",\n'
        '                             payload={"prompt": target.prompt}, system=prompts.SYSTEM_AGENT,\n'
        '                             max_tokens=220, json_mode=True)\n'
        '    except BudgetExhausted:\n'
        '        return []\n'
        '    new_prompt = (r.json({}) or {}).get("prompt", "")\n'
        '    if not isinstance(new_prompt, str) or len(new_prompt) < 20 or new_prompt.strip() == target.prompt.strip():\n'
        '        return []\n'
        '    p = plan.clone()\n'
        '    p.op(target.name).prompt = new_prompt.strip()[:600]\n'
        '    return [(p, f"agent-rewritten instruction for \'{target.name}\'")]\n'
        '\n'
        '\n'
        '# ------------------------------------------------------------------ SQL family\n'
        'def add_schema_link(plan: Pipeline, ctx: DirectiveContext):\n'
        '    g = plan.find("sql_generate")\n'
        '    if g is None or plan.find("retrieve") is not None:\n'
        '        return []\n'
        '    p = plan.clone()\n'
        '    p.insert_before(g.name, Operator(p.unique_name("link_schema"), "retrieve",\n'
        '                                     {"top_k": 4, "index": "flat", "where": ctx.filter_template}))\n'
        '    return [(p, "retrieve the 4 most relevant tables instead of the whole schema")]\n'
        '\n'
        '\n'
        'def add_fewshot(plan: Pipeline, ctx: DirectiveContext):\n'
        '    g = plan.find("sql_generate")\n'
        '    if g is None or int(g.params.get("fewshot_k", 0)) > 0:\n'
        '        return []\n'
        '    p = plan.clone()\n'
        '    p.op(g.name).params["fewshot_k"] = 3\n'
        '    return [(p, "3 nearest training question/SQL pairs as demonstrations")]\n'
        '\n'
        '\n'
        'def add_sql_repair(plan: Pipeline, ctx: DirectiveContext):\n'
        '    g = plan.find("sql_generate")\n'
        '    if g is None or plan.find("sql_repair") is not None:\n'
        '        return []\n'
        '    p = plan.clone()\n'
        '    p.insert_after(g.name, Operator(p.unique_name("repair_sql"), "sql_repair"))\n'
        '    return [(p, "execute once; on error or empty result ask the LLM to fix the SQL")]\n'
        '\n'
        '\n'
        'class DirectiveStore:\n'
        '    """Registry of all directives with a human-readable catalog for the agent."""\n'
        '\n'
        '    def __init__(self) -> None:\n'
        '        self._d: dict[str, Directive] = {}\n'
        '        for name, fam, desc, fn in [\n'
        '            ("ADD_RERANK", "retrieval", "two-stage plan: over-fetch with ANN, keep top-k by cross-encoder", add_rerank),\n'
        '            ("DROP_RERANK", "retrieval", "single-stage plan: remove the rerank stage", drop_rerank),\n'
        '            ("FILTERED_PROBE", "retrieval", "evaluate the metadata predicate inside the ANN probe", push_filter_into_probe),\n'
        '            ("POST_FILTER", "retrieval", "probe without the predicate, then filter (over-fetch x3)", pull_filter_out_of_probe),\n'
        '            ("ADD_FILTER", "logical", "add an early metadata predicate from the query\'s fields", add_filter),\n'
        '            ("SWAP_INDEX", "physical", "change the ANN index family (hnsw / ivf / flat)", swap_index),\n'
        '            ("TUNE_TOPK", "physical", "halve or double how many records reach the LLM", tune_topk),\n'
        '            ("DECOMPOSE", "logical", "extract relevant sentences per record before answering", decompose_generate),\n'
        '            ("MERGE_MAP", "logical", "fuse the extraction step into the final LLM call", merge_map),\n'
        '            ("PROMPT_REWRITE", "logical", "let the agent rewrite the final operator\'s instruction", prompt_rewrite),\n'
        '            ("SCHEMA_LINK", "sql", "retrieve only the relevant tables before writing SQL", add_schema_link),\n'
        '            ("FEWSHOT", "sql", "add retrieved question/SQL demonstrations", add_fewshot),\n'
        '            ("SQL_REPAIR", "sql", "execute and repair failing or empty SQL", add_sql_repair),\n'
        '        ]:\n'
        '            self.register(Directive(name, fam, desc, fn))\n'
        '\n'
        '    def register(self, d: Directive) -> None:\n'
        '        self._d[d.name] = d\n'
        '\n'
        '    def get(self, name: str) -> Directive:\n'
        '        return self._d[name]\n'
        '\n'
        '    def names(self) -> list[str]:\n'
        '        return list(self._d)\n'
        '\n'
        '    def catalog(self) -> list[dict]:\n'
        '        return [{"name": d.name, "family": d.family, "description": d.description} for d in self._d.values()]\n'
        '\n'
        '    def applicable(self, plan: Pipeline, ctx: DirectiveContext) -> list[str]:\n'
        '        """Directives whose cheap (non-LLM) form produces at least one candidate."""\n'
        '        names = []\n'
        '        probe_ctx = DirectiveContext(**{**ctx.__dict__, "llm": None})\n'
        '        for d in self._d.values():\n'
        '            if d.name == "PROMPT_REWRITE":\n'
        '                if _terminal_llm(plan) is not None and "PROMPT_REWRITE" not in plan.lineage:\n'
        '                    names.append(d.name)\n'
        '                continue\n'
        '            if d.fn(plan, probe_ctx):\n'
        '                names.append(d.name)\n'
        '        return names\n'
        '\n'
        '    def expand(self, plan: Pipeline, ctx: DirectiveContext, names: list[str] | None = None,\n'
        '               cap: int | None = None) -> list[Candidate]:\n'
        '        out: list[Candidate] = []\n'
        '        seen = {plan.fingerprint()}\n'
        '        for name in names or self.names():\n'
        '            if name not in self._d or (cap is not None and len(out) >= cap):\n'
        '                continue\n'
        '            for cand in self._d[name].apply(plan, ctx):\n'
        '                fp = cand.plan.fingerprint()\n'
        '                if fp not in seen and (cap is None or len(out) < cap):\n'
        '                    seen.add(fp)\n'
        '                    out.append(cand)\n'
        '        return out\n'
    ),
    'core/generation.py': (
        '"""Generation agent: chooses which directives to apply (Section 5.1).\n'
        '\n'
        'The LLM sees the current plan, the data statistics and the catalog of\n'
        'directives that are actually applicable, and nominates the promising ones.\n'
        'Its picks are tried first; the remaining applicable directives follow, so a\n'
        'bad or empty reply never leaves the search without candidates (the\n'
        'rule-based fallback the base system also relies on).\n'
        '"""\n'
        'from __future__ import annotations\n'
        '\n'
        'from ..engine import prompts\n'
        'from ..llm.base import BudgetExhausted\n'
        'from .directives import Candidate, DirectiveContext, DirectiveStore\n'
        'from .pipeline import Pipeline\n'
        '\n'
        'STAT_KEYS = ("n_rows", "sel_est", "avg_doc_tokens", "n_tables", "avg_query_tokens")\n'
        '\n'
        '\n'
        'class GenerationAgent:\n'
        '    def __init__(self, llm, store: DirectiveStore, use_llm: bool = True) -> None:\n'
        '        self.llm = llm\n'
        '        self.store = store\n'
        '        self.use_llm = use_llm\n'
        '        self.last_proposal: list[dict] = []\n'
        '\n'
        '    def propose(self, plan: Pipeline, ctx: DirectiveContext, cap: int) -> list[Candidate]:\n'
        '        applicable = self.store.applicable(plan, ctx)\n'
        '        chosen: list[str] = []\n'
        '        self.last_proposal = []\n'
        '        if self.use_llm and applicable:\n'
        '            catalog = "\\n".join(f"- {d[\'name\']}: {d[\'description\']}" for d in self.store.catalog()\n'
        '                                if d["name"] in applicable)\n'
        '            stats = {k: ctx.stats[k] for k in STAT_KEYS if k in ctx.stats}\n'
        '            msg = prompts.PROPOSE.format(task=ctx.task_text, pipeline=plan.describe(), stats=stats, catalog=catalog)\n'
        '            try:\n'
        '                r = self.llm.complete(msg, purpose="generation_agent", task="propose",\n'
        '                                      payload={"directives": applicable, "pipeline": plan.describe()},\n'
        '                                      system=prompts.SYSTEM_AGENT, max_tokens=300, json_mode=True)\n'
        '                picks = (r.json({}) or {}).get("directives") or []\n'
        '                for p in picks:\n'
        '                    name = (p.get("name") if isinstance(p, dict) else str(p)).strip().upper()\n'
        '                    if name in applicable and name not in chosen:\n'
        '                        chosen.append(name)\n'
        '                        self.last_proposal.append({"name": name, "why": p.get("why", "") if isinstance(p, dict) else ""})\n'
        '            except BudgetExhausted:\n'
        '                pass\n'
        '        order = chosen + [n for n in applicable if n not in chosen]\n'
        '        cands = self.store.expand(plan, ctx, order, cap)\n'
        '        for c in cands:\n'
        '            c.proposed_by = "agent" if c.directive in chosen else "rules"\n'
        '        return cands[:cap]\n'
    ),
    'core/optimizer.py': (
        '"""PlannerLLM - Algorithm 1 of the methodology, plus the comparison systems.\n'
        '\n'
        '    1: cands <- {P} U ApplyDirectives(P, D)\n'
        '    2: for c in cands: c.score <- C^(c) - lambda * M.predAccuracyGain(c)\n'
        '    3: sort cands by score; take top-m\n'
        '    4: for c in top-m:\n'
        '    5:    if M.signatureHit(c): verdict <- M.lookup(c)          (no LLM call)\n'
        '    6:    elif VoI(c, incumbent) > theta * tokenCost(validation):\n'
        '    7:        verdict <- ValidationAgent(c, sample)\n'
        '    8:        M.write(c, contextSignature, verdict)\n'
        '    9:    if verdict == prefer(c): incumbent <- c\n'
        '   10: return incumbent\n'
        '\n'
        "The loop is repeated for a few rounds from the new incumbent (DocETL's\n"
        'opportunistic top-down traversal).  Every decision is recorded as an event;\n'
        'the UI streams them live and the report quotes them as transcripts.\n'
        '"""\n'
        'from __future__ import annotations\n'
        '\n'
        'import random\n'
        'import time\n'
        'from dataclasses import asdict, dataclass, field\n'
        'from typing import Callable\n'
        '\n'
        'from ..llm.base import BudgetExhausted\n'
        'from .directives import Candidate\n'
        'from .pipeline import Pipeline\n'
        'from .sampling import sample_queries\n'
        'from .session import Session, mean_observed\n'
        'from .signature import context_features, numeric_vector, signature_of\n'
        'from .validation import Verdict\n'
        '\n'
        'EventFn = Callable[[dict], None]\n'
        '\n'
        '\n'
        '@dataclass\n'
        'class AuditEntry:\n'
        '    round: int\n'
        '    directive: str\n'
        '    detail: str\n'
        '    plan: str\n'
        '    score: float\n'
        '    cost_usd: float\n'
        '    source: str           # memory | validation | gate-skip | cost-only | budget | random | tournament\n'
        '    verdict: str          # prefer | reject | skip\n'
        '    note: str = ""\n'
        '    gate: dict | None = None\n'
        '    memory: dict | None = None\n'
        '    validation: dict | None = None\n'
        '    signature: str = ""\n'
        '\n'
        '    def to_dict(self) -> dict:\n'
        '        return asdict(self)\n'
        '\n'
        '\n'
        '@dataclass\n'
        'class OptimizationResult:\n'
        '    system: str\n'
        '    task_id: str\n'
        '    base: Pipeline\n'
        '    plan: Pipeline\n'
        '    audit: list[AuditEntry] = field(default_factory=list)\n'
        '    events: list[dict] = field(default_factory=list)\n'
        '    opt_tokens: int = 0\n'
        '    opt_usd: float = 0.0\n'
        '    opt_calls: int = 0\n'
        '    opt_api_calls: int = 0\n'
        '    n_candidates: int = 0\n'
        '    n_validations: int = 0\n'
        '    n_memory_hits: int = 0\n'
        '    n_gate_skips: int = 0\n'
        '    wall_s: float = 0.0\n'
        '    budget_exhausted: bool = False\n'
        '\n'
        '    def summary(self) -> dict:\n'
        '        return {k: v for k, v in asdict(self).items() if k not in ("base", "plan", "audit", "events")} | {\n'
        '            "plan": self.plan.describe(), "base": self.base.describe(), "lineage": self.plan.lineage,\n'
        '            "plan_yaml": self.plan.to_yaml(), "plan_ops": self.plan.to_dict()["operators"],\n'
        '            "base_ops": self.base.to_dict()["operators"]}\n'
        '\n'
        '\n'
        'def diverse_top(scored: list, m: int) -> list:\n'
        '    """Top-m by score, but one candidate per directive first, so two variants of\n'
        '    the same rewrite (e.g. k/2 and 2k) cannot crowd every other idea out."""\n'
        '    first, rest, seen = [], [], set()\n'
        '    for item in scored:\n'
        '        (rest if item[1].directive in seen else first).append(item)\n'
        '        seen.add(item[1].directive)\n'
        '    picked = (first + rest)[:m]\n'
        '    return sorted(picked, key=lambda t: t[0])\n'
        '\n'
        '\n'
        'class _Run:\n'
        '    """Shared bookkeeping: ledger diff, budget window, event emission."""\n'
        '\n'
        '    def __init__(self, s: Session, system: str, task, on_event: EventFn | None) -> None:\n'
        '        self.s, self.system, self.task, self.on_event = s, system, task, on_event\n'
        '        self.base = s.workload.base_pipeline()\n'
        '        self.res = OptimizationResult(system, task.task_id, self.base, self.base)\n'
        '        self.t0 = time.perf_counter()\n'
        '        self.start = s.ledger.totals(optimization_only=True)\n'
        '        self._saved_budget = s.ledger.optimization_budget\n'
        '        if s.settings.budget.optimization_tokens is not None:\n'
        '            s.ledger.optimization_budget = s.ledger.optimization_tokens() + s.settings.budget.optimization_tokens\n'
        '        s.fresh_validator()\n'
        '\n'
        '    def emit(self, kind: str, **data) -> None:\n'
        '        ev = {"type": kind, "t": round(time.perf_counter() - self.t0, 3), **data}\n'
        '        self.res.events.append(ev)\n'
        '        if self.on_event:\n'
        '            self.on_event(ev)\n'
        '\n'
        '    def finish(self, plan: Pipeline) -> OptimizationResult:\n'
        '        end = self.s.ledger.totals(optimization_only=True)\n'
        '        r = self.res\n'
        '        r.plan = plan\n'
        '        r.opt_tokens = end["tokens"] - self.start["tokens"]\n'
        '        r.opt_usd = end["usd"] - self.start["usd"]\n'
        '        r.opt_calls = end["calls"] - self.start["calls"]\n'
        '        r.opt_api_calls = end["api_calls"] - self.start["api_calls"]\n'
        '        r.wall_s = time.perf_counter() - self.t0\n'
        '        self.s.ledger.optimization_budget = self._saved_budget\n'
        '        self.emit("done", plan=plan.describe(), lineage=plan.lineage, tokens=r.opt_tokens,\n'
        '                  validations=r.n_validations, memory_hits=r.n_memory_hits, gate_skips=r.n_gate_skips)\n'
        '        return r\n'
        '\n'
        '\n'
        'class PlannerLLM:\n'
        '    """The proposed optimizer with its ablation switches."""\n'
        '\n'
        '    def __init__(self, session: Session, *, use_gate: bool = True, use_memory: bool = True,\n'
        '                 use_validation: bool = True, shadow_gate: bool = False, audit_memory: bool = False,\n'
        '                 cost_mode: str | None = None, theta: float | None = None, name: str = "plannerllm") -> None:\n'
        '        self.s = session\n'
        '        self.use_gate, self.use_memory, self.use_validation = use_gate, use_memory, use_validation\n'
        "        self.shadow_gate = shadow_gate        # log the gate's decision but validate anyway (Sec. 10.3)\n"
        '        self.audit_memory = audit_memory      # re-validate memory hits to score reuse accuracy (Sec. 10.4)\n'
        '        self.cost_mode = cost_mode\n'
        '        self.theta = theta\n'
        '        self.name = name\n'
        '        self.memory_audit: list[dict] = []\n'
        '\n'
        '    def _cost(self, plan: Pipeline, stats: dict):\n'
        '        return self.s.cost_model.predict(plan, stats, self.cost_mode)\n'
        '\n'
        '    def _log_telemetry(self, plan: Pipeline, stats: dict, results) -> None:\n'
        '        if not results:\n'
        '            return\n'
        '        cm = self.s.cost_model\n'
        '        sym = cm.symbolic(plan, stats)\n'
        '        usd, ms = mean_observed(self.s, results)\n'
        '        self.s.memory.add_telemetry(self.s.workload.name, plan.fingerprint(), cm.features(plan, stats, sym),\n'
        '                                    sym.usd, sym.latency_ms, usd, ms)\n'
        '\n'
        '    # ------------------------------------------------------------------\n'
        '    def optimize(self, task, on_event: EventFn | None = None) -> OptimizationResult:\n'
        '        s, cfg = self.s, self.s.settings\n'
        '        run = _Run(s, self.name, task, on_event)\n'
        '        stats, fam = task.stats, s.workload.family\n'
        '        ctx = s.directive_context(stats)\n'
        '        incumbent = run.base\n'
        '        cost_inc = self._cost(incumbent, stats)\n'
        '        tried = {incumbent.fingerprint()}\n'
        '        run.emit("start", task=task.task_id, plan=incumbent.describe(), yaml=incumbent.to_yaml(),\n'
        '                 ops=incumbent.to_dict()["operators"],\n'
        '                 cost=cost_inc.to_dict(), stats=stats)\n'
        '        try:\n'
        '            for rnd in range(cfg.search.rounds):\n'
        '                cands = [c for c in s.agent.propose(incumbent, ctx, cfg.search.max_candidates)\n'
        '                         if c.plan.fingerprint() not in tried]\n'
        '                if not cands:\n'
        '                    break\n'
        '                run.res.n_candidates += len(cands)\n'
        '                scored = []\n'
        '                for c in cands:\n'
        '                    cost = self._cost(c.plan, stats)\n'
        "                    # the agent's nomination is a weak prior on accuracy gain\n"
        '                    prior = cfg.gate.prior_gain * (1.5 if c.proposed_by == "agent" else 1.0)\n'
        '                    gain = (s.memory.predicted_gain(c.directive, fam, prior)[0]\n'
        '                            if self.use_memory else prior)\n'
        '                    score = cfg.gate.reuse_runs * cost.usd - cfg.search.lambda_gain * cfg.gate.w_acc * gain\n'
        '                    scored.append((score, c, cost, gain))\n'
        '                scored.sort(key=lambda t: t[0])\n'
        '                run.emit("candidates", round=rnd, proposal=s.agent.last_proposal, items=[\n'
        '                    {"directive": c.directive, "detail": c.detail, "plan": c.plan.describe(),\n'
        '                     "by": c.proposed_by, "cost_usd": cost.usd, "latency_ms": cost.latency_ms,\n'
        '                     "pred_gain": gain, "score": sc,\n'
        '                     "shortlisted": any(c is x[1] for x in diverse_top(scored, cfg.search.top_m))}\n'
        '                    for sc, c, cost, gain in scored])\n'
        '                shortlist = diverse_top(scored, cfg.search.top_m)\n'
        '                changed = False\n'
        '                for score, c, cost_c, _ in shortlist:\n'
        '                    tried.add(c.plan.fingerprint())\n'
        '                    verdict, entry = self._decide(run, rnd, c, incumbent, cost_c, cost_inc, score, task)\n'
        '                    run.res.audit.append(entry)\n'
        '                    if verdict is None:\n'
        '                        continue\n'
        '                    if verdict.prefer_candidate:\n'
        '                        incumbent, cost_inc, changed = c.plan, cost_c, True\n'
        '                        run.emit("incumbent", round=rnd, plan=incumbent.describe(), directive=c.directive,\n'
        '                                 yaml=incumbent.to_yaml(), ops=incumbent.to_dict()["operators"])\n'
        '                retrain = s.memory.maybe_retrain(s.cost_model)\n'
        '                if retrain:\n'
        '                    run.emit("retrain", **{k: v for k, v in retrain.items() if k != "gate_calibration"})\n'
        '                if not changed:\n'
        '                    break\n'
        '        except BudgetExhausted as exc:\n'
        '            run.res.budget_exhausted = True\n'
        '            run.emit("budget", message=str(exc))\n'
        '        return run.finish(incumbent)\n'
        '\n'
        '    # ------------------------------------------------------------------\n'
        '    def _decide(self, run: _Run, rnd: int, c: Candidate, inc: Pipeline, cost_c, cost_inc, score, task):\n'
        '        s, cfg = self.s, self.s.settings\n'
        '        stats, fam = task.stats, s.workload.family\n'
        '        feats = context_features(inc, c.plan, c.directive, stats)\n'
        '        sig, vec = signature_of(feats), numeric_vector(stats, c.plan)\n'
        '        entry = AuditEntry(rnd, c.directive, c.detail, c.plan.describe(), score, cost_c.usd,\n'
        '                           source="", verdict="", signature=sig)\n'
        '\n'
        '        hit = (s.memory.lookup(sig, vec, c.directive, feats["parent_shape"], fam, feats)\n'
        '               if self.use_memory else None)\n'
        '        if hit is not None and s.memory.usable(hit):\n'
        '            run.res.n_memory_hits += 1\n'
        '            s.memory.touch(hit.sig)\n'
        '            v = Verdict(hit.prefer, source="memory", confidence=hit.confidence,\n'
        '                        note=f"{hit.match} signature hit, confidence {hit.confidence:.2f} "\n'
        '                             f"(age {hit.age_days:.1f}d, drift {hit.drift:.2f}, n={hit.n_obs})")\n'
        '            entry.source, entry.verdict, entry.note, entry.memory = "memory", \\\n'
        '                "prefer" if v.prefer_candidate else "reject", v.note, hit.to_dict()\n'
        '            run.emit("memory_hit", round=rnd, plan=c.plan.describe(), **hit.to_dict())\n'
        '            if self.audit_memory and self.use_validation:\n'
        '                sample = sample_queries(task.opt_queries, c.plan, stats, cfg.sampling, salt=task.task_id)\n'
        '                fresh = s.validator.validate(c.plan, inc, sample)\n'
        '                self.memory_audit.append({"task": task.task_id, "directive": c.directive,\n'
        '                                          "match": hit.match, "confidence": hit.confidence,\n'
        '                                          "reused": hit.prefer, "fresh": fresh.prefer_candidate})\n'
        '            return v, entry\n'
        '\n'
        '        if not self.use_validation:   # cost-only baseline: trust the cost model alone\n'
        '            v = Verdict(cost_c.usd < cost_inc.usd, source="cost-only",\n'
        '                        note=f"cost ${cost_c.usd:.6f} vs incumbent ${cost_inc.usd:.6f}")\n'
        '            entry.source, entry.verdict, entry.note = "cost-only", "prefer" if v.prefer_candidate else "reject", v.note\n'
        '            return v, entry\n'
        '\n'
        '        sample = sample_queries(task.opt_queries, c.plan, stats, cfg.sampling, salt=task.task_id)\n'
        '        judge_p = s.judge.prefer_probability(c.plan, inc, sample[:4]) if (self.use_gate or self.shadow_gate) else None\n'
        '        crit_n = len(s.validator._criteria or []) or cfg.validation.max_criteria\n'
        '        gd = s.gate.decide(c.directive, fam, c.plan, inc, stats, cost_c, cost_inc, judge_p,\n'
        '                           len(sample), crit_n, theta=self.theta)\n'
        '        entry.gate = gd.to_dict()\n'
        '        gate_active = self.use_gate and not self.shadow_gate\n'
        '        log_id = s.memory.log_gate(c.directive, gd.p_raw, gd.p_flip, gd.voi, gd.threshold,\n'
        '                                   gd.validate, None, s.workload.name)\n'
        '        run.emit("gate", round=rnd, directive=c.directive, plan=c.plan.describe(), **gd.to_dict(),\n'
        '                 enforced=gate_active)\n'
        '        if gate_active and not gd.validate:\n'
        '            run.res.n_gate_skips += 1\n'
        '            entry.source, entry.verdict, entry.note = "gate-skip", "skip", gd.rationale\n'
        '            return None, entry\n'
        '\n'
        '        run.emit("validating", round=rnd, directive=c.directive, n_items=len(sample))\n'
        '        v = s.validator.validate(c.plan, inc, sample)\n'
        '        run.res.n_validations += 1\n'
        '        flipped = v.prefer_candidate\n'
        '        s.memory.set_gate_outcome(log_id, flipped)\n'
        '        s.memory.update_flip(c.directive, flipped)\n'
        '        if judge_p is not None and abs(judge_p - 0.5) > 0.05:\n'
        '            s.memory.update_agreement(c.directive, (judge_p > 0.5) == flipped)\n'
        '        s.memory.write(sig, feats, vec, c.directive, flipped, v.acc_delta, cost_c.usd - cost_inc.usd,\n'
        '                       s.workload.name)\n'
        '        self._log_telemetry(c.plan, stats, s.validator.outputs(c.plan, sample))\n'
        '        self._log_telemetry(inc, stats, s.validator.outputs(inc, sample))\n'
        '        entry.source, entry.verdict, entry.note = "validation", "prefer" if flipped else "reject", v.note\n'
        '        entry.validation = v.to_dict()\n'
        '        run.emit("validation", round=rnd, directive=c.directive, plan=c.plan.describe(), **{\n'
        '            k: val for k, val in v.to_dict().items() if k != "criteria"}, criteria=[c_["name"] for c_ in v.criteria])\n'
        '        return v, entry\n'
        '\n'
        '\n'
        '# ====================================================================== baselines\n'
        'class NoOptimization:\n'
        '    """B2: run the user\'s pipeline as written."""\n'
        '\n'
        '    name = "no_opt"\n'
        '\n'
        '    def __init__(self, session: Session) -> None:\n'
        '        self.s = session\n'
        '\n'
        '    def optimize(self, task, on_event=None) -> OptimizationResult:\n'
        '        run = _Run(self.s, self.name, task, on_event)\n'
        '        return run.finish(run.base)\n'
        '\n'
        '\n'
        'class RandomRewrite:\n'
        '    """B3: apply one uniformly random applicable directive, no evaluation."""\n'
        '\n'
        '    name = "random"\n'
        '\n'
        '    def __init__(self, session: Session, seed: int = 1) -> None:\n'
        '        self.s, self.seed = session, seed\n'
        '\n'
        '    def optimize(self, task, on_event=None) -> OptimizationResult:\n'
        '        run = _Run(self.s, self.name, task, on_event)\n'
        '        ctx = self.s.directive_context(task.stats)\n'
        '        ctx.llm = None   # random choice must not spend tokens\n'
        '        cands = self.s.directives.expand(run.base, ctx)\n'
        '        if not cands:\n'
        '            return run.finish(run.base)\n'
        '        pick = random.Random(f"{self.seed}:{task.task_id}").choice(cands)\n'
        '        run.res.n_candidates = len(cands)\n'
        '        run.res.audit.append(AuditEntry(0, pick.directive, pick.detail, pick.plan.describe(), 0.0, 0.0,\n'
        '                                        "random", "prefer", "picked uniformly at random"))\n'
        '        return run.finish(pick.plan)\n'
        '\n'
        '\n'
        'class DocETLStyle:\n'
        '    """B1: DocETL\'s agentic optimizer - validate every candidate, pairwise top-A.\n'
        '\n'
        '    No cost ranking, no gate, no memory: each round all candidates are run on\n'
        '    the sample, rated, and the top A are compared pairwise; the winner becomes\n'
        '    the incumbent for the next round.\n'
        '    """\n'
        '\n'
        '    name = "docetl"\n'
        '\n'
        '    def __init__(self, session: Session) -> None:\n'
        '        self.s = session\n'
        '\n'
        '    def optimize(self, task, on_event=None) -> OptimizationResult:\n'
        '        s, cfg = self.s, self.s.settings\n'
        '        run = _Run(s, self.name, task, on_event)\n'
        '        ctx = s.directive_context(task.stats)\n'
        '        incumbent = run.base\n'
        '        try:\n'
        '            for rnd in range(cfg.search.rounds):\n'
        '                cands = s.agent.propose(incumbent, ctx, cfg.search.max_candidates)\n'
        '                if not cands:\n'
        '                    break\n'
        '                run.res.n_candidates += len(cands)\n'
        '                sample = sample_queries(task.opt_queries, incumbent, task.stats, cfg.sampling, salt=task.task_id)\n'
        '                best, table = s.validator.tournament([incumbent] + [c.plan for c in cands], sample)\n'
        '                run.res.n_validations += len(table)\n'
        '                for c in cands:\n'
        '                    run.res.audit.append(AuditEntry(rnd, c.directive, c.detail, c.plan.describe(), 0.0, 0.0,\n'
        '                                                    "tournament", "prefer" if c.plan is best else "reject"))\n'
        '                run.emit("tournament", round=rnd, table=table)\n'
        '                if best is incumbent:\n'
        '                    break\n'
        '                incumbent = best\n'
        '        except BudgetExhausted as exc:\n'
        '            run.res.budget_exhausted = True\n'
        '            run.emit("budget", message=str(exc))\n'
        '        return run.finish(incumbent)\n'
        '\n'
        '\n'
        'def make_system(name: str, session: Session, **kw):\n'
        '    """Factory for every system / ablation used in the experiments."""\n'
        '    table = {\n'
        '        "no_opt": lambda: NoOptimization(session),\n'
        '        "random": lambda: RandomRewrite(session),\n'
        '        "cost_only": lambda: PlannerLLM(session, use_validation=False, use_memory=False, name="cost_only"),\n'
        '        "docetl": lambda: DocETLStyle(session),\n'
        '        "plannerllm": lambda: PlannerLLM(session, name="plannerllm", **kw),\n'
        '        "no_gate": lambda: PlannerLLM(session, use_gate=False, name="no_gate", **kw),\n'
        '        "no_memory": lambda: PlannerLLM(session, use_memory=False, name="no_memory", **kw),\n'
        '        "symbolic_only": lambda: PlannerLLM(session, cost_mode="symbolic", name="symbolic_only", **kw),\n'
        '        "neural_only": lambda: PlannerLLM(session, cost_mode="neural", name="neural_only", **kw),\n'
        '    }\n'
        '    if name not in table:\n'
        '        raise KeyError(f"unknown system \'{name}\'")\n'
        '    return table[name]()\n'
        '\n'
        '\n'
        'SYSTEMS = ["no_opt", "random", "cost_only", "docetl", "plannerllm"]\n'
        'ABLATIONS = ["plannerllm", "no_gate", "no_memory", "symbolic_only", "neural_only"]\n'
    ),
    'core/gate.py': (
        '"""Value-of-information gate (Section 5.4) - the economic core.\n'
        '\n'
        'For candidate c against incumbent p:\n'
        '\n'
        '    P(flip) = alpha * h(dir(c)) + (1 - alpha) * A\n'
        '    A       = a * j + (1 - a) * (1 - j)\n'
        '    dV      = w_acc * dAcc^ - |dC|\n'
        '    VoI     = P(flip) * |dV|\n'
        '    validate  iff  VoI > theta * tokenCost(validation)\n'
        '\n'
        "h(dir) is the directive's historical flip rate from plan memory.  The\n"
        "agreement term A turns the cheap local judge's opinion j = P_judge(c > p)\n"
        'into a flip estimate, trusting it only as much as the judge has historically\n'
        'agreed with the LLM validator (rate a).  With no history a = 0.5, so A = 0.5\n'
        'and the term is uninformative, as it should be.\n'
        '\n'
        "dAcc^ is the memory's prediction of the accuracy gain (clipped at 0 so a\n"
        'candidate expected to be worse never *buys* validation through |dV|), and dC\n'
        'is the lifetime cost difference over ``reuse_runs`` executions.  Both sides\n'
        'of the inequality are in US dollars.\n'
        '"""\n'
        'from __future__ import annotations\n'
        '\n'
        'from dataclasses import asdict, dataclass\n'
        '\n'
        'from .cost_model import CostEstimate, HybridCostModel\n'
        'from .memory import PlanMemory\n'
        'from .pipeline import Pipeline\n'
        '\n'
        '\n'
        '@dataclass\n'
        'class GateDecision:\n'
        '    validate: bool\n'
        '    voi: float\n'
        '    threshold: float\n'
        '    p_flip: float\n'
        '    p_raw: float\n'
        '    h_dir: float\n'
        '    h_trials: int\n'
        '    judge_p: float | None\n'
        '    agreement: float\n'
        '    d_acc: float\n'
        '    d_cost: float\n'
        '    d_value: float\n'
        '    val_cost_usd: float\n'
        '    val_cost_tokens: float\n'
        '    rationale: str = ""\n'
        '\n'
        '    def to_dict(self) -> dict:\n'
        '        return asdict(self)\n'
        '\n'
        '\n'
        'class VoIGate:\n'
        '    def __init__(self, gate_settings, memory: PlanMemory, cost_model: HybridCostModel) -> None:\n'
        '        self.cfg = gate_settings\n'
        '        self.memory = memory\n'
        '        self.cost = cost_model\n'
        '\n'
        '    def flip_probability(self, directive: str, judge_p: float | None) -> tuple[float, float, float, int, float]:\n'
        '        h, trials = self.memory.flip_rate(directive)\n'
        '        a, n_agree = self.memory.agreement_rate(directive)\n'
        '        if judge_p is None or n_agree < self.cfg.min_agreement_samples:\n'
        '            term = 0.5\n'
        '        else:\n'
        '            term = a * judge_p + (1 - a) * (1 - judge_p)\n'
        '        p_raw = self.cfg.alpha * h + (1 - self.cfg.alpha) * term\n'
        '        p = self.memory.calibrated(p_raw) if self.cfg.calibrate else p_raw\n'
        '        return min(max(p, 0.01), 0.99), p_raw, h, trials, a\n'
        '\n'
        '    def decide(self, directive: str, family: str, cand: Pipeline, inc: Pipeline, stats: dict,\n'
        '               cost_c: CostEstimate, cost_p: CostEstimate, judge_p: float | None,\n'
        '               n_items: int, n_criteria: int, theta: float | None = None) -> GateDecision:\n'
        '        theta = self.cfg.theta if theta is None else theta\n'
        '        p_flip, p_raw, h, trials, a = self.flip_probability(directive, judge_p)\n'
        '        gain, _ = self.memory.predicted_gain(directive, family, self.cfg.prior_gain)\n'
        '        d_acc = max(gain, 0.0)\n'
        '        d_cost = self.cfg.reuse_runs * (cost_c.usd - cost_p.usd)\n'
        '        d_value = self.cfg.w_acc * d_acc - abs(d_cost)\n'
        '        voi = p_flip * abs(d_value)\n'
        '        val_usd, val_tok = self.cost.validation_cost(cand, inc, stats, n_items, n_criteria)\n'
        '        threshold = theta * val_usd\n'
        '        ok = voi > threshold\n'
        '        why = (f"P(flip)={p_flip:.2f} [h={h:.2f} over {trials}, judge={\'-\' if judge_p is None else f\'{judge_p:.2f}\'},"\n'
        '               f" agree={a:.2f}] x |dV|=${abs(d_value):.5f} (dAcc^={d_acc:+.3f}, dC=${d_cost:+.5f})"\n'
        '               f" = VoI ${voi:.5f} {\'>\' if ok else \'<=\'} theta*tokenCost ${threshold:.5f}"\n'
        '               f" -> {\'VALIDATE\' if ok else \'SKIP\'}")\n'
        '        return GateDecision(ok, voi, threshold, p_flip, p_raw, h, trials, judge_p, a, d_acc, d_cost,\n'
        '                            d_value, val_usd, val_tok, why)\n'
    ),
    'core/judge.py': (
        '"""Local judge: a small CPU model that gives the gate a cheap opinion.\n'
        '\n'
        'The judge runs only the *non-LLM prefix* of each plan (filters, ANN probe,\n'
        'rerank, schema linking) on the validation sample and scores what each plan\n'
        'would hand to the LLM with the MiniLM cross-encoder (22M parameters, well\n'
        'inside the <=3B budget of Section 6).  It costs no API tokens.  When two\n'
        'plans differ only in their LLM operators (prompt rewrite, decompose,\n'
        'few-shot...) the prefixes are identical and the judge abstains (returns\n'
        'None), which the gate treats as "no information".\n'
        '"""\n'
        'from __future__ import annotations\n'
        '\n'
        'import math\n'
        '\n'
        'import numpy as np\n'
        '\n'
        'from .pipeline import Pipeline\n'
        '\n'
        '\n'
        'def cheap_prefix(plan: Pipeline) -> Pipeline:\n'
        '    ops = []\n'
        '    for o in plan.topological():\n'
        '        if o.uses_llm:\n'
        '            break\n'
        '        ops.append(o)\n'
        '    p = plan.clone()\n'
        '    p.operators = [o for o in p.operators if o.name in {x.name for x in ops}]\n'
        '    return p\n'
        '\n'
        '\n'
        'class LocalJudge:\n'
        '    def __init__(self, workload, executor, temperature: float = 0.6, top: int = 3) -> None:\n'
        '        self.wl = workload\n'
        '        self.ex = executor\n'
        '        self.temperature = temperature\n'
        '        self.top = top\n'
        '        self.calls = 0\n'
        '\n'
        '    def _context_quality(self, plan: Pipeline, queries: list[dict]) -> float | None:\n'
        '        prefix = cheap_prefix(plan)\n'
        '        if not prefix.operators:\n'
        '            return None\n'
        '        scores = []\n'
        '        for q in queries:\n'
        '            res = self.ex.run(prefix, q, purpose="judge")\n'
        '            texts = [self.wl.records[self.wl.id_to_pos[i]]["text"] for i in res.retrieved_ids[: self.top]]\n'
        '            if not texts:\n'
        '                scores.append(-10.0)\n'
        '                continue\n'
        '            s = self.wl.reranker.score(q["question"], texts)\n'
        '            scores.append(float(np.mean(np.sort(s)[::-1][: self.top])))\n'
        '        return float(np.mean(scores)) if scores else None\n'
        '\n'
        '    def prefer_probability(self, cand: Pipeline, inc: Pipeline, queries: list[dict]) -> float | None:\n'
        '        """P_judge(candidate better than incumbent), or None when uninformative."""\n'
        '        if cheap_prefix(cand).fingerprint() == cheap_prefix(inc).fingerprint():\n'
        '            return None\n'
        '        self.calls += 1\n'
        '        sc = self._context_quality(cand, queries)\n'
        '        sp = self._context_quality(inc, queries)\n'
        '        if sc is None or sp is None:\n'
        '            return None\n'
        '        return 1.0 / (1.0 + math.exp(-(sc - sp) / self.temperature))\n'
    ),
    'core/sampling.py': (
        '"""Selectivity-aware, size-weighted sampling for optimisation-time validation.\n'
        '\n'
        'Following DocETL, the sample grows when downstream operators are selective\n'
        '(a filter or an extract-map that drops most records would otherwise leave\n'
        'too few informative outputs), and items are drawn with probability that\n'
        'grows with document/question length so long, hard items are not starved.\n'
        '"""\n'
        'from __future__ import annotations\n'
        '\n'
        'import math\n'
        'import random\n'
        '\n'
        'from .pipeline import Pipeline\n'
        '\n'
        '\n'
        'def downstream_selectivity(plan: Pipeline, stats: dict) -> float:\n'
        '    sel = 1.0\n'
        '    for op in plan.topological():\n'
        '        if op.kind == "filter" or (op.kind == "retrieve" and op.params.get("where")):\n'
        '            sel *= max(stats.get("sel_est", 1.0), 0.02) ** 0.25   # soften: queries, not rows\n'
        '        if op.kind == "map":\n'
        '            sel *= 0.5\n'
        '    return max(sel, 0.1)\n'
        '\n'
        '\n'
        'def sample_size(n_pool: int, plan: Pipeline, stats: dict, cfg) -> int:\n'
        '    sel = downstream_selectivity(plan, stats)\n'
        '    n = math.ceil(max(cfg.min_items, cfg.frac * n_pool) / sel)\n'
        '    return int(min(max(n, cfg.min_items), cfg.max_items, n_pool))\n'
        '\n'
        '\n'
        'def sample_queries(queries: list[dict], plan: Pipeline, stats: dict, cfg, salt: str = "") -> list[dict]:\n'
        '    n = sample_size(len(queries), plan, stats, cfg)\n'
        '    rng = random.Random(f"{cfg.seed}:{salt}")\n'
        '    weights = [math.sqrt(len(q["question"]) + len(q.get("contract_text", "")) / 50 + 1) for q in queries]\n'
        '    pool = list(range(len(queries)))\n'
        '    picked: list[int] = []\n'
        '    while pool and len(picked) < n:   # weighted sampling without replacement\n'
        '        total = sum(weights[i] for i in pool)\n'
        '        r, acc = rng.random() * total, 0.0\n'
        '        for i in pool:\n'
        '            acc += weights[i]\n'
        '            if acc >= r:\n'
        '                picked.append(i)\n'
        '                pool.remove(i)\n'
        '                break\n'
        '    return [queries[i] for i in sorted(picked)]\n'
    ),
    'core/validation.py': (
        '"""Validation agent (the base-paper evaluation protocol, Sections 5.4 / 8).\n'
        '\n'
        "1. synthesise task-specific criteria for the pipeline's final operator;\n"
        '2. execute candidate and incumbent on a selectivity-aware sample;\n'
        '3. rate every output 1-5 per criterion with the LLM;\n'
        '4. when the average ratings are close, run a pairwise comparison (with the\n'
        '   A/B positions swapped on alternate items to cancel position bias);\n'
        '5. return a verdict plus a transcript the UI and report can show.\n'
        '\n'
        'The agent never sees reference answers - it judges outputs the way a human\n'
        'reviewer would, from the question and the context the plan used.\n'
        '"""\n'
        'from __future__ import annotations\n'
        '\n'
        'import time\n'
        'from dataclasses import asdict, dataclass, field\n'
        '\n'
        'from ..engine import prompts\n'
        'from ..engine.executor import ExecResult, Executor\n'
        'from ..llm.base import BudgetExhausted\n'
        'from .pipeline import Pipeline\n'
        '\n'
        '\n'
        '@dataclass\n'
        'class Verdict:\n'
        '    prefer_candidate: bool\n'
        '    score_c: float = 0.0\n'
        '    score_p: float = 0.0\n'
        '    criteria: list = field(default_factory=list)\n'
        '    pairwise_win_rate: float | None = None\n'
        '    source: str = "validation"     # validation | memory | gate-skip | cost-only | budget\n'
        '    confidence: float = 1.0\n'
        '    tokens: int = 0\n'
        '    note: str = ""\n'
        '    items: list = field(default_factory=list)\n'
        '\n'
        '    @property\n'
        '    def acc_delta(self) -> float:\n'
        '        """Rating gap mapped onto the 0-1 accuracy scale used by the gate."""\n'
        '        return (self.score_c - self.score_p) / 4.0\n'
        '\n'
        '    def to_dict(self) -> dict:\n'
        '        d = asdict(self)\n'
        '        d["acc_delta"] = self.acc_delta\n'
        '        return d\n'
        '\n'
        '\n'
        'def output_text(res: ExecResult) -> str:\n'
        '    if res.sql is not None:\n'
        '        rows = res.rows[:5] if res.rows else res.rows\n'
        '        return f"SQL: {res.sql}\\nResult (first rows): {rows}\\nError: {res.error or \'none\'}"\n'
        '    return res.answer or "(empty)"\n'
        '\n'
        '\n'
        'def context_text(res: ExecResult, limit: int = 3, chars: int = 350) -> str:\n'
        '    return "\\n".join(f"- {c[:chars]}" for c in res.context[:limit]) or "(none)"\n'
        '\n'
        '\n'
        'class ValidationAgent:\n'
        '    def __init__(self, llm, executor: Executor, settings, task_text: str, final_kind: str) -> None:\n'
        '        self.llm = llm\n'
        '        self.ex = executor\n'
        '        self.cfg = settings.validation\n'
        '        self.task_text = task_text\n'
        '        self.final_kind = final_kind\n'
        '        self._criteria: list[dict] | None = None\n'
        '        self._outputs: dict[tuple[str, str], ExecResult] = {}\n'
        '        self._ratings: dict[tuple[str, str], float] = {}\n'
        '\n'
        '    # ------------------------------------------------------------ steps\n'
        '    def criteria(self) -> list[dict]:\n'
        '        if self._criteria is None:\n'
        '            msg = prompts.CRITERIA.format(task=self.task_text, kind=self.final_kind, k=self.cfg.max_criteria)\n'
        '            r = self.llm.complete(msg, purpose="criteria", task="criteria",\n'
        '                                  payload={"max_criteria": self.cfg.max_criteria},\n'
        '                                  system=prompts.SYSTEM_AGENT, max_tokens=300, json_mode=True)\n'
        '            crits = (r.json({}) or {}).get("criteria") or []\n'
        '            crits = [c for c in crits if isinstance(c, dict) and c.get("name")][: self.cfg.max_criteria]\n'
        '            self._criteria = crits or [{"name": "accuracy", "description": "the output answers the question correctly"},\n'
        '                                       {"name": "faithfulness", "description": "the output is supported by the context"}]\n'
        '        return self._criteria\n'
        '\n'
        '    def outputs(self, plan: Pipeline, sample: list[dict]) -> list[ExecResult]:\n'
        '        fp = plan.fingerprint()\n'
        '        out = []\n'
        '        for q in sample:\n'
        '            key = (fp, q["qid"])\n'
        '            if key not in self._outputs:\n'
        '                self._outputs[key] = self.ex.run(plan, q, purpose="validation_exec")\n'
        '            out.append(self._outputs[key])\n'
        '        return out\n'
        '\n'
        '    def _rate(self, plan_fp: str, q: dict, res: ExecResult) -> float:\n'
        '        key = (plan_fp, q["qid"])\n'
        '        if key in self._ratings:\n'
        '            return self._ratings[key]\n'
        '        crits = self.criteria()\n'
        '        crit_txt = "\\n".join(f"- {c[\'name\']}: {c.get(\'description\', \'\')}" for c in crits)\n'
        '        msg = prompts.RATE.format(task=self.task_text, question=q["question"], context=context_text(res),\n'
        '                                  output=output_text(res), criteria=crit_txt)\n'
        '        r = self.llm.complete(msg, purpose="validation", task="rate", system=prompts.SYSTEM_AGENT,\n'
        '                              payload={"question": q["question"], "output": output_text(res),\n'
        '                                       "context": res.context, "criteria": [c["name"] for c in crits]},\n'
        '                              max_tokens=120, json_mode=True)\n'
        '        scores = (r.json({}) or {}).get("scores") or {}\n'
        '        vals = []\n'
        '        for v in scores.values():\n'
        '            try:\n'
        '                vals.append(min(5.0, max(1.0, float(v))))\n'
        '            except (TypeError, ValueError):\n'
        '                continue\n'
        '        score = sum(vals) / len(vals) if vals else 3.0\n'
        '        self._ratings[key] = score\n'
        '        return score\n'
        '\n'
        '    def _rate_many(self, plan_fp: str, sample: list[dict], results: list[ExecResult]) -> list[float]:\n'
        '        """Rate all not-yet-rated outputs of one plan in a single LLM call.\n'
        '\n'
        '        The judge still scores every item on every criterion; grouping the items\n'
        '        only saves round trips, which matters under a requests-per-day quota.\n'
        '        """\n'
        '        todo = [(q, r) for q, r in zip(sample, results) if (plan_fp, q["qid"]) not in self._ratings]\n'
        '        if todo:\n'
        '            crits = self.criteria()\n'
        '            names = [c["name"] for c in crits]\n'
        '            crit_txt = "\\n".join(f"- {c[\'name\']}: {c.get(\'description\', \'\')}" for c in crits)\n'
        '            items = "\\n\\n".join(\n'
        '                f"Item {k}\\nQuestion: {q[\'question\']}\\nContext the pipeline used:\\n{context_text(r, chars=300)}\\n"\n'
        '                f"Output:\\n{output_text(r)[:500]}" for k, (q, r) in enumerate(todo, 1))\n'
        '            msg = prompts.RATE_BATCH.format(task=self.task_text, criteria=crit_txt, items=items, n=len(todo))\n'
        '            r = self.llm.complete(msg, purpose="validation", task="rate_batch", system=prompts.SYSTEM_AGENT,\n'
        '                                  payload={"items": [{"question": q["question"], "output": output_text(x),\n'
        '                                                      "context": x.context} for q, x in todo], "criteria": names},\n'
        '                                  max_tokens=60 + 45 * len(todo), json_mode=True)\n'
        '            parsed = (r.json({}) or {}).get("items") or []\n'
        '            by_id = {}\n'
        '            for it in parsed if isinstance(parsed, list) else []:\n'
        '                try:\n'
        '                    by_id[int(it.get("id"))] = it.get("scores") or {}\n'
        '                except (TypeError, ValueError, AttributeError):\n'
        '                    continue\n'
        '            for k, (q, _) in enumerate(todo, 1):\n'
        '                vals = []\n'
        '                for v in (by_id.get(k) or {}).values():\n'
        '                    try:\n'
        '                        vals.append(min(5.0, max(1.0, float(v))))\n'
        '                    except (TypeError, ValueError):\n'
        '                        continue\n'
        '                self._ratings[(plan_fp, q["qid"])] = sum(vals) / len(vals) if vals else 3.0\n'
        '        return [self._ratings[(plan_fp, q["qid"])] for q in sample]\n'
        '\n'
        '    def _pairwise(self, q: dict, a: ExecResult, b: ExecResult, swap: bool) -> bool:\n'
        '        """True if the *candidate* (a) wins."""\n'
        '        first, second = (b, a) if swap else (a, b)\n'
        '        msg = prompts.PAIRWISE.format(task=self.task_text, question=q["question"],\n'
        '                                      context=context_text(a), a=output_text(first), b=output_text(second))\n'
        '        r = self.llm.complete(msg, purpose="pairwise", task="pairwise", system=prompts.SYSTEM_AGENT,\n'
        '                              payload={"question": q["question"], "context": a.context + b.context,\n'
        '                                       "output_a": output_text(first), "output_b": output_text(second)},\n'
        '                              max_tokens=80, json_mode=True)\n'
        '        winner = str((r.json({}) or {}).get("winner", "A")).strip().upper()[:1]\n'
        '        return (winner == "B") if swap else (winner == "A")\n'
        '\n'
        '    # ------------------------------------------------------------ public API\n'
        '    def validate(self, cand: Pipeline, inc: Pipeline, sample: list[dict]) -> Verdict:\n'
        '        t0 = time.perf_counter()\n'
        '        before = self.llm.ledger.optimization_tokens()\n'
        '        crits = self.criteria()\n'
        '        out_c, out_p = self.outputs(cand, sample), self.outputs(inc, sample)\n'
        '        fc, fp = cand.fingerprint(), inc.fingerprint()\n'
        '        sc_all, sp_all = self._rate_many(fc, sample, out_c), self._rate_many(fp, sample, out_p)\n'
        '        items = []\n'
        '        for q, rc, rp, sc, sp in zip(sample, out_c, out_p, sc_all, sp_all):\n'
        '            items.append({"qid": q["qid"], "question": q["question"], "candidate": output_text(rc)[:400],\n'
        '                          "incumbent": output_text(rp)[:400], "score_c": sc, "score_p": sp})\n'
        '        avg_c = sum(sc_all) / max(len(sc_all), 1)\n'
        '        avg_p = sum(sp_all) / max(len(sp_all), 1)\n'
        '        win_rate = None\n'
        '        if abs(avg_c - avg_p) < self.cfg.pairwise_margin:\n'
        '            # close call: pairwise comparison on the items where the plans disagree most\n'
        '            order = sorted(range(len(sample)), key=lambda i: -abs(sc_all[i] - sp_all[i]))\n'
        '            picks = order[: self.cfg.pairwise_items]\n'
        '            wins = 0\n'
        '            for n, i in enumerate(picks):\n'
        '                won = self._pairwise(sample[i], out_c[i], out_p[i], swap=bool(n % 2))\n'
        '                items[i]["pairwise"] = "candidate" if won else "incumbent"\n'
        '                wins += int(won)\n'
        '            win_rate = wins / max(len(picks), 1)\n'
        '            prefer = win_rate > 0.5 and avg_c >= avg_p - 0.25\n'
        '        else:\n'
        '            prefer = avg_c > avg_p\n'
        '        spent = self.llm.ledger.optimization_tokens() - before\n'
        '        note = (f"rating c={avg_c:.2f} vs p={avg_p:.2f}"\n'
        '                + (f", pairwise win-rate {win_rate:.2f}" if win_rate is not None else "")\n'
        '                + f", {len(sample)} items, {time.perf_counter() - t0:.1f}s")\n'
        '        return Verdict(prefer, avg_c, avg_p, crits, win_rate, "validation", 1.0, spent, note, items)\n'
        '\n'
        '    def tournament(self, plans: list[Pipeline], sample: list[dict]) -> tuple[Pipeline, list[dict]]:\n'
        '        """DocETL-style selection: rate every plan, pairwise round-robin over the top A."""\n'
        '        table = []\n'
        '        for plan in plans:\n'
        '            try:\n'
        '                outs = self.outputs(plan, sample)\n'
        '                fp = plan.fingerprint()\n'
        '                avg = sum(self._rate_many(fp, sample, outs)) / max(len(sample), 1)\n'
        '            except BudgetExhausted:\n'
        '                break\n'
        '            table.append({"plan": plan, "score": avg, "wins": 0})\n'
        '        if not table:\n'
        '            return plans[0], []\n'
        '        table.sort(key=lambda r: -r["score"])\n'
        '        top = table[: self.cfg.top_a]\n'
        '        try:\n'
        '            for i in range(len(top)):\n'
        '                for j in range(i + 1, len(top)):\n'
        '                    a, b = top[i], top[j]\n'
        '                    oa, ob = self.outputs(a["plan"], sample), self.outputs(b["plan"], sample)\n'
        '                    k = (i + j) % len(sample)\n'
        '                    if self._pairwise(sample[k], oa[k], ob[k], swap=bool((i + j) % 2)):\n'
        '                        a["wins"] += 1\n'
        '                    else:\n'
        '                        b["wins"] += 1\n'
        '        except BudgetExhausted:\n'
        '            pass\n'
        '        best = max(top, key=lambda r: (r["wins"], r["score"]))\n'
        '        return best["plan"], [{"plan": r["plan"].describe(), "score": r["score"], "wins": r["wins"]} for r in table]\n'
    ),
})
run_selected(['pareto', 'gate'])

plannerllm/core/directives.py
plannerllm/core/generation.py
plannerllm/core/optimizer.py
plannerllm/core/gate.py
plannerllm/core/judge.py
plannerllm/core/sampling.py
plannerllm/core/validation.py
pareto | seed 0 | 120 progress entries | reproduced_runs\recorded_groq\openai_gpt-oss-20b_seed0\pareto.json
   plannerllm | spider:world_1 | pass 1
   plannerllm | spider:orchestra | pass 1
   plannerllm | spider:world_1 | pass 1
pareto | seed 1 | 120 progress entries | reproduced_runs\recorded_groq\openai_gpt-oss-20b_seed1\pareto.json
   plannerllm | spider:dog_kennels | pass 1
   plannerllm | spider:pets_1 | pass 1
   plannerllm | spider:dog_kennels | pass 1
pareto | seed 2 | 120 progress entries | reproduced_runs\recorded_groq\openai_gpt-oss-20b_seed2\pareto.json
   plannerllm | spider:employee_hire_evaluation | pass 1
   plannerllm | spider:flight_2 | pass 1
   plannerllm | spider:employee_hire_evaluation | pass 1
gate | seed 0 | 12 progress entries | reproduced_runs\recorded_groq\openai_g

"""Rewrite directives (Section 5.1).

Each directive is a small, named plan transformation.  Logical directives
change *what* the LLM is asked to do (decompose, merge, prompt rewrite);
physical ones change *how* data is accessed (index family, top-k); the
retrieval-subplan family added by PlannerLLM moves predicates in and out of
the ANN probe and adds/removes a rerank stage ("and vice versa").  SQL
directives cover the text-to-SQL workloads.

A directive never decides whether its rewrite is good - that is the job of
the cost model, the plan memory and the gated validation agent.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Callable

from . import pipeline as P
from .pipeline import Operator, Pipeline

MAP_INSTRUCTION = ("Copy the sentence(s) from the text that help answer the question, word for word. "
                   "If none do, reply NONE.")


@dataclass
class DirectiveContext:
    workload: str
    family: str
    task_text: str
    filter_template: str
    stats: dict
    llm: object | None = None


@dataclass
class Candidate:
    plan: Pipeline
    directive: str
    detail: str = ""
    proposed_by: str = "rules"   # rules | agent


@dataclass
class Directive:
    name: str
    family: str          # retrieval | physical | logical | sql
    description: str
    fn: Callable[[Pipeline, DirectiveContext], list[tuple[Pipeline, str]]] = field(repr=False)

    def apply(self, plan: Pipeline, ctx: DirectiveContext) -> list[Candidate]:
        out = []
        for new, detail in self.fn(plan, ctx):
            new.lineage = plan.lineage + [self.name]
            try:
                new.validate()
            except P.PipelineError:
                continue
            out.append(Candidate(new, self.name, detail))
        return out


# ------------------------------------------------------------------ helpers
def _first_llm_op(plan: Pipeline) -> Operator | None:
    return next((o for o in plan.topological() if o.uses_llm), None)


def _terminal_llm(plan: Pipeline) -> Operator | None:
    llm_ops = [o for o in plan.topological() if o.kind in ("generate", "sql_generate")]
    return llm_ops[-1] if llm_ops else None


# ------------------------------------------------------------------ retrieval family
def add_rerank(plan: Pipeline, ctx: DirectiveContext):
    r = plan.find("retrieve")
    if r is None or plan.find("rerank") is not None or ctx.family == "sql-heavy":
        return []
    p = plan.clone()
    r2 = p.op(r.name)
    keep = int(r2.params.get("top_k", 10))
    r2.params["top_k"] = min(max(keep * 2, 20), 40)
    anchor = _first_llm_op(p)
    p.insert_before(anchor.name, Operator(p.unique_name("rerank"), "rerank", {"top_k": keep}))
    return [(p, f"retrieve top_k {keep}->{r2.params['top_k']}, cross-encoder keeps {keep}")]


def drop_rerank(plan: Pipeline, ctx: DirectiveContext):
    rr, r = plan.find("rerank"), plan.find("retrieve")
    if rr is None or r is None:
        return []
    p = plan.clone()
    p.op(r.name).params["top_k"] = int(rr.params.get("top_k", 5))
    p.remove(rr.name)
    return [(p, "single-stage probe, top_k = former rerank k")]


def push_filter_into_probe(plan: Pipeline, ctx: DirectiveContext):
    """filter (before or after) + retrieve  ==>  retrieve[where]  (filtered vector probe)."""
    r, f = plan.find("retrieve"), plan.find("filter")
    if r is None or f is None or r.params.get("where"):
        return []
    p = plan.clone()
    where = p.op(f.name).params["where"]
    p.remove(f.name)
    p.op(r.name).params["where"] = where
    return [(p, f"predicate '{where}' evaluated inside the ANN probe")]


def pull_filter_out_of_probe(plan: Pipeline, ctx: DirectiveContext):
    """retrieve[where]  ==>  retrieve(over-fetch) -> filter   (post-filtering)."""
    r = plan.find("retrieve")
    if r is None or not r.params.get("where"):
        return []
    p = plan.clone()
    r2 = p.op(r.name)
    where = r2.params.pop("where")
    r2.pa

"""Generation agent: chooses which directives to apply (Section 5.1).

The LLM sees the current plan, the data statistics and the catalog of
directives that are actually applicable, and nominates the promising ones.
Its picks are tried first; the remaining applicable directives follow, so a
bad or empty reply never leaves the search without candidates (the
rule-based fallback the base system also relies on).
"""
from __future__ import annotations

from ..engine import prompts
from ..llm.base import BudgetExhausted
from .directives import Candidate, DirectiveContext, DirectiveStore
from .pipeline import Pipeline

STAT_KEYS = ("n_rows", "sel_est", "avg_doc_tokens", "n_tables", "avg_query_tokens")


class GenerationAgent:
    def __init__(self, llm, store: DirectiveStore, use_llm: bool = True) -> None:
        self.llm = llm
        self.store = store
        self.use_llm = use_llm
        self.last_proposal: list[dict] = []

    def propose(self, plan: Pipeline, ctx: DirectiveContext, cap: int) -> list[Candidate]:
        applicable = self.store.applicable(plan, ctx)
        chosen: list[str] = []
        self.last_proposal = []
        if self.use_llm and applicable:
            catalog = "\n".join(f"- {d['name']}: {d['description']}" for d in self.store.catalog()
                                if d["name"] in applicable)
            stats = {k: ctx.stats[k] for k in STAT_KEYS if k in ctx.stats}
            msg = prompts.PROPOSE.format(task=ctx.task_text, pipeline=plan.describe(), stats=stats, catalog=catalog)
            try:
                r = self.llm.complete(msg, purpose="generation_agent", task="propose",
                                      payload={"directives": applicable, "pipeline": plan.describe()},
                                      system=prompts.SYSTEM_AGENT, max_tokens=300, json_mode=True)
                picks = (r.json({}) or {}).get("directives") or []
                for p in picks:
                    name = (p.get("name") if isinstance(p, dict) else str(p)).strip().upper()
                    if name in applicable and name not in chosen:
                        chosen.append(name)
                        self.last_proposal.append({"name": name, "why": p.get("why", "") if isinstance(p, dict) else ""})
            except BudgetExhausted:
                pass
        order = chosen + [n for n in applicable if n not in chosen]
        cands = self.store.expand(plan, ctx, order, cap)
        for c in cands:
            c.proposed_by = "agent" if c.directive in chosen else "rules"
        return cands[:cap]

"""PlannerLLM - Algorithm 1 of the methodology, plus the comparison systems.

    1: cands <- {P} U ApplyDirectives(P, D)
    2: for c in cands: c.score <- C^(c) - lambda * M.predAccuracyGain(c)
    3: sort cands by score; take top-m
    4: for c in top-m:
    5:    if M.signatureHit(c): verdict <- M.lookup(c)          (no LLM call)
    6:    elif VoI(c, incumbent) > theta * tokenCost(validation):
    7:        verdict <- ValidationAgent(c, sample)
    8:        M.write(c, contextSignature, verdict)
    9:    if verdict == prefer(c): incumbent <- c
   10: return incumbent

The loop is repeated for a few rounds from the new incumbent (DocETL's
opportunistic top-down traversal).  Every decision is recorded as an event;
the UI streams them live and the report quotes them as transcripts.
"""
from __future__ import annotations

import random
import time
from dataclasses import asdict, dataclass, field
from typing import Callable

from ..llm.base import BudgetExhausted
from .directives import Candidate
from .pipeline import Pipeline
from .sampling import sample_queries
from .session import Session, mean_observed
from .signature import context_features, numeric_vector, signature_of
from .validation import Verdict

EventFn = Callable[[dict], None]


@dataclass
class AuditEntry:
    round: int
    directive: str
    detail: str
    plan: str
    score: float
    cost_usd: float
    source: str           # memory | validation | gate-skip | cost-only | budget | random | tournament
    verdict: str          # prefer | reject | skip
    note: str = ""
    gate: dict | None = None
    memory: dict | None = None
    validation: dict | None = None
    signature: str = ""

    def to_dict(self) -> dict:
        return asdict(self)


@dataclass
class OptimizationResult:
    system: str
    task_id: str
    base: Pipeline
    plan: Pipeline
    audit: list[AuditEntry] = field(default_factory=list)
    events: list[dict] = field(default_factory=list)
    opt_tokens: int = 0
    opt_usd: float = 0.0
    opt_calls: int = 0
    opt_api_calls: int = 0
    n_candidates: int = 0
    n_validations: int = 0
    n_memory_hits: int = 0
    n_gate_skips: int = 0
    wall_s: float = 0.0
    budget_exhausted: bool = False

    def summary(self) -> dict:
        return {k: v for k, v in asdict(self).items() if k not in ("base", "plan", "audit", "events")} | {
            "plan": self.plan.describe(), "base": self.base.describe(), "lineage": self.plan.lineage,
            "plan_yaml": self.plan.to_yaml(), "plan_ops": self.plan.to_dict()["operators"],
            "base_ops": self.base.to_dict()["operators"]}


def diverse_top(scored: list, m: int) -> list:
    """Top-m by score, but one candidate per directive first, so two variants of
    the same rewrite (e.g. k/2 and 2k) cannot crowd every other idea out."""
    first, rest, seen = [], [], set()
    for item in scored:
        (rest if item[1].directive in seen else first).append(item)
        seen.add(item[1].directive)
    picked = (first + rest)[:m]
    return sorted(picked, key=lambda t: t[0])


class _Run:
    """Shared bookkeeping: ledger diff, budget window, event emission."""

    def __init__(self, s: Session, system: str, task, on_event: EventFn | None) -> None:
        self.s, self.system, self.task, self.on_event = s, system, task, on_event
        self.base = s.workload.base_pipeline()
        self.res = OptimizationResult(system, task.task_id, self.base, self.base)
        self.t0 = time.perf_counter()
        self.start = s.ledger.totals(optimization_only=True)
        self._saved_budget = s.ledger.optimization_budget
        if s.settings.budget.optimization_tokens is not None:
            s.ledger.optimization_budget = s.ledger.optimization_tokens() + s.settings.budget.optimization_tokens
        s.fresh_validator()

    def emit(self, kind: str, **data) -> None:
        ev = {"type": kind, "t": round(time.perf_counter() - self.t0, 3), **data}
        self.res.events.append(ev)
        if sel

"""Value-of-information gate (Section 5.4) - the economic core.

For candidate c against incumbent p:

    P(flip) = alpha * h(dir(c)) + (1 - alpha) * A
    A       = a * j + (1 - a) * (1 - j)
    dV      = w_acc * dAcc^ - |dC|
    VoI     = P(flip) * |dV|
    validate  iff  VoI > theta * tokenCost(validation)

h(dir) is the directive's historical flip rate from plan memory.  The
agreement term A turns the cheap local judge's opinion j = P_judge(c > p)
into a flip estimate, trusting it only as much as the judge has historically
agreed with the LLM validator (rate a).  With no history a = 0.5, so A = 0.5
and the term is uninformative, as it should be.

dAcc^ is the memory's prediction of the accuracy gain (clipped at 0 so a
candidate expected to be worse never *buys* validation through |dV|), and dC
is the lifetime cost difference over ``reuse_runs`` executions.  Both sides
of the inequality are in US dollars.
"""
from __future__ import annotations

from dataclasses import asdict, dataclass

from .cost_model import CostEstimate, HybridCostModel
from .memory import PlanMemory
from .pipeline import Pipeline


@dataclass
class GateDecision:
    validate: bool
    voi: float
    threshold: float
    p_flip: float
    p_raw: float
    h_dir: float
    h_trials: int
    judge_p: float | None
    agreement: float
    d_acc: float
    d_cost: float
    d_value: float
    val_cost_usd: float
    val_cost_tokens: float
    rationale: str = ""

    def to_dict(self) -> dict:
        return asdict(self)


class VoIGate:
    def __init__(self, gate_settings, memory: PlanMemory, cost_model: HybridCostModel) -> None:
        self.cfg = gate_settings
        self.memory = memory
        self.cost = cost_model

    def flip_probability(self, directive: str, judge_p: float | None) -> tuple[float, float, float, int, float]:
        h, trials = self.memory.flip_rate(directive)
        a, n_agree = self.memory.agreement_rate(directive)
        if judge_p is None or n_agree < self.cfg.min_agreement_samples:
            term = 0.5
        else:
            term = a * judge_p + (1 - a) * (1 - judge_p)
        p_raw = self.cfg.alpha * h + (1 - self.cfg.alpha) * term
        p = self.memory.calibrated(p_raw) if self.cfg.calibrate else p_raw
        return min(max(p, 0.01), 0.99), p_raw, h, trials, a

    def decide(self, directive: str, family: str, cand: Pipeline, inc: Pipeline, stats: dict,
               cost_c: CostEstimate, cost_p: CostEstimate, judge_p: float | None,
               n_items: int, n_criteria: int, theta: float | None = None) -> GateDecision:
        theta = self.cfg.theta if theta is None else theta
        p_flip, p_raw, h, trials, a = self.flip_probability(directive, judge_p)
        gain, _ = self.memory.predicted_gain(directive, family, self.cfg.prior_gain)
        d_acc = max(gain, 0.0)
        d_cost = self.cfg.reuse_runs * (cost_c.usd - cost_p.usd)
        d_value = self.cfg.w_acc * d_acc - abs(d_cost)
        voi = p_flip * abs(d_value)
        val_usd, val_tok = self.cost.validation_cost(cand, inc, stats, n_items, n_criteria)
        threshold = theta * val_usd
        ok = voi > threshold
        why = (f"P(flip)={p_flip:.2f} [h={h:.2f} over {trials}, judge={'-' if judge_p is None else f'{judge_p:.2f}'},"
               f" agree={a:.2f}] x |dV|=${abs(d_value):.5f} (dAcc^={d_acc:+.3f}, dC=${d_cost:+.5f})"
               f" = VoI ${voi:.5f} {'>' if ok else '<='} theta*tokenCost ${threshold:.5f}"
               f" -> {'VALIDATE' if ok else 'SKIP'}")
        return GateDecision(ok, voi, threshold, p_flip, p_raw, h, trials, judge_p, a, d_acc, d_cost,
                            d_value, val_usd, val_tok, why)

"""Local judge: a small CPU model that gives the gate a cheap opinion.

The judge runs only the *non-LLM prefix* of each plan (filters, ANN probe,
rerank, schema linking) on the validation sample and scores what each plan
would hand to the LLM with the MiniLM cross-encoder (22M parameters, well
inside the <=3B budget of Section 6).  It costs no API tokens.  When two
plans differ only in their LLM operators (prompt rewrite, decompose,
few-shot...) the prefixes are identical and the judge abstains (returns
None), which the gate treats as "no information".
"""
from __future__ import annotations

import math

import numpy as np

from .pipeline import Pipeline


def cheap_prefix(plan: Pipeline) -> Pipeline:
    ops = []
    for o in plan.topological():
        if o.uses_llm:
            break
        ops.append(o)
    p = plan.clone()
    p.operators = [o for o in p.operators if o.name in {x.name for x in ops}]
    return p


class LocalJudge:
    def __init__(self, workload, executor, temperature: float = 0.6, top: int = 3) -> None:
        self.wl = workload
        self.ex = executor
        self.temperature = temperature
        self.top = top
        self.calls = 0

    def _context_quality(self, plan: Pipeline, queries: list[dict]) -> float | None:
        prefix = cheap_prefix(plan)
        if not prefix.operators:
            return None
        scores = []
        for q in queries:
            res = self.ex.run(prefix, q, purpose="judge")
            texts = [self.wl.records[self.wl.id_to_pos[i]]["text"] for i in res.retrieved_ids[: self.top]]
            if not texts:
                scores.append(-10.0)
                continue
            s = self.wl.reranker.score(q["question"], texts)
            scores.append(float(np.mean(np.sort(s)[::-1][: self.top])))
        return float(np.mean(scores)) if scores else None

    def prefer_probability(self, cand: Pipeline, inc: Pipeline, queries: list[dict]) -> float | None:
        """P_judge(candidate better than incumbent), or None when uninformative."""
        if cheap_prefix(cand).fingerprint() == cheap_prefix(inc).fingerprint():
            return None
        self.calls += 1
        sc = self._context_quality(cand, queries)
        sp = self._context_quality(inc, queries)
        if sc is None or sp is None:
            return None
        return 1.0 / (1.0 + math.exp(-(sc - sp) / self.temperature))

"""Selectivity-aware, size-weighted sampling for optimisation-time validation.

Following DocETL, the sample grows when downstream operators are selective
(a filter or an extract-map that drops most records would otherwise leave
too few informative outputs), and items are drawn with probability that
grows with document/question length so long, hard items are not starved.
"""
from __future__ import annotations

import math
import random

from .pipeline import Pipeline


def downstream_selectivity(plan: Pipeline, stats: dict) -> float:
    sel = 1.0
    for op in plan.topological():
        if op.kind == "filter" or (op.kind == "retrieve" and op.params.get("where")):
            sel *= max(stats.get("sel_est", 1.0), 0.02) ** 0.25   # soften: queries, not rows
        if op.kind == "map":
            sel *= 0.5
    return max(sel, 0.1)


def sample_size(n_pool: int, plan: Pipeline, stats: dict, cfg) -> int:
    sel = downstream_selectivity(plan, stats)
    n = math.ceil(max(cfg.min_items, cfg.frac * n_pool) / sel)
    return int(min(max(n, cfg.min_items), cfg.max_items, n_pool))


def sample_queries(queries: list[dict], plan: Pipeline, stats: dict, cfg, salt: str = "") -> list[dict]:
    n = sample_size(len(queries), plan, stats, cfg)
    rng = random.Random(f"{cfg.seed}:{salt}")
    weights = [math.sqrt(len(q["question"]) + len(q.get("contract_text", "")) / 50 + 1) for q in queries]
    pool = list(range(len(queries)))
    picked: list[int] = []
    while pool and len(picked) < n:   # weighted sampling without replacement
        total = sum(weights[i] for i in pool)
        r, acc = rng.random() * total, 0.0
        for i in pool:
            acc += weights[i]
            if acc >= r:
                picked.append(i)
                pool.remove(i)
                break
    return [queries[i] for i in sorted(picked)]

"""Validation agent (the base-paper evaluation protocol, Sections 5.4 / 8).

1. synthesise task-specific criteria for the pipeline's final operator;
2. execute candidate and incumbent on a selectivity-aware sample;
3. rate every output 1-5 per criterion with the LLM;
4. when the average ratings are close, run a pairwise comparison (with the
   A/B positions swapped on alternate items to cancel position bias);
5. return a verdict plus a transcript the UI and report can show.

The agent never sees reference answers - it judges outputs the way a human
reviewer would, from the question and the context the plan used.
"""
from __future__ import annotations

import time
from dataclasses import asdict, dataclass, field

from ..engine import prompts
from ..engine.executor import ExecResult, Executor
from ..llm.base import BudgetExhausted
from .pipeline import Pipeline


@dataclass
class Verdict:
    prefer_candidate: bool
    score_c: float = 0.0
    score_p: float = 0.0
    criteria: list = field(default_factory=list)
    pairwise_win_rate: float | None = None
    source: str = "validation"     # validation | memory | gate-skip | cost-only | budget
    confidence: float = 1.0
    tokens: int = 0
    note: str = ""
    items: list = field(default_factory=list)

    @property
    def acc_delta(self) -> float:
        """Rating gap mapped onto the 0-1 accuracy scale used by the gate."""
        return (self.score_c - self.score_p) / 4.0

    def to_dict(self) -> dict:
        d = asdict(self)
        d["acc_delta"] = self.acc_delta
        return d


def output_text(res: ExecResult) -> str:
    if res.sql is not None:
        rows = res.rows[:5] if res.rows else res.rows
        return f"SQL: {res.sql}\nResult (first rows): {rows}\nError: {res.error or 'none'}"
    return res.answer or "(empty)"


def context_text(res: ExecResult, limit: int = 3, chars: int = 350) -> str:
    return "\n".join(f"- {c[:chars]}" for c in res.context[:limit]) or "(none)"


class ValidationAgent:
    def __init__(self, llm, executor: Executor, settings, task_text: str, final_kind: str) -> None:
        self.llm = llm
        self.ex = executor
        self.cfg = settings.validation
        self.task_text = task_text
        self.final_kind = final_kind
        self._criteria: list[dict] | None = None
        self._outputs: dict[tuple[str, str], ExecResult] = {}
        self._ratings: dict[tuple[str, str], float] = {}

    # ------------------------------------------------------------ steps
    def criteria(self) -> list[dict]:
        if self._criteria is None:
            msg = prompts.CRITERIA.format(task=self.task_text, kind=self.final_kind, k=self.cfg.max_criteria)
            r = self.llm.complete(msg, purpose="criteria", task="criteria",
                                  payload={"max_criteria": self.cfg.max_criteria},
                                  system=prompts.SYSTEM_AGENT, max_tokens=300, json_mode=True)
            crits = (r.json({}) or {}).get("criteria") or []
            crits = [c for c in crits if isinstance(c, dict) and c.get("name")][: self.cfg.max_criteria]
            self._criteria = crits or [{"name": "accuracy", "description": "the output answers the question correctly"},
                                       {"name": "faithfulness", "description": "the output is supported by the context"}]
        return self._criteria

    def outputs(self, plan: Pipeline, sample: list[dict]) -> list[ExecResult]:
        fp = plan.fingerprint()
        out = []
        for q in sample:
            key = (fp, q["qid"])
            if key not in self._outputs:
                self._outputs[key] = self.ex.run(plan, q, purpose="validation_exec")
            out.append(self._outputs[key])
        return out

    def _rate(self, plan_fp: str, q: dict, res: ExecResult) -> float:
        key = (plan_fp, q["qid"])
        if key in self._ratings:
            return self._ratings[key]
        crits = self.criteria()
        crit_txt = "\n".join(f"- {c['na

## Plan memory, cost and latency

In [6]:
# Source snapshot. The experiment below executes the imported copy.
show_implementation({
    'core/memory.py': (
        '"""Plan memory (Sections 5.2 and 5.5).\n'
        '\n'
        'Persistent SQLite store that turns one-shot agentic optimisation into a\n'
        'learning optimiser.  It keeps\n'
        '\n'
        '(i)   validated rewrite verdicts keyed by applicability signature,\n'
        '(ii)  per-directive flip statistics  h(dir),\n'
        '(iii) local-judge vs LLM-validator agreement, per directive,\n'
        '(iv)  observed-cost telemetry for the residual cost model,\n'
        '(v)   a gate log used to calibrate P(flip) and to score the gate offline.\n'
        '\n'
        'Verdict confidence decays with age and with context drift:\n'
        '\n'
        '    delta(t) = exp(-kappa * age_days) * exp(-drift_penalty * drift) * n/(n + 1)\n'
        '\n'
        'With the default reuse threshold a verdict must be confirmed by two\n'
        'independent validations before it is reused at all (n/(n+1) = 0.5 -> 0.67).\n'
        '\n'
        'so a pattern confirmed several times recently is trusted, a stale one or one\n'
        'validated under different statistics is re-validated instead of reused.\n'
        '"""\n'
        'from __future__ import annotations\n'
        '\n'
        'import json\n'
        'import math\n'
        'import sqlite3\n'
        'import threading\n'
        'import time\n'
        'from dataclasses import dataclass\n'
        'from pathlib import Path\n'
        '\n'
        'import numpy as np\n'
        '\n'
        'from .signature import drift as ctx_drift\n'
        '\n'
        'SCHEMA = """\n'
        'CREATE TABLE IF NOT EXISTS rewrites (\n'
        '  sig TEXT PRIMARY KEY, directive TEXT, family TEXT, parent_shape TEXT,\n'
        '  features TEXT, vec TEXT, verdict TEXT, acc_delta REAL, cost_delta REAL,\n'
        '  n_obs INT, n_agree INT, created REAL, updated REAL, workload TEXT);\n'
        'CREATE TABLE IF NOT EXISTS flip_stats (directive TEXT PRIMARY KEY, flips INT, trials INT);\n'
        'CREATE TABLE IF NOT EXISTS agreement (directive TEXT PRIMARY KEY, agree INT, total INT);\n'
        'CREATE TABLE IF NOT EXISTS gate_log (\n'
        '  id INTEGER PRIMARY KEY AUTOINCREMENT, directive TEXT, p_raw REAL, p_used REAL,\n'
        '  voi REAL, token_cost REAL, validated INT, flipped INT, ts REAL, workload TEXT);\n'
        'CREATE TABLE IF NOT EXISTS telemetry (\n'
        '  id INTEGER PRIMARY KEY AUTOINCREMENT, workload TEXT, plan_fp TEXT, features TEXT,\n'
        '  sym_usd REAL, sym_ms REAL, obs_usd REAL, obs_ms REAL, ts REAL);\n'
        'CREATE TABLE IF NOT EXISTS retrain_log (ts REAL, n_telemetry INT, n_gate INT, note TEXT);\n'
        '"""\n'
        '\n'
        '\n'
        'STAT_KEYS = {"rows_b", "sel_b", "doclen_b", "tables_b"}\n'
        '\n'
        '\n'
        'def same_structure(a: dict, b: dict) -> bool:\n'
        '    """Equal on every signature feature except the bucketed data statistics."""\n'
        '    keys = (set(a) | set(b)) - STAT_KEYS\n'
        '    return all(str(a.get(k)) == str(b.get(k)) for k in keys)\n'
        '\n'
        '\n'
        '@dataclass\n'
        'class MemoryHit:\n'
        '    sig: str\n'
        '    directive: str\n'
        '    prefer: bool\n'
        '    confidence: float\n'
        '    acc_delta: float\n'
        '    match: str          # exact | soft\n'
        '    age_days: float\n'
        '    drift: float\n'
        '    n_obs: int\n'
        '\n'
        '    def to_dict(self) -> dict:\n'
        '        return dict(vars(self))\n'
        '\n'
        '\n'
        'class PlanMemory:\n'
        '    def __init__(self, path: str | Path = ":memory:", kappa: float = 0.02, drift_penalty: float = 3.0,\n'
        '                 reuse_threshold: float = 0.55, soft_match_threshold: float = 0.90,\n'
        '                 retrain_every_days: float = 7.0, clock=None) -> None:\n'
        '        self.path = str(path)\n'
        '        if self.path != ":memory:":\n'
        '            Path(self.path).parent.mkdir(parents=True, exist_ok=True)\n'
        '        self._db = sqlite3.connect(self.path, check_same_thread=False)\n'
        '        self._lock = threading.RLock()\n'
        '        with self._lock:\n'
        '            self._db.executescript(SCHEMA)\n'
        '            self._db.commit()\n'
        '        self.kappa, self.drift_penalty = kappa, drift_penalty\n'
        '        self.reuse_threshold, self.soft_threshold = reuse_threshold, soft_match_threshold\n'
        '        self.retrain_every = retrain_every_days\n'
        '        self.clock = clock or (lambda: time.time() / 86400.0)\n'
        '        self._last_retrain = self.clock()\n'
        '        self._calib: tuple[float, float, int] | None = None\n'
        '\n'
        '    @classmethod\n'
        '    def from_settings(cls, ms, path: str | None = None, clock=None) -> "PlanMemory":\n'
        '        return cls(path or ms.path, ms.kappa, ms.drift_penalty, ms.reuse_threshold,\n'
        '                   ms.soft_match_threshold, ms.retrain_every_days, clock)\n'
        '\n'
        '    def _q(self, sql: str, args: tuple = ()):\n'
        '        with self._lock:\n'
        '            return self._db.execute(sql, args).fetchall()\n'
        '\n'
        '    def _x(self, sql: str, args: tuple = ()) -> None:\n'
        '        with self._lock:\n'
        '            self._db.execute(sql, args)\n'
        '            self._db.commit()\n'
        '\n'
        '    # ------------------------------------------------------------ verdicts\n'
        '    def confidence(self, age_days: float, drift: float, n_obs: int) -> float:\n'
        '        return (math.exp(-self.kappa * max(age_days, 0.0))\n'
        '                * math.exp(-self.drift_penalty * drift)\n'
        '                * (n_obs / (n_obs + 1.0)))\n'
        '\n'
        '    def lookup(self, sig: str, vec: np.ndarray, directive: str, parent_shape: str,\n'
        '               family: str, features: dict | None = None) -> MemoryHit | None:\n'
        '        """Exact signature first; otherwise the closest context that has the same\n'
        '        plan structure and differs only in data statistics (a *soft* hit)."""\n'
        '        now = self.clock()\n'
        '        row = self._q("SELECT sig, directive, verdict, acc_delta, n_obs, updated, vec "\n'
        '                      "FROM rewrites WHERE sig=?", (sig,))\n'
        '        best: MemoryHit | None = None\n'
        '        if row:\n'
        '            s, d, v, acc, n, upd, vjs = row[0]\n'
        '            dr = ctx_drift(np.array(json.loads(vjs)), vec)\n'
        '            best = MemoryHit(s, d, v == "prefer", self.confidence(now - upd, dr, n), acc,\n'
        '                             "exact", now - upd, dr, n)\n'
        '        else:\n'
        '            rows = self._q("SELECT sig, directive, verdict, acc_delta, n_obs, updated, vec, features "\n'
        '                           "FROM rewrites WHERE directive=? AND parent_shape=? AND family=?",\n'
        '                           (directive, parent_shape, family))\n'
        '            for s, d, v, acc, n, upd, vjs, fjs in rows:\n'
        '                if features is not None and not same_structure(json.loads(fjs), features):\n'
        '                    continue\n'
        '                dr = ctx_drift(np.array(json.loads(vjs)), vec)\n'
        '                sim = math.exp(-dr)\n'
        '                if sim < self.soft_threshold:\n'
        '                    continue\n'
        '                hit = MemoryHit(s, d, v == "prefer", self.confidence(now - upd, dr, n) * sim, acc,\n'
        '                                "soft", now - upd, dr, n)\n'
        '                if best is None or hit.confidence > best.confidence:\n'
        '                    best = hit\n'
        '        return best\n'
        '\n'
        '    def usable(self, hit: MemoryHit | None) -> bool:\n'
        '        return hit is not None and hit.confidence >= self.reuse_threshold\n'
        '\n'
        '    def write(self, sig: str, features: dict, vec: np.ndarray, directive: str, prefer: bool,\n'
        '              acc_delta: float, cost_delta: float, workload: str) -> None:\n'
        '        now = self.clock()\n'
        '        verdict = "prefer" if prefer else "reject"\n'
        '        row = self._q("SELECT verdict, acc_delta, n_obs, n_agree FROM rewrites WHERE sig=?", (sig,))\n'
        '        if row:\n'
        '            old_v, old_acc, n, agree = row[0]\n'
        '            if old_v == verdict:\n'
        '                n, agree = n + 1, agree + 1\n'
        '                acc = 0.6 * old_acc + 0.4 * acc_delta\n'
        '            else:   # contradicted: the newest evidence wins, confidence restarts\n'
        '                n, agree, acc = 1, 0, acc_delta\n'
        '            self._x("UPDATE rewrites SET verdict=?, acc_delta=?, cost_delta=?, n_obs=?, n_agree=?, "\n'
        '                    "updated=?, vec=? WHERE sig=?",\n'
        '                    (verdict, acc, cost_delta, n, agree, now, json.dumps(vec.tolist()), sig))\n'
        '        else:\n'
        '            self._x("INSERT INTO rewrites VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)",\n'
        '                    (sig, directive, features.get("family"), features.get("parent_shape"),\n'
        '                     json.dumps(features, default=str), json.dumps(vec.tolist()), verdict,\n'
        '                     acc_delta, cost_delta, 1, 1, now, now, workload))\n'
        '\n'
        '    def touch(self, sig: str) -> None:\n'
        '        """A reused verdict counts as a confirmation of recency (not of correctness)."""\n'
        '        self._x("UPDATE rewrites SET updated=? WHERE sig=?", (self.clock(), sig))\n'
        '\n'
        '    # ------------------------------------------------------------ gate statistics\n'
        '    def update_flip(self, directive: str, flipped: bool) -> None:\n'
        '        row = self._q("SELECT flips, trials FROM flip_stats WHERE directive=?", (directive,))\n'
        '        f, t = row[0] if row else (0, 0)\n'
        '        self._x("INSERT OR REPLACE INTO flip_stats VALUES (?,?,?)", (directive, f + int(flipped), t + 1))\n'
        '\n'
        '    def flip_rate(self, directive: str) -> tuple[float, int]:\n'
        '        """Laplace-smoothed historical flip rate h(dir) and its sample size."""\n'
        '        row = self._q("SELECT flips, trials FROM flip_stats WHERE directive=?", (directive,))\n'
        '        f, t = row[0] if row else (0, 0)\n'
        '        return (f + 1) / (t + 2), t\n'
        '\n'
        '    def update_agreement(self, directive: str, agreed: bool) -> None:\n'
        '        row = self._q("SELECT agree, total FROM agreement WHERE directive=?", (directive,))\n'
        '        a, t = row[0] if row else (0, 0)\n'
        '        self._x("INSERT OR REPLACE INTO agreement VALUES (?,?,?)", (directive, a + int(agreed), t + 1))\n'
        '\n'
        '    def agreement_rate(self, directive: str) -> tuple[float, int]:\n'
        '        """Judge/validator concordance for a directive, pooled when data is thin."""\n'
        '        row = self._q("SELECT agree, total FROM agreement WHERE directive=?", (directive,))\n'
        '        a, t = row[0] if row else (0, 0)\n'
        '        pooled = self._q("SELECT COALESCE(SUM(agree),0), COALESCE(SUM(total),0) FROM agreement")[0]\n'
        '        w = t / (t + 5.0)\n'
        '        own = (a + 1) / (t + 2)\n'
        '        pool = (pooled[0] + 1) / (pooled[1] + 2)\n'
        '        return w * own + (1 - w) * pool, int(pooled[1])\n'
        '\n'
        '    def predicted_gain(self, directive: str, family: str, prior: float, n0: float = 2.0) -> tuple[float, int]:\n'
        '        """Confidence-weighted mean accuracy delta of a directive, shrunk to ``prior``."""\n'
        '        now = self.clock()\n'
        '        rows = self._q("SELECT acc_delta, n_obs, updated FROM rewrites WHERE directive=? AND family=?",\n'
        '                       (directive, family))\n'
        '        num, den = n0 * prior, n0\n'
        '        for acc, n, upd in rows:\n'
        '            w = self.confidence(now - upd, 0.0, n)\n'
        '            num += w * acc\n'
        '            den += w\n'
        '        return num / den, len(rows)\n'
        '\n'
        '    def log_gate(self, directive: str, p_raw: float, p_used: float, voi: float, token_cost: float,\n'
        '                 validated: bool, flipped: bool | None, workload: str) -> int:\n'
        '        with self._lock:\n'
        '            cur = self._db.execute(\n'
        '                "INSERT INTO gate_log (directive, p_raw, p_used, voi, token_cost, validated, flipped, ts, workload)"\n'
        '                " VALUES (?,?,?,?,?,?,?,?,?)",\n'
        '                (directive, p_raw, p_used, voi, token_cost, int(validated),\n'
        '                 None if flipped is None else int(flipped), self.clock(), workload))\n'
        '            self._db.commit()\n'
        '            return cur.lastrowid\n'
        '\n'
        '    def set_gate_outcome(self, log_id: int, flipped: bool) -> None:\n'
        '        self._x("UPDATE gate_log SET flipped=? WHERE id=?", (int(flipped), log_id))\n'
        '\n'
        '    def calibrate_gate(self, min_samples: int = 12) -> tuple[float, float, int] | None:\n'
        '        """Platt scaling of logit(p_raw) -> realised flip, fitted on validated rows."""\n'
        '        rows = self._q("SELECT p_raw, flipped FROM gate_log WHERE flipped IS NOT NULL")\n'
        '        if len(rows) < min_samples or len({r[1] for r in rows}) < 2:\n'
        '            return self._calib\n'
        '        from sklearn.linear_model import LogisticRegression\n'
        '        p = np.clip(np.array([r[0] for r in rows]), 1e-3, 1 - 1e-3)\n'
        '        X = np.log(p / (1 - p)).reshape(-1, 1)\n'
        '        y = np.array([r[1] for r in rows])\n'
        '        lr = LogisticRegression(C=1.0).fit(X, y)\n'
        '        self._calib = (float(lr.coef_[0][0]), float(lr.intercept_[0]), len(rows))\n'
        '        return self._calib\n'
        '\n'
        '    def calibrated(self, p_raw: float) -> float:\n'
        '        if self._calib is None:\n'
        '            return p_raw\n'
        '        a, b, n = self._calib\n'
        '        p = min(max(p_raw, 1e-3), 1 - 1e-3)\n'
        '        cal = 1 / (1 + math.exp(-(a * math.log(p / (1 - p)) + b)))\n'
        '        w = n / (n + 30.0)          # trust the calibration map as evidence accumulates\n'
        '        return w * cal + (1 - w) * p_raw\n'
        '\n'
        '    # ------------------------------------------------------------ telemetry\n'
        '    def add_telemetry(self, workload: str, plan_fp: str, features: list[float], sym_usd: float,\n'
        '                      sym_ms: float, obs_usd: float, obs_ms: float) -> None:\n'
        '        self._x("INSERT INTO telemetry (workload, plan_fp, features, sym_usd, sym_ms, obs_usd, obs_ms, ts)"\n'
        '                " VALUES (?,?,?,?,?,?,?,?)",\n'
        '                (workload, plan_fp, json.dumps(features), sym_usd, sym_ms, obs_usd, obs_ms, self.clock()))\n'
        '\n'
        '    def telemetry(self, workloads: list[str] | None = None) -> dict:\n'
        '        rows = self._q("SELECT workload, features, sym_usd, sym_ms, obs_usd, obs_ms FROM telemetry")\n'
        '        if workloads:\n'
        '            rows = [r for r in rows if r[0] in workloads]\n'
        '        return {"workload": [r[0] for r in rows], "X": [json.loads(r[1]) for r in rows],\n'
        '                "sym_usd": [r[2] for r in rows], "sym_ms": [r[3] for r in rows],\n'
        '                "obs_usd": [r[4] for r in rows], "obs_ms": [r[5] for r in rows]}\n'
        '\n'
        '    def maybe_retrain(self, cost_model, force: bool = False) -> dict | None:\n'
        '        """The \'weekly CPU retraining\' hook: residual cost model + gate calibration."""\n'
        '        now = self.clock()\n'
        '        if not force and now - self._last_retrain < self.retrain_every:\n'
        '            return None\n'
        '        t = self.telemetry()\n'
        '        fitted = cost_model.fit(t["X"], t["obs_usd"], t["obs_ms"], t["sym_usd"], t["sym_ms"])\n'
        '        calib = self.calibrate_gate()\n'
        '        self._last_retrain = now\n'
        '        n_gate = self._q("SELECT COUNT(*) FROM gate_log WHERE flipped IS NOT NULL")[0][0]\n'
        '        self._x("INSERT INTO retrain_log VALUES (?,?,?,?)",\n'
        '                (now, len(t["X"]), n_gate, f"residual={\'yes\' if fitted else \'no\'} calib={calib}"))\n'
        '        return {"residual_fitted": fitted, "n_telemetry": len(t["X"]), "gate_calibration": calib}\n'
        '\n'
        '    # ------------------------------------------------------------ reporting\n'
        '    def report(self) -> dict:\n'
        '        rw = self._q("SELECT sig, directive, family, verdict, acc_delta, n_obs, updated, workload, features "\n'
        '                     "FROM rewrites ORDER BY updated DESC")\n'
        '        now = self.clock()\n'
        '        return {\n'
        '            "rewrites": [{"sig": r[0], "directive": r[1], "family": r[2], "verdict": r[3],\n'
        '                          "acc_delta": r[4], "n_obs": r[5], "age_days": now - r[6], "workload": r[7],\n'
        '                          "confidence": self.confidence(now - r[6], 0.0, r[5]),\n'
        '                          "features": json.loads(r[8])} for r in rw],\n'
        '            "flip_stats": [{"directive": d, "flips": f, "trials": t, "h": (f + 1) / (t + 2)}\n'
        '                           for d, f, t in self._q("SELECT * FROM flip_stats ORDER BY directive")],\n'
        '            "agreement": [{"directive": d, "agree": a, "total": t, "rate": (a + 1) / (t + 2)}\n'
        '                          for d, a, t in self._q("SELECT * FROM agreement ORDER BY directive")],\n'
        '            "gate_log": [dict(zip(("directive", "p_raw", "p_used", "voi", "token_cost", "validated",\n'
        '                                   "flipped", "ts", "workload"), r))\n'
        '                         for r in self._q("SELECT directive, p_raw, p_used, voi, token_cost, validated, "\n'
        '                                          "flipped, ts, workload FROM gate_log ORDER BY id")],\n'
        '            "telemetry_rows": self._q("SELECT COUNT(*) FROM telemetry")[0][0],\n'
        '            "retrains": [dict(zip(("ts", "n_telemetry", "n_gate", "note"), r))\n'
        '                         for r in self._q("SELECT * FROM retrain_log ORDER BY ts")],\n'
        '            "calibration": self._calib,\n'
        '        }\n'
    ),
    'core/signature.py': (
        '"""Applicability signatures (Section 5.2).\n'
        '\n'
        "A rewrite's verdict is only reusable in a context that looks like the one in\n"
        'which it was validated.  The signature captures that context from four\n'
        'sources named in the methodology: template type (operator shape), data\n'
        'statistics (cardinality, selectivity, document length), operator\n'
        'configuration (index family, top-k), and prompt structure.  Continuous\n'
        'statistics are bucketed so that "the same kind of query" hashes identically\n'
        'while a real shift (e.g. selectivity moving an order of magnitude) does not.\n'
        '"""\n'
        'from __future__ import annotations\n'
        '\n'
        'import hashlib\n'
        'import json\n'
        'import math\n'
        'import re\n'
        '\n'
        'import numpy as np\n'
        '\n'
        'from .pipeline import Pipeline\n'
        '\n'
        '\n'
        'def _bucket(value: float | None, edges: list[float]) -> int:\n'
        '    if value is None:\n'
        '        return -1\n'
        '    for i, e in enumerate(edges):\n'
        '        if value <= e:\n'
        '            return i\n'
        '    return len(edges)\n'
        '\n'
        '\n'
        'def prompt_structure(plan: Pipeline) -> str:\n'
        '    """Shape of the instruction text, not its words: length band + format cues."""\n'
        '    parts = []\n'
        '    for o in plan.topological():\n'
        '        if not o.prompt:\n'
        '            continue\n'
        '        n_sent = len([s for s in re.split(r"[.!?]+", o.prompt) if s.strip()])\n'
        '        cues = "".join(c for c, rx in (("N", r"\\bNONE\\b"), ("Q", r"quote|verbatim|word for word"),\n'
        '                                       ("F", r"format|json|one sentence|short"))\n'
        '                       if re.search(rx, o.prompt, re.I))\n'
        '        parts.append(f"{o.kind}:{min(n_sent, 4)}{cues}")\n'
        '    return "|".join(parts)\n'
        '\n'
        '\n'
        'def _retrieve_cfg(plan: Pipeline) -> tuple[int | None, str | None, bool]:\n'
        '    r = plan.find("retrieve")\n'
        '    if r is None:\n'
        '        return None, None, False\n'
        '    return int(r.params.get("top_k", 10)), r.params.get("index", "hnsw"), bool(r.params.get("where"))\n'
        '\n'
        '\n'
        'def config_shape(plan: Pipeline) -> str:\n'
        '    """Physical configuration of a plan with top-k bucketed (part of the context)."""\n'
        '    k, index, filtered = _retrieve_cfg(plan)\n'
        '    gen = plan.find("sql_generate")\n'
        '    shots = int(gen.params.get("fewshot_k", 0)) if gen else 0\n'
        '    return f"{index}|{int(filtered)}|{_bucket(k, [5, 10, 20])}|{shots}|{prompt_structure(plan)}"\n'
        '\n'
        '\n'
        'def context_features(parent: Pipeline, cand: Pipeline, directive: str, stats: dict) -> dict:\n'
        '    k, index, filtered = _retrieve_cfg(cand)\n'
        '    gen = cand.find("sql_generate")\n'
        '    return {\n'
        '        "directive": directive,\n'
        '        "family": stats.get("family"),\n'
        '        "parent_shape": "->".join(parent.kinds()),\n'
        '        "parent_cfg": config_shape(parent),\n'
        '        "cand_shape": "->".join(cand.kinds()),\n'
        '        "rows_b": _bucket(math.log2(max(stats.get("n_rows", 1), 1)), [8, 10, 12, 14, 17, 20]),\n'
        '        "sel_b": _bucket(stats.get("sel_est"), [0.005, 0.02, 0.06, 0.15, 0.4, 0.99]),\n'
        '        "doclen_b": _bucket(stats.get("avg_doc_tokens"), [64, 128, 256, 512]),\n'
        '        "topk_b": _bucket(k, [5, 10, 20]),\n'
        '        "index": index,\n'
        '        "filtered_probe": filtered,\n'
        '        "fewshot": int(gen.params.get("fewshot_k", 0)) if gen else None,\n'
        '        "tables_b": _bucket(stats.get("n_tables"), [3, 5, 8, 12]),\n'
        '        "prompt": prompt_structure(cand),\n'
        '    }\n'
        '\n'
        '\n'
        'def signature_of(features: dict) -> str:\n'
        '    blob = json.dumps(features, sort_keys=True, default=str)\n'
        '    return hashlib.sha1(blob.encode()).hexdigest()[:16]\n'
        '\n'
        '\n'
        'def numeric_vector(stats: dict, cand: Pipeline) -> np.ndarray:\n'
        '    """Continuous description used for near-signature (soft) matches and drift."""\n'
        '    k, _, filtered = _retrieve_cfg(cand)\n'
        '    return np.array([\n'
        '        math.log10(max(stats.get("n_rows", 1), 1)),\n'
        '        math.log10(max(stats.get("sel_est", 1.0), 1e-4)),\n'
        '        math.log10(max(stats.get("avg_doc_tokens", 1.0), 1.0)),\n'
        '        math.log2(max(k or 1, 1)),\n'
        '        float(filtered),\n'
        '        math.log2(max(stats.get("n_tables", 1) or 1, 1)),\n'
        '    ], dtype=float)\n'
        '\n'
        '\n'
        'def drift(a: np.ndarray, b: np.ndarray) -> float:\n'
        '    """Distance between two contexts in log-space (0 == identical)."""\n'
        '    return float(np.linalg.norm(np.asarray(a) - np.asarray(b)) / math.sqrt(len(a)))\n'
        '\n'
        '\n'
        'def cosine(a: np.ndarray, b: np.ndarray) -> float:\n'
        '    na, nb = np.linalg.norm(a), np.linalg.norm(b)\n'
        '    if na == 0 or nb == 0:\n'
        '        return 0.0\n'
        '    return float(np.dot(a, b) / (na * nb))\n'
    ),
    'core/cost_model.py': (
        '"""Hybrid symbolic-neural cost model (Section 5.3).\n'
        '\n'
        '    C^(c) = sum_op f_sym(op; stats)  +  f_ANN(index, n, d, k)  +  sum_llm tokens * price\n'
        '    C^_hybrid(c) = C^(c) * exp(r^_nn(features(c)))\n'
        '\n'
        'The symbolic part walks the plan once, propagating estimated row counts with\n'
        "DuckDB's optimizer selectivity and the classical independence assumption.\n"
        'It is deliberately the "textbook" estimator: it is wrong exactly where the\n'
        'literature says it is wrong (correlated predicates - e.g. a post-filter after\n'
        'a semantic probe keeps far more rows than the corpus-wide selectivity says).\n'
        'The learned residual is a small gradient-boosted model on the log-ratio\n'
        'between observed and symbolic cost, so it can only *correct* the backbone,\n'
        'never replace it - unless the ``neural`` ablation is requested.\n'
        '"""\n'
        'from __future__ import annotations\n'
        '\n'
        'import math\n'
        'from dataclasses import dataclass, field\n'
        '\n'
        'import numpy as np\n'
        '\n'
        'from .pipeline import Pipeline\n'
        '\n'
        '# token sizes of the fixed parts of our prompt templates (measured once)\n'
        'TEMPLATE_TOKENS = {"generate": 70, "map": 60, "sql_generate": 45, "sql_repair": 70}\n'
        'OUTPUT_TOKENS = {"generate": 45, "map": 35, "sql_generate": 40, "sql_repair": 40}\n'
        'MAP_KEEP_RATE = 0.5\n'
        'REPAIR_RATE = 0.3\n'
        '\n'
        '\n'
        '@dataclass\n'
        'class CostEstimate:\n'
        '    usd: float\n'
        '    latency_ms: float\n'
        '    tokens: float\n'
        '    llm_calls: float\n'
        '    breakdown: dict = field(default_factory=dict)\n'
        '    mode: str = "symbolic"\n'
        '\n'
        '    def to_dict(self) -> dict:\n'
        '        return {"usd": self.usd, "latency_ms": self.latency_ms, "tokens": self.tokens,\n'
        '                "llm_calls": self.llm_calls, "mode": self.mode, "breakdown": self.breakdown}\n'
        '\n'
        '\n'
        '@dataclass\n'
        'class Calibration:\n'
        '    """Machine-specific constants, measured by microbenchmarks at start-up."""\n'
        '    ann_ms: dict = field(default_factory=lambda: {"hnsw": 0.15, "ivf": 0.10, "flat": 0.6})\n'
        '    ce_ms_per_pair: float = 2.5\n'
        '    db_ms_per_krow: float = 0.4\n'
        '    llm_overhead_s: float = 0.25\n'
        '    llm_s_per_in_token: float = 0.00015\n'
        '    llm_s_per_out_token: float = 0.004\n'
        '\n'
        '\n'
        'class HybridCostModel:\n'
        '    FEATURES = ["log_rows", "sel_est", "log_doc_tokens", "llm_calls", "log_tokens", "top_k",\n'
        '                "has_rerank", "has_map", "filtered_probe", "post_filter", "idx_hnsw", "idx_ivf",\n'
        '                "idx_flat", "fewshot", "has_repair", "fam_retrieval", "fam_mixed", "fam_sql",\n'
        '                "log_sym_usd", "log_sym_ms"]\n'
        '\n'
        '    def __init__(self, price_in_per_m: float, price_out_per_m: float, cpu_usd_per_hour: float,\n'
        '                 calibration: Calibration | None = None, mode: str = "hybrid") -> None:\n'
        '        self.p_in = price_in_per_m / 1e6\n'
        '        self.p_out = price_out_per_m / 1e6\n'
        '        self.cpu_usd_per_ms = cpu_usd_per_hour / 3.6e6\n'
        '        self.calib = calibration or Calibration()\n'
        '        self.mode = mode                      # symbolic | neural | hybrid\n'
        '        self._res_usd = self._res_ms = None   # residual regressors\n'
        '        self._nn_usd = self._nn_ms = None     # neural-only regressors\n'
        '        self.n_train = 0\n'
        '\n'
        '    # ------------------------------------------------------------ symbolic\n'
        '    def _llm_call(self, kind: str, prompt_tokens: float) -> tuple[float, float, float]:\n'
        '        out = OUTPUT_TOKENS.get(kind, 40)\n'
        '        usd = prompt_tokens * self.p_in + out * self.p_out\n'
        '        sec = self.calib.llm_overhead_s + prompt_tokens * self.calib.llm_s_per_in_token \\\n'
        '            + out * self.calib.llm_s_per_out_token\n'
        '        return usd, sec * 1000, prompt_tokens + out\n'
        '\n'
        '    def symbolic(self, plan: Pipeline, stats: dict) -> CostEstimate:\n'
        '        n = float(stats.get("n_rows", 1000))\n'
        '        sel = float(stats.get("sel_est", 1.0))\n'
        '        doc_tok = min(float(stats.get("avg_doc_tokens", 120)), 175.0)\n'
        '        q_tok = float(stats.get("avg_query_tokens", 12))\n'
        '        rows: float | None = None            # None == whole corpus still in play\n'
        '        restricted = 1.0                     # fraction of corpus visible to the probe\n'
        '        db_ms = ann_ms = ce_ms = llm_ms = usd_tok = tokens = calls = 0.0\n'
        '        per_op = []\n'
        '        for op in plan.topological():\n'
        '            before = (db_ms, ann_ms, ce_ms, llm_ms, usd_tok, tokens)\n'
        '            k = op.kind\n'
        '            if k == "filter":\n'
        '                scanned = n if rows is None else rows\n'
        '                db_ms += scanned / 1000 * self.calib.db_ms_per_krow + 0.3\n'
        '                if rows is None:\n'
        '                    restricted *= sel\n'
        '                    rows = n * restricted\n'
        '                else:\n'
        '                    rows = rows * sel        # independence assumption (the classic error)\n'
        '            elif k == "retrieve":\n'
        '                fam = op.params.get("index", "hnsw")\n'
        '                base = self.calib.ann_ms.get(fam, 0.2) * (n / 1e4) ** (0.3 if fam == "hnsw" else 1.0)\n'
        '                vis = restricted\n'
        '                if op.params.get("where"):\n'
        '                    db_ms += n / 1000 * self.calib.db_ms_per_krow + 0.3\n'
        '                    vis *= sel\n'
        '                if vis < 1.0:   # predicate-restricted search: graph/list walks get longer\n'
        '                    base *= {"hnsw": min(1 / max(vis, 1e-3), 50) ** 0.5, "ivf": 4.0, "flat": 1.0}.get(fam, 1.0)\n'
        '                ann_ms += base * (int(op.params.get("top_k", 10)) / 10) ** 0.2\n'
        '                rows = min(float(op.params.get("top_k", 10)), n * vis)\n'
        '            elif k == "rerank":\n'
        '                r_in = rows if rows is not None else n\n'
        '                ce_ms += r_in * self.calib.ce_ms_per_pair\n'
        '                rows = min(float(op.params.get("top_k", 5)), r_in)\n'
        '            elif k == "map":\n'
        '                r_in = rows if rows is not None else n\n'
        '                if op.params.get("max_records"):\n'
        '                    r_in = min(r_in, float(op.params["max_records"]))\n'
        '                u, ms, t = self._llm_call("map", TEMPLATE_TOKENS["map"] + q_tok + min(doc_tok * 1.6, 300))\n'
        '                usd_tok += u * r_in\n'
        '                llm_ms += ms * r_in\n'
        '                tokens += t * r_in\n'
        '                calls += r_in\n'
        '                rows = r_in * MAP_KEEP_RATE\n'
        '            elif k == "generate":\n'
        '                r_in = min(rows if rows is not None else n, float(op.params.get("max_docs", 6)))\n'
        '                per_doc = 60 if plan.find("map") is not None else doc_tok\n'
        '                u, ms, t = self._llm_call("generate", TEMPLATE_TOKENS["generate"] + q_tok + r_in * per_doc)\n'
        '                usd_tok, llm_ms, tokens, calls = usd_tok + u, llm_ms + ms, tokens + t, calls + 1\n'
        '            elif k == "sql_generate":\n'
        '                schema_tok = float(stats.get("schema_tokens", 150))\n'
        '                n_tab = float(stats.get("n_tables", 5))\n'
        '                if rows is not None:\n'
        '                    schema_tok *= min(1.0, rows / max(n_tab, 1.0))\n'
        '                shots = int(op.params.get("fewshot_k", 0)) * 45\n'
        '                u, ms, t = self._llm_call("sql_generate", TEMPLATE_TOKENS["sql_generate"] + q_tok + schema_tok + shots)\n'
        '                usd_tok, llm_ms, tokens, calls = usd_tok + u, llm_ms + ms, tokens + t, calls + 1\n'
        '            elif k == "sql_repair":\n'
        '                db_ms += 3.0\n'
        '                u, ms, t = self._llm_call("sql_repair", TEMPLATE_TOKENS["sql_repair"] + q_tok\n'
        '                                          + float(stats.get("schema_tokens", 150)))\n'
        '                usd_tok += REPAIR_RATE * u\n'
        '                llm_ms += REPAIR_RATE * ms\n'
        '                tokens += REPAIR_RATE * t\n'
        '                calls += REPAIR_RATE\n'
        '            elif k == "sql_execute":\n'
        '                db_ms += 3.0\n'
        '            after = (db_ms, ann_ms, ce_ms, llm_ms, usd_tok, tokens)\n'
        '            per_op.append({"op": op.name, "kind": k, "rows_out": rows,\n'
        '                           "ms": sum(after[i] - before[i] for i in range(4)),\n'
        '                           "usd": after[4] - before[4], "tokens": after[5] - before[5]})\n'
        '        cpu_ms = db_ms + ann_ms + ce_ms\n'
        '        usd = usd_tok + cpu_ms * self.cpu_usd_per_ms\n'
        '        return CostEstimate(usd=usd, latency_ms=cpu_ms + llm_ms, tokens=tokens, llm_calls=calls,\n'
        '                            breakdown={"db_ms": db_ms, "ann_ms": ann_ms, "rerank_ms": ce_ms,\n'
        '                                       "llm_ms": llm_ms, "token_usd": usd_tok,\n'
        '                                       "cpu_usd": cpu_ms * self.cpu_usd_per_ms, "per_op": per_op},\n'
        '                            mode="symbolic")\n'
        '\n'
        '    # ------------------------------------------------------------ features\n'
        '    def features(self, plan: Pipeline, stats: dict, sym: CostEstimate | None = None) -> list[float]:\n'
        '        sym = sym or self.symbolic(plan, stats)\n'
        '        r = plan.find("retrieve")\n'
        '        idx = r.params.get("index", "hnsw") if r else None\n'
        '        g = plan.find("sql_generate")\n'
        '        fam = stats.get("family", "")\n'
        '        return [\n'
        '            math.log10(max(stats.get("n_rows", 1), 1)), float(stats.get("sel_est", 1.0)),\n'
        '            math.log10(max(stats.get("avg_doc_tokens", 1), 1)), sym.llm_calls,\n'
        '            math.log10(max(sym.tokens, 1)), float(r.params.get("top_k", 0)) if r else 0.0,\n'
        '            float(plan.find("rerank") is not None), float(plan.find("map") is not None),\n'
        '            float(bool(r and r.params.get("where"))),\n'
        '            float(r is not None and plan.find("filter") is not None\n'
        '                  and plan.index_of(plan.find("filter").name) > plan.index_of(r.name)),\n'
        '            float(idx == "hnsw"), float(idx == "ivf"), float(idx == "flat"),\n'
        '            float(g.params.get("fewshot_k", 0)) if g else 0.0, float(plan.find("sql_repair") is not None),\n'
        '            float(fam == "retrieval-heavy"), float(fam == "mixed"), float(fam == "sql-heavy"),\n'
        '            math.log10(max(sym.usd, 1e-9)), math.log10(max(sym.latency_ms, 1e-3)),\n'
        '        ]\n'
        '\n'
        '    # ------------------------------------------------------------ learning\n'
        '    def fit(self, X: list[list[float]], obs_usd: list[float], obs_ms: list[float],\n'
        '            sym_usd: list[float], sym_ms: list[float], min_samples: int = 12) -> bool:\n'
        '        if len(X) < min_samples:\n'
        '            return False\n'
        '        from sklearn.ensemble import GradientBoostingRegressor\n'
        '        Xa = np.asarray(X, dtype=float)\n'
        '        lo = lambda v: np.log(np.maximum(np.asarray(v, dtype=float), 1e-9))\n'
        '        mk = lambda: GradientBoostingRegressor(n_estimators=150, max_depth=3, learning_rate=0.06,\n'
        '                                               subsample=0.9, random_state=0)\n'
        '        self._res_usd = mk().fit(Xa, lo(obs_usd) - lo(sym_usd))\n'
        '        self._res_ms = mk().fit(Xa, lo(obs_ms) - lo(sym_ms))\n'
        '        Xn = Xa[:, :-2]   # neural-only model may not see the symbolic estimate\n'
        '        self._nn_usd = mk().fit(Xn, lo(obs_usd))\n'
        '        self._nn_ms = mk().fit(Xn, lo(obs_ms))\n'
        '        self.n_train = len(X)\n'
        '        return True\n'
        '\n'
        '    @property\n'
        '    def fitted(self) -> bool:\n'
        '        return self._res_usd is not None\n'
        '\n'
        '    def predict(self, plan: Pipeline, stats: dict, mode: str | None = None) -> CostEstimate:\n'
        '        mode = mode or self.mode\n'
        '        sym = self.symbolic(plan, stats)\n'
        '        if mode == "symbolic" or not self.fitted:\n'
        '            return sym\n'
        '        x = np.asarray(self.features(plan, stats, sym), dtype=float).reshape(1, -1)\n'
        '        if mode == "neural":\n'
        '            usd = float(np.exp(self._nn_usd.predict(x[:, :-2])[0]))\n'
        '            ms = float(np.exp(self._nn_ms.predict(x[:, :-2])[0]))\n'
        '        else:\n'
        '            usd = sym.usd * float(np.exp(self._res_usd.predict(x)[0]))\n'
        '            ms = sym.latency_ms * float(np.exp(self._res_ms.predict(x)[0]))\n'
        '        scale = usd / sym.usd if sym.usd > 0 else 1.0\n'
        '        return CostEstimate(usd=usd, latency_ms=ms, tokens=sym.tokens * scale,\n'
        '                            llm_calls=sym.llm_calls, breakdown={**sym.breakdown, "symbolic_usd": sym.usd,\n'
        '                                                                "symbolic_ms": sym.latency_ms},\n'
        '                            mode=mode)\n'
        '\n'
        '    def validation_cost(self, cand: Pipeline, inc: Pipeline, stats: dict, n_items: int,\n'
        '                        n_criteria: int) -> tuple[float, float]:\n'
        '        """Expected (usd, tokens) of validating ``cand`` against ``inc`` on a sample.\n'
        '\n'
        '        Covers executing both plans on the sample plus one rating call per\n'
        '        output and a few pairwise calls - the spend the VoI gate weighs.\n'
        '        """\n'
        '        c = self.predict(cand, stats)\n'
        '        p = self.predict(inc, stats)\n'
        '        rate_tok = 420 + 25 * n_criteria\n'
        '        tokens = n_items * (c.tokens + p.tokens + 2 * rate_tok) + 3 * 520\n'
        '        usd = n_items * (c.usd + p.usd) + (2 * n_items * rate_tok + 3 * 520) * (0.9 * self.p_in + 0.1 * self.p_out)\n'
        '        return usd, tokens\n'
    ),
})
run_selected(['memory', 'cost_model', 'latency', 'ablation'])

plannerllm/core/memory.py
plannerllm/core/signature.py
plannerllm/core/cost_model.py
memory | seed 0 | 60 progress entries | reproduced_runs\recorded_groq\openai_gpt-oss-20b_seed0\memory.json
   plannerllm | spider:world_1 | pass 1
   plannerllm | spider:orchestra | pass 2
   plannerllm | spider:world_1 | pass 2
memory | seed 1 | 60 progress entries | reproduced_runs\recorded_groq\openai_gpt-oss-20b_seed1\memory.json
   plannerllm | spider:dog_kennels | pass 1
   plannerllm | spider:pets_1 | pass 2
   plannerllm | spider:dog_kennels | pass 2
memory | seed 2 | 60 progress entries | reproduced_runs\recorded_groq\openai_gpt-oss-20b_seed2\memory.json
   plannerllm | spider:employee_hire_evaluation | pass 1
   plannerllm | spider:flight_2 | pass 2
   plannerllm | spider:employee_hire_evaluation | pass 2
cost_model | seed 0 | 3 progress entries | reproduced_runs\recorded_groq\openai_gpt-oss-20b_seed0\cost_model.json
   cost telemetry | msmarco
   cost telemetry | cuad
   cost telemetry | spi

"""Plan memory (Sections 5.2 and 5.5).

Persistent SQLite store that turns one-shot agentic optimisation into a
learning optimiser.  It keeps

(i)   validated rewrite verdicts keyed by applicability signature,
(ii)  per-directive flip statistics  h(dir),
(iii) local-judge vs LLM-validator agreement, per directive,
(iv)  observed-cost telemetry for the residual cost model,
(v)   a gate log used to calibrate P(flip) and to score the gate offline.

Verdict confidence decays with age and with context drift:

    delta(t) = exp(-kappa * age_days) * exp(-drift_penalty * drift) * n/(n + 1)

With the default reuse threshold a verdict must be confirmed by two
independent validations before it is reused at all (n/(n+1) = 0.5 -> 0.67).

so a pattern confirmed several times recently is trusted, a stale one or one
validated under different statistics is re-validated instead of reused.
"""
from __future__ import annotations

import json
import math
import sqlite3
import threading
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np

from .signature import drift as ctx_drift

SCHEMA = """
CREATE TABLE IF NOT EXISTS rewrites (
  sig TEXT PRIMARY KEY, directive TEXT, family TEXT, parent_shape TEXT,
  features TEXT, vec TEXT, verdict TEXT, acc_delta REAL, cost_delta REAL,
  n_obs INT, n_agree INT, created REAL, updated REAL, workload TEXT);
CREATE TABLE IF NOT EXISTS flip_stats (directive TEXT PRIMARY KEY, flips INT, trials INT);
CREATE TABLE IF NOT EXISTS agreement (directive TEXT PRIMARY KEY, agree INT, total INT);
CREATE TABLE IF NOT EXISTS gate_log (
  id INTEGER PRIMARY KEY AUTOINCREMENT, directive TEXT, p_raw REAL, p_used REAL,
  voi REAL, token_cost REAL, validated INT, flipped INT, ts REAL, workload TEXT);
CREATE TABLE IF NOT EXISTS telemetry (
  id INTEGER PRIMARY KEY AUTOINCREMENT, workload TEXT, plan_fp TEXT, features TEXT,
  sym_usd REAL, sym_ms REAL, obs_usd REAL, obs_ms REAL, ts REAL);
CREATE TABLE IF NOT EXISTS retrain_log (ts REAL, n_telemetry INT, n_gate INT, note TEXT);
"""


STAT_KEYS = {"rows_b", "sel_b", "doclen_b", "tables_b"}


def same_structure(a: dict, b: dict) -> bool:
    """Equal on every signature feature except the bucketed data statistics."""
    keys = (set(a) | set(b)) - STAT_KEYS
    return all(str(a.get(k)) == str(b.get(k)) for k in keys)


@dataclass
class MemoryHit:
    sig: str
    directive: str
    prefer: bool
    confidence: float
    acc_delta: float
    match: str          # exact | soft
    age_days: float
    drift: float
    n_obs: int

    def to_dict(self) -> dict:
        return dict(vars(self))


class PlanMemory:
    def __init__(self, path: str | Path = ":memory:", kappa: float = 0.02, drift_penalty: float = 3.0,
                 reuse_threshold: float = 0.55, soft_match_threshold: float = 0.90,
                 retrain_every_days: float = 7.0, clock=None) -> None:
        self.path = str(path)
        if self.path != ":memory:":
            Path(self.path).parent.mkdir(parents=True, exist_ok=True)
        self._db = sqlite3.connect(self.path, check_same_thread=False)
        self._lock = threading.RLock()
        with self._lock:
            self._db.executescript(SCHEMA)
            self._db.commit()
        self.kappa, self.drift_penalty = kappa, drift_penalty
        self.reuse_threshold, self.soft_threshold = reuse_threshold, soft_match_threshold
        self.retrain_every = retrain_every_days
        self.clock = clock or (lambda: time.time() / 86400.0)
        self._last_retrain = self.clock()
        self._calib: tuple[float, float, int] | None = None

    @classmethod
    def from_settings(cls, ms, path: str | None = None, clock=None) -> "PlanMemory":
        return cls(path or ms.path, ms.kappa, ms.drift_penalty, ms.reuse_threshold,
                   ms.soft_match_threshold, ms.retrain_every_days, clock)

    def _q(self, sql: str, args: tuple = ()):
        with self._lock:
            return self._db.execute(sql, args).fetchall()

    def _x

"""Applicability signatures (Section 5.2).

A rewrite's verdict is only reusable in a context that looks like the one in
which it was validated.  The signature captures that context from four
sources named in the methodology: template type (operator shape), data
statistics (cardinality, selectivity, document length), operator
configuration (index family, top-k), and prompt structure.  Continuous
statistics are bucketed so that "the same kind of query" hashes identically
while a real shift (e.g. selectivity moving an order of magnitude) does not.
"""
from __future__ import annotations

import hashlib
import json
import math
import re

import numpy as np

from .pipeline import Pipeline


def _bucket(value: float | None, edges: list[float]) -> int:
    if value is None:
        return -1
    for i, e in enumerate(edges):
        if value <= e:
            return i
    return len(edges)


def prompt_structure(plan: Pipeline) -> str:
    """Shape of the instruction text, not its words: length band + format cues."""
    parts = []
    for o in plan.topological():
        if not o.prompt:
            continue
        n_sent = len([s for s in re.split(r"[.!?]+", o.prompt) if s.strip()])
        cues = "".join(c for c, rx in (("N", r"\bNONE\b"), ("Q", r"quote|verbatim|word for word"),
                                       ("F", r"format|json|one sentence|short"))
                       if re.search(rx, o.prompt, re.I))
        parts.append(f"{o.kind}:{min(n_sent, 4)}{cues}")
    return "|".join(parts)


def _retrieve_cfg(plan: Pipeline) -> tuple[int | None, str | None, bool]:
    r = plan.find("retrieve")
    if r is None:
        return None, None, False
    return int(r.params.get("top_k", 10)), r.params.get("index", "hnsw"), bool(r.params.get("where"))


def config_shape(plan: Pipeline) -> str:
    """Physical configuration of a plan with top-k bucketed (part of the context)."""
    k, index, filtered = _retrieve_cfg(plan)
    gen = plan.find("sql_generate")
    shots = int(gen.params.get("fewshot_k", 0)) if gen else 0
    return f"{index}|{int(filtered)}|{_bucket(k, [5, 10, 20])}|{shots}|{prompt_structure(plan)}"


def context_features(parent: Pipeline, cand: Pipeline, directive: str, stats: dict) -> dict:
    k, index, filtered = _retrieve_cfg(cand)
    gen = cand.find("sql_generate")
    return {
        "directive": directive,
        "family": stats.get("family"),
        "parent_shape": "->".join(parent.kinds()),
        "parent_cfg": config_shape(parent),
        "cand_shape": "->".join(cand.kinds()),
        "rows_b": _bucket(math.log2(max(stats.get("n_rows", 1), 1)), [8, 10, 12, 14, 17, 20]),
        "sel_b": _bucket(stats.get("sel_est"), [0.005, 0.02, 0.06, 0.15, 0.4, 0.99]),
        "doclen_b": _bucket(stats.get("avg_doc_tokens"), [64, 128, 256, 512]),
        "topk_b": _bucket(k, [5, 10, 20]),
        "index": index,
        "filtered_probe": filtered,
        "fewshot": int(gen.params.get("fewshot_k", 0)) if gen else None,
        "tables_b": _bucket(stats.get("n_tables"), [3, 5, 8, 12]),
        "prompt": prompt_structure(cand),
    }


def signature_of(features: dict) -> str:
    blob = json.dumps(features, sort_keys=True, default=str)
    return hashlib.sha1(blob.encode()).hexdigest()[:16]


def numeric_vector(stats: dict, cand: Pipeline) -> np.ndarray:
    """Continuous description used for near-signature (soft) matches and drift."""
    k, _, filtered = _retrieve_cfg(cand)
    return np.array([
        math.log10(max(stats.get("n_rows", 1), 1)),
        math.log10(max(stats.get("sel_est", 1.0), 1e-4)),
        math.log10(max(stats.get("avg_doc_tokens", 1.0), 1.0)),
        math.log2(max(k or 1, 1)),
        float(filtered),
        math.log2(max(stats.get("n_tables", 1) or 1, 1)),
    ], dtype=float)


def drift(a: np.ndarray, b: np.ndarray) -> float:
    """Distance between two contexts in log-space (0 == identical)."""
    return float(np.linalg.norm(np.asarray(a) - np.asarray(b)) / math.sqrt(len(a)))


def cosi

"""Hybrid symbolic-neural cost model (Section 5.3).

    C^(c) = sum_op f_sym(op; stats)  +  f_ANN(index, n, d, k)  +  sum_llm tokens * price
    C^_hybrid(c) = C^(c) * exp(r^_nn(features(c)))

The symbolic part walks the plan once, propagating estimated row counts with
DuckDB's optimizer selectivity and the classical independence assumption.
It is deliberately the "textbook" estimator: it is wrong exactly where the
literature says it is wrong (correlated predicates - e.g. a post-filter after
a semantic probe keeps far more rows than the corpus-wide selectivity says).
The learned residual is a small gradient-boosted model on the log-ratio
between observed and symbolic cost, so it can only *correct* the backbone,
never replace it - unless the ``neural`` ablation is requested.
"""
from __future__ import annotations

import math
from dataclasses import dataclass, field

import numpy as np

from .pipeline import Pipeline

# token sizes of the fixed parts of our prompt templates (measured once)
TEMPLATE_TOKENS = {"generate": 70, "map": 60, "sql_generate": 45, "sql_repair": 70}
OUTPUT_TOKENS = {"generate": 45, "map": 35, "sql_generate": 40, "sql_repair": 40}
MAP_KEEP_RATE = 0.5
REPAIR_RATE = 0.3


@dataclass
class CostEstimate:
    usd: float
    latency_ms: float
    tokens: float
    llm_calls: float
    breakdown: dict = field(default_factory=dict)
    mode: str = "symbolic"

    def to_dict(self) -> dict:
        return {"usd": self.usd, "latency_ms": self.latency_ms, "tokens": self.tokens,
                "llm_calls": self.llm_calls, "mode": self.mode, "breakdown": self.breakdown}


@dataclass
class Calibration:
    """Machine-specific constants, measured by microbenchmarks at start-up."""
    ann_ms: dict = field(default_factory=lambda: {"hnsw": 0.15, "ivf": 0.10, "flat": 0.6})
    ce_ms_per_pair: float = 2.5
    db_ms_per_krow: float = 0.4
    llm_overhead_s: float = 0.25
    llm_s_per_in_token: float = 0.00015
    llm_s_per_out_token: float = 0.004


class HybridCostModel:
    FEATURES = ["log_rows", "sel_est", "log_doc_tokens", "llm_calls", "log_tokens", "top_k",
                "has_rerank", "has_map", "filtered_probe", "post_filter", "idx_hnsw", "idx_ivf",
                "idx_flat", "fewshot", "has_repair", "fam_retrieval", "fam_mixed", "fam_sql",
                "log_sym_usd", "log_sym_ms"]

    def __init__(self, price_in_per_m: float, price_out_per_m: float, cpu_usd_per_hour: float,
                 calibration: Calibration | None = None, mode: str = "hybrid") -> None:
        self.p_in = price_in_per_m / 1e6
        self.p_out = price_out_per_m / 1e6
        self.cpu_usd_per_ms = cpu_usd_per_hour / 3.6e6
        self.calib = calibration or Calibration()
        self.mode = mode                      # symbolic | neural | hybrid
        self._res_usd = self._res_ms = None   # residual regressors
        self._nn_usd = self._nn_ms = None     # neural-only regressors
        self.n_train = 0

    # ------------------------------------------------------------ symbolic
    def _llm_call(self, kind: str, prompt_tokens: float) -> tuple[float, float, float]:
        out = OUTPUT_TOKENS.get(kind, 40)
        usd = prompt_tokens * self.p_in + out * self.p_out
        sec = self.calib.llm_overhead_s + prompt_tokens * self.calib.llm_s_per_in_token \
            + out * self.calib.llm_s_per_out_token
        return usd, sec * 1000, prompt_tokens + out

    def symbolic(self, plan: Pipeline, stats: dict) -> CostEstimate:
        n = float(stats.get("n_rows", 1000))
        sel = float(stats.get("sel_est", 1.0))
        doc_tok = min(float(stats.get("avg_doc_tokens", 120)), 175.0)
        q_tok = float(stats.get("avg_query_tokens", 12))
        rows: float | None = None            # None == whole corpus still in play
        restricted = 1.0                     # fraction of corpus visible to the probe
        db_ms = ann_ms = ce_ms = llm_ms = usd_tok = tokens = calls = 0.0
        per_op = []
        for op in plan.topological():
 

## Decision traces and independent evaluation

In [7]:
# Source snapshot. The experiment below executes the imported copy.
show_implementation({
    'eval/indep_judge.py': (
        '"""Independent judge: a second LLM, from a different model family, used only for evaluation.\n'
        '\n'
        "The optimizer's validation agent and the pipeline's answer generator are the same model\n"
        "(gpt-oss-20b), so a verdict could share the generator's blind spots.  This module asks a\n"
        'model from another family (Qwen, by default) two questions the optimizer never sees:\n'
        '\n'
        '* ``grade``    - is a held-out answer correct with respect to the human reference answers?\n'
        '                 This gives an accuracy that does not depend on word overlap.\n'
        '* ``compare``  - given the same two outputs the validation agent compared, which one is better?\n'
        "                 Agreement with the validation agent's verdict measures judge bias.\n"
        '\n'
        'Both calls go through the normal client, so they are cached and rate limited, and their\n'
        'tokens are kept in a separate ledger (they are evaluation cost, not optimisation cost).\n'
        '"""\n'
        'from __future__ import annotations\n'
        '\n'
        'import hashlib\n'
        'import json\n'
        '\n'
        'from ..llm.base import TokenLedger\n'
        'from ..llm.cache import ResponseCache\n'
        'from ..llm.client import GroqClient\n'
        '\n'
        'SYSTEM = ("You are an impartial grader of question-answering and information-extraction systems. "\n'
        '          "You compare outputs carefully and reply only with the JSON object you are asked for.")\n'
        '\n'
        'GRADE = """Task of the system: {task}\n'
        '\n'
        'Question: {question}\n'
        '\n'
        'Reference answer(s) written by people:\n'
        '{refs}\n'
        '\n'
        'System answer:\n'
        '{answer}\n'
        '\n'
        'Grade the system answer against the references.\n'
        '- "correct": it states the information of at least one reference answer. Wording may differ and extra correct detail is fine.\n'
        '- "partial": it contains only part of that information, or is too vague to be fully correct.\n'
        '- "incorrect": it is wrong, unrelated, or says the information is missing when a reference exists.\n'
        'A reference of NONE means the source has no such information; then an answer of NONE (or "not found") is correct and any other answer is incorrect.\n'
        'Reply with JSON: {{"verdict": "correct" | "partial" | "incorrect"}}"""\n'
        '\n'
        'COMPARE = """Task of the system: {task}\n'
        '\n'
        'Question: {question}\n'
        '\n'
        'Output A:\n'
        '{a}\n'
        '\n'
        'Output B:\n'
        '{b}\n'
        '\n'
        'Which output accomplishes the task better for this question? Judge correctness, completeness and\n'
        'faithfulness to the task; do not prefer an output only because it is longer.\n'
        'Reply with JSON: {{"better": "A" | "B" | "tie"}}"""\n'
        '\n'
        'SCORES = {"correct": 1.0, "partial": 0.5, "incorrect": 0.0}\n'
        '\n'
        '\n'
        'class IndependentJudge:\n'
        '    def __init__(self, settings) -> None:\n'
        '        cfg = settings.llm\n'
        '        self.model = cfg.judge_model\n'
        '        self.ledger = TokenLedger(cfg.price_in_per_m, cfg.price_out_per_m, None)\n'
        '        self.client = GroqClient(self.ledger, ResponseCache(cfg.cache_path), model=self.model,\n'
        '                                 base_url=cfg.base_url, temperature=0.0, rpm=cfg.requests_per_minute,\n'
        '                                 tpm=cfg.tokens_per_minute, max_retries=cfg.max_retries,\n'
        '                                 timeout_s=cfg.timeout_s, extra_body={"reasoning_effort": "none"},\n'
        '                                 reasoning_tokens=0)\n'
        '\n'
        '    def grade(self, task: str, question: str, refs: list[str], answer: str) -> float | None:\n'
        '        refs_txt = "\\n".join(f"- {r[:600]}" for r in refs[:4]) or "- NONE"\n'
        '        msg = GRADE.format(task=task, question=question, refs=refs_txt, answer=(answer or "(empty)")[:800])\n'
        '        r = self.client.complete(msg, purpose="judge", task="judge_grade", system=SYSTEM, max_tokens=60,\n'
        '                                 json_mode=True)\n'
        '        v = str((r.json({}) or {}).get("verdict", "")).strip().lower()\n'
        '        return SCORES.get(v)\n'
        '\n'
        '    def compare(self, task: str, question: str, candidate: str, incumbent: str, key: str) -> str | None:\n'
        '        """Returns \'candidate\', \'incumbent\' or \'tie\'.  The A/B position is decided by a hash of ``key``\n'
        '        so that position bias averages out over items."""\n'
        '        swap = int(hashlib.sha256(key.encode()).hexdigest(), 16) % 2 == 1\n'
        '        a, b = (incumbent, candidate) if swap else (candidate, incumbent)\n'
        '        msg = COMPARE.format(task=task, question=question, a=(a or "(empty)")[:800], b=(b or "(empty)")[:800])\n'
        '        r = self.client.complete(msg, purpose="judge", task="judge_compare", system=SYSTEM, max_tokens=60,\n'
        '                                 json_mode=True)\n'
        '        v = str((r.json({}) or {}).get("better", "")).strip().upper()\n'
        '        if v not in ("A", "B", "TIE"):\n'
        '            return None\n'
        '        if v == "TIE":\n'
        '            return "tie"\n'
        '        first = "incumbent" if swap else "candidate"\n'
        '        second = "candidate" if swap else "incumbent"\n'
        '        return first if v == "A" else second\n'
        '\n'
        '    def usage(self) -> dict:\n'
        '        return self.ledger.totals()\n'
        '\n'
        '\n'
        'def cohen_kappa(a: list[bool], b: list[bool]) -> float | None:\n'
        '    n = len(a)\n'
        '    if n == 0:\n'
        '        return None\n'
        '    po = sum(x == y for x, y in zip(a, b)) / n\n'
        '    pa, pb = sum(a) / n, sum(b) / n\n'
        '    pe = pa * pb + (1 - pa) * (1 - pb)\n'
        '    return (po - pe) / (1 - pe) if pe < 1 else None\n'
        '\n'
        '\n'
        'def dumps(o) -> str:\n'
        '    return json.dumps(o, sort_keys=True)\n'
    ),
    'eval/experiments.py': (
        '"""Experiment suite for Section 10 (one function per subsection).\n'
        '\n'
        'Every experiment writes ``runs/<backend>/<experiment>.json`` so the dashboard\n'
        'and ``reporting.py`` can render it without re-running anything.  Runs are\n'
        'resumable for free: every LLM response is cached on disk, so re-running an\n'
        'interrupted experiment replays finished calls instantly.\n'
        '\n'
        '    E1 baselines        10.1 / 10.2 table: accuracy, spend, latency per system\n'
        '    E2 pareto           10.2 accuracy vs optimisation-token budget, theta sweep\n'
        '    E3 gate             10.3 gate precision / recall / calibration (shadow mode)\n'
        '    E4 memory           10.4 hit rate, reuse accuracy, decay, perturbed shapes\n'
        '    E5 cost_model       10.5 calibration error: symbolic / neural / hybrid, zero-shot\n'
        '    E6 latency          10.6 wall-clock decomposition, ANN family sweep\n'
        '    E7 ablation         10.7 -gate, -memory, symbolic-only, neural-only\n'
        '    E8 transcripts      10.8 validation transcripts for accepted rewrites\n'
        '"""\n'
        'from __future__ import annotations\n'
        '\n'
        'import json\n'
        'import math\n'
        'import random\n'
        'import time\n'
        'from pathlib import Path\n'
        'from typing import Callable\n'
        '\n'
        'import numpy as np\n'
        '\n'
        'from ..core.directives import DirectiveContext\n'
        'from ..core.memory import PlanMemory\n'
        'from ..core.optimizer import ABLATIONS, SYSTEMS, make_system\n'
        'from ..core.session import Session, build_session, mean_observed\n'
        'from ..settings import RUNS_DIR, Settings\n'
        '\n'
        'PROFILES = {   # groups per workload, optimisation questions, held-out questions\n'
        '    "smoke": {"n_tasks": 1, "n_opt": 5, "n_test": 5},\n'
        '    "quick": {"n_tasks": 2, "n_opt": 6, "n_test": 6},\n'
        '    "full": {"n_tasks": 4, "n_opt": 8, "n_test": 10},\n'
        '}\n'
        'WORKLOADS = ["msmarco", "cuad", "spider"]\n'
        'Progress = Callable[[str], None]\n'
        '\n'
        '\n'
        'def _noop(_: str) -> None:\n'
        '    pass\n'
        '\n'
        '\n'
        'def out_dir(settings: Settings) -> Path:\n'
        '    tag = "sim" if settings.llm.backend == "sim" else settings.llm.model.replace("/", "_")\n'
        '    d = RUNS_DIR / f"{tag}_seed{settings.sampling.task_seed}"\n'
        '    d.mkdir(parents=True, exist_ok=True)\n'
        '    return d\n'
        '\n'
        '\n'
        'def save(settings: Settings, name: str, payload: dict) -> Path:\n'
        '    payload = {"experiment": name, "backend": settings.llm.backend,\n'
        '               "model": settings.llm.model if settings.llm.backend != "sim" else "simulated-extractive",\n'
        '               "created": time.strftime("%Y-%m-%d %H:%M:%S"), "seed": settings.sampling.task_seed,\n'
        '               "sampling_seed": settings.sampling.seed, **payload}\n'
        '    path = out_dir(settings) / f"{name}.json"\n'
        '    path.write_text(json.dumps(payload, indent=1, default=_json_default), encoding="utf-8")\n'
        '    return path\n'
        '\n'
        '\n'
        'def load(settings: Settings, name: str) -> dict | None:\n'
        '    path = out_dir(settings) / f"{name}.json"\n'
        '    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None\n'
        '\n'
        '\n'
        'def _json_default(o):\n'
        '    if isinstance(o, (np.floating, np.integer)):\n'
        '        return o.item()\n'
        '    if isinstance(o, np.ndarray):\n'
        '        return o.tolist()\n'
        '    if hasattr(o, "to_dict"):\n'
        '        return o.to_dict()\n'
        '    return str(o)\n'
        '\n'
        '\n'
        'class LogicalClock:\n'
        '    """Days advance per optimised task, so decay is reproducible."""\n'
        '\n'
        '    def __init__(self, days_per_task: float = 1.0) -> None:\n'
        '        self.day, self.step = 0.0, days_per_task\n'
        '\n'
        '    def __call__(self) -> float:\n'
        '        return self.day\n'
        '\n'
        '    def tick(self) -> None:\n'
        '        self.day += self.step\n'
        '\n'
        '\n'
        'def _session(wl: str, settings: Settings, clock: LogicalClock | None = None, **kw) -> Session:\n'
        '    clock = clock or LogicalClock()\n'
        '    s = build_session(wl, settings, clock=clock, **kw)\n'
        '    s.clock = clock   # type: ignore[attr-defined]\n'
        '    return s\n'
        '\n'
        '\n'
        'def evaluate_plan(s: Session, plan, queries: list[dict], log_telemetry: bool = True) -> dict:\n'
        '    results = [s.executor.run(plan, q, purpose="execute") for q in queries]\n'
        '    sc = s.oracle.score(queries, results)\n'
        '    usd, ms = mean_observed(s, results)\n'
        '    br = [r.breakdown() for r in results]\n'
        '    sc.update(exec_usd=usd, exec_ms=ms,\n'
        '              db_ms=float(np.mean([b["db_ms"] for b in br])), ann_ms=float(np.mean([b["ann_ms"] for b in br])),\n'
        '              llm_ms=float(np.mean([b["llm_ms"] for b in br])), exec_tokens=float(np.mean([r.tokens for r in results])))\n'
        '    if log_telemetry:\n'
        '        cm = s.cost_model\n'
        '        stats = s.workload.group_stats(queries)\n'
        '        sym = cm.symbolic(plan, stats)\n'
        '        s.memory.add_telemetry(s.workload.name, plan.fingerprint(), cm.features(plan, stats, sym),\n'
        '                               sym.usd, sym.latency_ms, usd, ms)\n'
        '    return sc\n'
        '\n'
        '\n'
        'def run_stream(s: Session, system: str, tasks, progress: Progress = _noop, passes: int = 1,\n'
        '               on_event=None, **kw) -> list[dict]:\n'
        '    """Optimise each task in order (memory carries over), then score on held-out queries."""\n'
        '    sysm = make_system(system, s, **kw)\n'
        '    rows = []\n'
        '    for p in range(passes):\n'
        '        for t in tasks:\n'
        '            progress(f"{system} | {t.task_id} | pass {p + 1}")\n'
        '            lat0 = s.ledger.totals(optimization_only=True)["latency_s"]\n'
        '            r = sysm.optimize(t, on_event=on_event)\n'
        '            opt_llm_s = s.ledger.totals(optimization_only=True)["latency_s"] - lat0\n'
        '            sc = evaluate_plan(s, r.plan, t.test_queries)\n'
        '            clock = getattr(s, "clock", None)\n'
        '            if clock is not None:\n'
        '                clock.tick()\n'
        '            per_query = sc.pop("per_query")\n'
        '            rows.append({"system": system, "workload": s.workload.name, "task": t.task_id, "pass": p + 1,\n'
        '                         "seed": s.settings.sampling.task_seed, **r.summary(), **sc, "opt_llm_s": opt_llm_s,\n'
        '                         "metric": per_query[0].get("metric") if per_query else None, "per_query": per_query,\n'
        '                         "audit": [a.to_dict() for a in r.audit],\n'
        '                         "memory_audit": list(getattr(sysm, "memory_audit", []))})\n'
        '    return rows\n'
        '\n'
        '\n'
        'def _mean(rows: list[dict], key: str) -> float | None:\n'
        '    vals = [r[key] for r in rows if r.get(key) is not None]\n'
        '    return float(np.mean(vals)) if vals else None\n'
        '\n'
        '\n'
        'def summarise(rows: list[dict], keys=("accuracy", "f1", "precision", "recall", "rouge_l", "hallucinated",\n'
        '                                      "exec_acc", "opt_llm_s",\n'
        '                                      "recall_at_k", "opt_tokens", "opt_usd", "exec_usd", "exec_ms",\n'
        '                                      "exec_tokens", "n_validations", "n_memory_hits", "n_gate_skips",\n'
        '                                      "wall_s", "db_ms", "ann_ms", "llm_ms")) -> list[dict]:\n'
        '    groups: dict[tuple, list[dict]] = {}\n'
        '    for r in rows:\n'
        '        groups.setdefault((r["workload"], r["system"]), []).append(r)\n'
        '    out = []\n'
        '    for (wl, sysname), rs in groups.items():\n'
        '        row = {"workload": wl, "system": sysname, "n_tasks": len(rs)}\n'
        '        for k in keys:\n'
        '            row[k] = _mean(rs, k)\n'
        '        out.append(row)\n'
        '    return out\n'
        '\n'
        '\n'
        '# ====================================================================== E1\n'
        'def exp_baselines(settings: Settings, profile: str = "quick", workloads=WORKLOADS,\n'
        '                  systems=SYSTEMS, progress: Progress = _noop) -> dict:\n'
        '    prof = PROFILES[profile]\n'
        '    rows = []\n'
        '    for wl in workloads:\n'
        '        for system in systems:\n'
        '            s = _session(wl, settings)\n'
        '            tasks = s.workload.tasks(**prof, seed=settings.sampling.task_seed)\n'
        '            rows += run_stream(s, system, tasks, progress)\n'
        '    # decision quality: agreement with the full-validation (DocETL) choice per task\n'
        '    ref = {(r["workload"], r["task"]): r["plan"] for r in rows if r["system"] == "docetl"}\n'
        '    for r in rows:\n'
        '        if (r["workload"], r["task"]) in ref:\n'
        '            r["agrees_with_full_validation"] = float(r["plan"] == ref[(r["workload"], r["task"])])\n'
        '    summary = summarise(rows)\n'
        '    for srow in summary:\n'
        '        rs = [r for r in rows if r["workload"] == srow["workload"] and r["system"] == srow["system"]]\n'
        '        srow["decision_agreement"] = _mean(rs, "agrees_with_full_validation")\n'
        '    return {"profile": profile, "rows": rows, "summary": summary}\n'
        '\n'
        '\n'
        '# ====================================================================== E2\n'
        'def exp_pareto(settings: Settings, profile: str = "quick", workloads=WORKLOADS,\n'
        '               budgets=(3000, 6000, 12000, 24000, None), thetas=(0.25, 0.5, 1.0, 2.0, 4.0),\n'
        '               progress: Progress = _noop) -> dict:\n'
        '    prof = PROFILES[profile]\n'
        '    points = []\n'
        '    for wl in workloads:\n'
        '        for system in ("plannerllm", "docetl", "no_gate"):\n'
        '            for b in budgets:\n'
        '                st = settings.copy(**{"budget.optimization_tokens": b})\n'
        '                s = _session(wl, st)\n'
        '                rows = run_stream(s, system, s.workload.tasks(**prof, seed=settings.sampling.task_seed), progress)\n'
        '                points.append({"workload": wl, "system": system, "budget": b, "sweep": "budget",\n'
        '                               "accuracy": _mean(rows, "accuracy"), "opt_tokens": _mean(rows, "opt_tokens"),\n'
        '                               "exec_usd": _mean(rows, "exec_usd")})\n'
        '        for th in thetas:\n'
        '            st = settings.copy(**{"gate.theta": th})\n'
        '            s = _session(wl, st)\n'
        '            rows = run_stream(s, "plannerllm", s.workload.tasks(**prof, seed=settings.sampling.task_seed), progress)\n'
        '            points.append({"workload": wl, "system": "plannerllm", "theta": th, "sweep": "theta",\n'
        '                           "accuracy": _mean(rows, "accuracy"), "opt_tokens": _mean(rows, "opt_tokens"),\n'
        '                           "gate_skips": _mean(rows, "n_gate_skips"), "validations": _mean(rows, "n_validations")})\n'
        '    return {"profile": profile, "points": points}\n'
        '\n'
        '\n'
        '# ====================================================================== E3\n'
        'def gate_scores(log: list[dict], thetas) -> list[dict]:\n'
        '    """Offline gate P/R: replay logged VoI against theta * validation cost."""\n'
        '    rows = [g for g in log if g["flipped"] is not None]\n'
        '    out = []\n'
        '    for th in thetas:\n'
        '        admitted = [g for g in rows if g["voi"] > th * g["token_cost"]]\n'
        '        flips = [g for g in rows if g["flipped"]]\n'
        '        tp = sum(1 for g in admitted if g["flipped"])\n'
        '        spend_all = sum(g["token_cost"] for g in rows) or 1.0\n'
        '        out.append({"theta": th, "n": len(rows), "admitted": len(admitted),\n'
        '                    "precision": tp / len(admitted) if admitted else None,\n'
        '                    "recall": tp / len(flips) if flips else None,\n'
        '                    "flip_rate": len(flips) / len(rows) if rows else None,\n'
        '                    "spend_saved": 1 - sum(g["token_cost"] for g in admitted) / spend_all})\n'
        '    return out\n'
        '\n'
        '\n'
        'def calibration_bins(log: list[dict], n_bins: int = 5) -> list[dict]:\n'
        '    rows = [g for g in log if g["flipped"] is not None]\n'
        '    bins = []\n'
        '    for i in range(n_bins):\n'
        '        lo, hi = i / n_bins, (i + 1) / n_bins\n'
        '        rs = [g for g in rows if lo <= g["p_used"] < hi or (i == n_bins - 1 and g["p_used"] == 1.0)]\n'
        '        if rs:\n'
        '            bins.append({"lo": lo, "hi": hi, "n": len(rs), "predicted": float(np.mean([g["p_used"] for g in rs])),\n'
        '                         "observed": float(np.mean([g["flipped"] for g in rs]))})\n'
        '    return bins\n'
        '\n'
        '\n'
        'def exp_gate(settings: Settings, profile: str = "quick", workloads=WORKLOADS,\n'
        '             thetas=(0.1, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0), progress: Progress = _noop) -> dict:\n'
        '    prof = PROFILES[profile]\n'
        '    out = {"profile": profile, "per_workload": {}}\n'
        '    all_log = []\n'
        '    for wl in workloads:\n'
        '        s = _session(wl, settings)\n'
        '        run_stream(s, "plannerllm", s.workload.tasks(**prof, seed=settings.sampling.task_seed), progress, passes=2, shadow_gate=True)\n'
        '        rep = s.memory.report()\n'
        '        log = rep["gate_log"]\n'
        '        all_log += log\n'
        '        per_dir: dict[str, dict] = {}\n'
        '        for g in log:\n'
        '            if g["flipped"] is None:\n'
        '                continue\n'
        '            d = per_dir.setdefault(g["directive"], {"n": 0, "flips": 0, "admitted": 0, "p_mean": 0.0})\n'
        '            d["n"] += 1\n'
        '            d["flips"] += g["flipped"]\n'
        '            d["admitted"] += int(g["voi"] > g["token_cost"])\n'
        '            d["p_mean"] += g["p_used"]\n'
        '        for d in per_dir.values():\n'
        '            d["p_mean"] /= max(d["n"], 1)\n'
        '        out["per_workload"][wl] = {"scores": gate_scores(log, thetas), "calibration": calibration_bins(log),\n'
        '                                   "per_directive": per_dir, "agreement": rep["agreement"], "log": log}\n'
        '    out["overall"] = {"scores": gate_scores(all_log, thetas), "calibration": calibration_bins(all_log)}\n'
        '    return out\n'
        '\n'
        '\n'
        '# ====================================================================== E4\n'
        'def _perturb(s: Session, kind: str) -> None:\n'
        '    """Change the starting pipeline so later queries have a *perturbed shape*."""\n'
        '    base = s.workload.base_pipeline()\n'
        '    orig = s.workload.base_pipeline\n'
        '    if kind == "prompt_edit":\n'
        '        for o in base.operators:\n'
        '            if o.prompt:\n'
        '                o.prompt = o.prompt.rstrip(".") + ". Keep the answer short and precise."\n'
        '    elif kind == "config_change":\n'
        '        r = base.find("retrieve")\n'
        '        if r is not None:\n'
        '            r.params["top_k"] = int(r.params.get("top_k", 10)) * 3\n'
        '        g = base.find("sql_generate")\n'
        '        if g is not None:\n'
        '            g.params["fewshot_k"] = 1\n'
        '    s.workload.base_pipeline = lambda: base.clone()  # type: ignore[method-assign]\n'
        '    s._restore_base = orig   # type: ignore[attr-defined]\n'
        '\n'
        '\n'
        'def _restore(s: Session) -> None:\n'
        '    if hasattr(s, "_restore_base"):\n'
        '        s.workload.base_pipeline = s._restore_base   # type: ignore[method-assign]\n'
        '\n'
        '\n'
        'def exp_memory(settings: Settings, profile: str = "quick", workloads=WORKLOADS,\n'
        '               kappas=(0.0, 0.02, 0.1), progress: Progress = _noop) -> dict:\n'
        '    prof = PROFILES[profile]\n'
        '    out = {"profile": profile, "streams": [], "decay": [], "reuse": []}\n'
        '    for wl in workloads:\n'
        '        # repeated and perturbed shapes, with re-validation of every hit\n'
        '        s = _session(wl, settings)\n'
        '        tasks = s.workload.tasks(**prof, seed=settings.sampling.task_seed)\n'
        '        phases = [("first-seen", None), ("repeated", None), ("prompt_edit", "prompt_edit"),\n'
        '                  ("config_change", "config_change")]\n'
        '        for label, pert in phases:\n'
        '            if pert:\n'
        '                _perturb(s, pert)\n'
        '            rows = run_stream(s, "plannerllm", tasks, progress, audit_memory=True)\n'
        '            _restore(s)\n'
        '            cands = sum(len(r["audit"]) for r in rows)\n'
        '            hits = sum(r["n_memory_hits"] for r in rows)\n'
        '            audit = [m for r in rows for m in r["memory_audit"]]\n'
        '            out["streams"].append({"workload": wl, "phase": label, "hit_rate": hits / cands if cands else 0.0,\n'
        '                                   "hits": hits, "candidates": cands, "opt_tokens": _mean(rows, "opt_tokens"),\n'
        '                                   "accuracy": _mean(rows, "accuracy"),\n'
        '                                   "reuse_accuracy": (float(np.mean([m["reused"] == m["fresh"] for m in audit]))\n'
        '                                                      if audit else None), "n_audited": len(audit)})\n'
        '            out["reuse"] += [{"workload": wl, "phase": label, **m} for m in audit]\n'
        '        # decay: same stream twice, clock advancing 3 days per task, different kappa\n'
        '        for kappa in kappas:\n'
        '            st = settings.copy(**{"memory.kappa": kappa})\n'
        '            s = _session(wl, st, clock=LogicalClock(3.0))\n'
        '            rows = run_stream(s, "plannerllm", tasks, progress, passes=2)\n'
        '            second = [r for r in rows if r["pass"] == 2]\n'
        '            cands = sum(len(r["audit"]) for r in second)\n'
        '            out["decay"].append({"workload": wl, "kappa": kappa,\n'
        '                                 "hit_rate_pass2": sum(r["n_memory_hits"] for r in second) / cands if cands else 0,\n'
        '                                 "opt_tokens_pass2": _mean(second, "opt_tokens"),\n'
        '                                 "accuracy_pass2": _mean(second, "accuracy")})\n'
        '    mem = PlanMemory(kappa=0.02)\n'
        '    out["decay_curve"] = [{"age_days": a, **{f"kappa={k}": math.exp(-k * a) * (2 / 3) for k in kappas}}\n'
        '                          for a in range(0, 61, 5)]\n'
        '    out["reuse_threshold"] = mem.reuse_threshold\n'
        '    return out\n'
        '\n'
        '\n'
        '# ====================================================================== E5\n'
        'def collect_cost_telemetry(s: Session, tasks, n_queries: int = 3, depth: int = 2) -> list[dict]:\n'
        '    """Execute a spread of rewritten plans to obtain observed costs."""\n'
        '    rows = []\n'
        '    for t in tasks:\n'
        '        ctx: DirectiveContext = s.directive_context(t.stats)\n'
        '        ctx.llm = None\n'
        '        frontier = [s.workload.base_pipeline()]\n'
        '        plans = list(frontier)\n'
        '        for _ in range(depth):\n'
        '            nxt = []\n'
        '            for p in frontier:\n'
        '                nxt += [c.plan for c in s.directives.expand(p, ctx)]\n'
        '            random.Random(t.task_id).shuffle(nxt)\n'
        '            frontier = nxt[:6]\n'
        '            plans += frontier\n'
        '        seen = set()\n'
        '        for plan in plans:\n'
        '            if plan.fingerprint() in seen:\n'
        '                continue\n'
        '            seen.add(plan.fingerprint())\n'
        '            res = [s.executor.run(plan, q, purpose="execute") for q in t.opt_queries[:n_queries]]\n'
        '            usd, ms = mean_observed(s, res)\n'
        '            cm = s.cost_model\n'
        '            sym = cm.symbolic(plan, t.stats)\n'
        '            rows.append({"workload": s.workload.name, "task": t.task_id, "plan": plan.describe(),\n'
        '                         "features": cm.features(plan, t.stats, sym), "sym_usd": sym.usd, "sym_ms": sym.latency_ms,\n'
        '                         "obs_usd": usd, "obs_ms": ms})\n'
        '    return rows\n'
        '\n'
        '\n'
        'def _errors(pred: list[float], obs: list[float]) -> dict:\n'
        '    p, o = np.maximum(np.array(pred), 1e-12), np.maximum(np.array(obs), 1e-12)\n'
        '    rel = np.abs(p - o) / o\n'
        '    q = np.maximum(p / o, o / p)   # q-error, the standard cardinality/cost metric\n'
        '    return {"mae_log10": float(np.mean(np.abs(np.log10(p) - np.log10(o)))),\n'
        '            "median_rel": float(np.median(rel)), "p90_rel": float(np.percentile(rel, 90)),\n'
        '            "median_q": float(np.median(q)), "p90_q": float(np.percentile(q, 90)), "n": len(o)}\n'
        '\n'
        '\n'
        'def exp_cost_model(settings: Settings, profile: str = "quick", workloads=WORKLOADS,\n'
        '                   progress: Progress = _noop) -> dict:\n'
        '    from ..core.cost_model import HybridCostModel\n'
        '    prof = PROFILES[profile]\n'
        '    data = []\n'
        '    sessions = {}\n'
        '    for wl in workloads:\n'
        '        progress(f"cost telemetry | {wl}")\n'
        '        s = _session(wl, settings)\n'
        '        sessions[wl] = s\n'
        '        data += collect_cost_telemetry(s, s.workload.tasks(**prof, seed=settings.sampling.task_seed))\n'
        '    results = []\n'
        '\n'
        '    def fit_eval(train, test, label, target):\n'
        '        cm = HybridCostModel(settings.llm.price_in_per_m, settings.llm.price_out_per_m,\n'
        '                             settings.budget.cpu_usd_per_hour)\n'
        '        ok = cm.fit([r["features"] for r in train], [r["obs_usd"] for r in train], [r["obs_ms"] for r in train],\n'
        '                    [r["sym_usd"] for r in train], [r["sym_ms"] for r in train], min_samples=8)\n'
        '        X = np.array([r["features"] for r in test])\n'
        '        obs = [r[f"obs_{target}"] for r in test]\n'
        '        sym = [r[f"sym_{target}"] for r in test]\n'
        '        res = {"split": label, "target": target, "symbolic": _errors(sym, obs)}\n'
        '        if ok:\n'
        '            rm, nm = (cm._res_usd, cm._nn_usd) if target == "usd" else (cm._res_ms, cm._nn_ms)\n'
        '            res["hybrid"] = _errors(list(np.array(sym) * np.exp(rm.predict(X))), obs)\n'
        '            res["neural"] = _errors(list(np.exp(nm.predict(X[:, :-2]))), obs)\n'
        '        return res\n'
        '\n'
        '    rng = random.Random(0)\n'
        '    for target in ("usd", "ms"):\n'
        '        # in-distribution: 5-fold CV over all telemetry\n'
        '        idx = list(range(len(data)))\n'
        '        rng.shuffle(idx)\n'
        '        folds = [idx[i::5] for i in range(5)]\n'
        '        for f in folds:\n'
        '            test = [data[i] for i in f]\n'
        '            train = [data[i] for i in idx if i not in set(f)]\n'
        '            r = fit_eval(train, test, "cv", target)\n'
        '            if "hybrid" not in r:\n'
        '                continue\n'
        '            results.append(r)\n'
        '        # zero-shot transfer: train on the other workloads only\n'
        '        for wl in workloads:\n'
        '            train = [r for r in data if r["workload"] != wl]\n'
        '            test = [r for r in data if r["workload"] == wl]\n'
        '            if train and test:\n'
        '                results.append(fit_eval(train, test, f"zero-shot:{wl}", target))\n'
        '    # aggregate CV folds\n'
        '    agg = []\n'
        '    for target in ("usd", "ms"):\n'
        '        for split in sorted({r["split"] for r in results}):\n'
        '            rs = [r for r in results if r["split"] == split and r["target"] == target]\n'
        '            if not rs:\n'
        '                continue\n'
        '            row = {"split": split, "target": target}\n'
        '            for model in ("symbolic", "neural", "hybrid"):\n'
        '                ms = [r[model] for r in rs if model in r]\n'
        '                if ms:\n'
        '                    row[model] = {k: float(np.mean([m[k] for m in ms])) for k in ms[0]}\n'
        '            agg.append(row)\n'
        '    scatter = [{"workload": r["workload"], "plan": r["plan"], "sym_usd": r["sym_usd"], "obs_usd": r["obs_usd"],\n'
        '                "sym_ms": r["sym_ms"], "obs_ms": r["obs_ms"]} for r in data]\n'
        '    return {"profile": profile, "summary": agg, "folds": results, "scatter": scatter, "n_samples": len(data)}\n'
        '\n'
        '\n'
        '# ====================================================================== E6\n'
        'def exp_latency(settings: Settings, profile: str = "quick", workloads=WORKLOADS,\n'
        '                progress: Progress = _noop) -> dict:\n'
        '    out = {"profile": profile, "ann_sweep": [], "probe": [], "validation_path": []}\n'
        '    for wl in workloads:\n'
        '        s = _session(wl, settings)\n'
        '        idx = s.workload.index\n'
        '        rng = np.random.default_rng(0)\n'
        '        qs = [q["question"] for q in s.workload.queries]\n'
        '        picks = rng.choice(len(qs), size=min(40, len(qs)), replace=False)\n'
        '        qv = s.workload.embedder.encode([qs[i] for i in picks])\n'
        '        for k in (5, 10, 20, 40):\n'
        '            exact = [set(idx.search(v, k, "flat")[0]) for v in qv]\n'
        '            for fam, knobs in (("hnsw", [16, 32, 64, 128]), ("ivf", [1, 2, 4, 8, 16]), ("flat", [0])):\n'
        '                for knob in knobs:\n'
        '                    times, recalls = [], []\n'
        '                    for v, ex in zip(qv, exact):\n'
        '                        pos, _, ms = idx.search(v, k, fam, ef_search=knob or 64, nprobe=knob or 8)\n'
        '                        times.append(ms)\n'
        '                        recalls.append(len(ex & set(pos)) / max(len(ex), 1))\n'
        '                    out["ann_sweep"].append({"workload": wl, "k": k, "family": fam, "knob": knob,\n'
        '                                             "ms": float(np.median(times)), "recall": float(np.mean(recalls))})\n'
        "        # filtered probe vs post-filter at this workload's predicate selectivity\n"
        '        from ..engine.store import render_where\n'
        '        for q in s.workload.queries[:30]:\n'
        '            where = render_where(s.workload.filter_template, q["fields"])\n'
        '            allowed, db_ms = s.workload.store.positions(where)\n'
        '            v = s.workload.embedder.encode_query(q["question"])\n'
        '            truth = set(idx.search(v, 10, "flat", allowed=allowed)[0])\n'
        '            for fam in ("hnsw", "ivf", "flat"):\n'
        '                pos, _, ms = idx.search(v, 10, fam, allowed=allowed)\n'
        '                pos2, _, ms2 = idx.search(v, 30, fam)\n'
        '                post = [p for p in pos2 if p in set(allowed)][:10]\n'
        '                out["probe"].append({"workload": wl, "family": fam, "sel": len(allowed) / idx.n,\n'
        '                                     "filtered_ms": ms + db_ms, "filtered_recall": len(truth & set(pos)) / max(len(truth), 1),\n'
        '                                     "post_ms": ms2 + db_ms, "post_recall": len(truth & set(post)) / max(len(truth), 1)})\n'
        '    base = load(settings, "baselines")\n'
        '    if base:\n'
        '        for r in base["summary"]:\n'
        '            out["validation_path"].append({k: r.get(k) for k in ("workload", "system", "db_ms", "ann_ms", "llm_ms",\n'
        '                                                                 "exec_ms", "opt_tokens", "wall_s")})\n'
        '    # LLM service time spent on the optimisation path, per task, from the baseline runs\n'
        '    if base:\n'
        '        for r in base["rows"]:\n'
        '            out.setdefault("llm_time", []).append({"workload": r["workload"], "system": r["system"],\n'
        '                                                   "task": r["task"], "opt_llm_s": r.get("opt_llm_s"),\n'
        '                                                   "opt_tokens": r.get("opt_tokens")})\n'
        '    return out\n'
        '\n'
        '\n'
        '# ====================================================================== E7\n'
        'def exp_ablation(settings: Settings, profile: str = "quick", workloads=WORKLOADS,\n'
        '                 budget: int | None = 20000, progress: Progress = _noop) -> dict:\n'
        '    prof = PROFILES[profile]\n'
        '    rows = []\n'
        '    st = settings.copy(**{"budget.optimization_tokens": budget})\n'
        '    for wl in workloads:\n'
        '        for system in ABLATIONS:\n'
        '            s = _session(wl, st)\n'
        '            # warm the residual model and memory with one pass so the learned parts exist\n'
        '            tasks = s.workload.tasks(**prof, seed=settings.sampling.task_seed)\n'
        '            rows += [r for r in run_stream(s, system, tasks, progress, passes=2) if r["pass"] == 2]\n'
        '    return {"profile": profile, "budget": budget, "rows": rows, "summary": summarise(rows)}\n'
        '\n'
        '\n'
        '# ====================================================================== E8\n'
        'def exp_transcripts(settings: Settings, limit: int = 12) -> dict:\n'
        '    base = load(settings, "baselines")\n'
        '    picks = []\n'
        '    if base:\n'
        '        for r in base["rows"]:\n'
        '            if r["system"] != "plannerllm":\n'
        '                continue\n'
        '            for a in r["audit"]:\n'
        '                if a["source"] in ("validation", "memory", "gate-skip"):\n'
        '                    picks.append({"workload": r["workload"], "task": r["task"], **a})\n'
        '    accepted = [p for p in picks if p["verdict"] == "prefer"]\n'
        '    skipped = [p for p in picks if p["source"] == "gate-skip"]\n'
        '    reused = [p for p in picks if p["source"] == "memory"]\n'
        '    chosen = accepted[: limit // 2] + skipped[: limit // 4] + reused[: limit // 4]\n'
        '    return {"transcripts": chosen, "counts": {"accepted": len(accepted), "gate_skipped": len(skipped),\n'
        '                                              "memory_reused": len(reused)}}\n'
        '\n'
        '\n'
        '# ====================================================================== E9\n'
        'def exp_judge(settings: Settings, profile: str = "quick", workloads=WORKLOADS, max_validations: int = 20,\n'
        '              progress: Progress = _noop) -> dict:\n'
        '    """Independent-judge evaluation of this seed\'s baseline and ablation runs (see indep_judge.py)."""\n'
        '    from ..data.workloads import load_workload\n'
        '    from .indep_judge import IndependentJudge, cohen_kappa\n'
        '    judge = IndependentJudge(settings)\n'
        '    out = {"profile": profile, "judge_model": judge.model, "graded": [], "summary": [], "validator": [],\n'
        '           "validator_summary": []}\n'
        '    sources = {"baselines": load(settings, "baselines"), "ablation": load(settings, "ablation")}\n'
        '    for wl in [w for w in workloads if w != "spider"]:      # Spider is scored by execution, no judge needed\n'
        '        W = load_workload(wl, settings)\n'
        '        qmap = {q["qid"]: q for q in W.queries}\n'
        '        cache: dict[tuple, float | None] = {}\n'
        '        for src, payload in sources.items():\n'
        '            if not payload:\n'
        '                continue\n'
        '            for r in payload["rows"]:\n'
        '                if r["workload"] != wl:\n'
        '                    continue\n'
        '                scores = []\n'
        '                for pq in r["per_query"]:\n'
        '                    q = qmap.get(pq["qid"])\n'
        '                    if q is None or "answer" not in pq:\n'
        '                        continue\n'
        '                    key = (pq["qid"], " ".join(str(pq["answer"]).split()))\n'
        '                    if key not in cache:\n'
        '                        progress(f"judge grade | {wl} | {len(cache)}")\n'
        '                        cache[key] = judge.grade(W.task_text, q["question"], q["answers"], pq["answer"])\n'
        '                    pq["judge"] = cache[key]\n'
        '                    if cache[key] is not None:\n'
        '                        scores.append(cache[key])\n'
        '                        out["graded"].append({"workload": wl, "source": src, "system": r["system"],\n'
        '                                              "task": r["task"], "qid": pq["qid"], "judge": cache[key],\n'
        '                                              "metric": pq.get("accuracy"), "f1": pq.get("f1"),\n'
        '                                              "rouge_l": pq.get("rouge_l")})\n'
        '                r["judge_acc"] = float(np.mean(scores)) if scores else None\n'
        '        # validator agreement: re-judge the items the validation agent compared\n'
        '        rng = random.Random(f"judge:{wl}:{settings.sampling.task_seed}")\n'
        '        vals = []\n'
        '        for r in (sources["baselines"] or {}).get("rows", []):\n'
        '            if r["workload"] != wl or r["system"] not in ("plannerllm", "docetl"):\n'
        '                continue\n'
        '            for a in r["audit"]:\n'
        '                v = a.get("validation") or {}\n'
        '                if a.get("source") in ("validation", "tournament") and v.get("items"):\n'
        '                    vals.append((r["system"], r["task"], a))\n'
        '        rng.shuffle(vals)\n'
        '        for system, task, a in vals[:max_validations]:\n'
        '            wins = {"candidate": 0, "incumbent": 0, "tie": 0}\n'
        '            for it in a["validation"]["items"]:\n'
        '                q = qmap.get(it["qid"])\n'
        '                res = judge.compare(W.task_text, q["question"] if q else it["question"], it["candidate"],\n'
        '                                    it["incumbent"], key=f"{task}:{a[\'plan\']}:{it[\'qid\']}")\n'
        '                if res:\n'
        '                    wins[res] += 1\n'
        '            progress(f"judge compare | {wl} | {a[\'directive\']}")\n'
        '            out["validator"].append({"workload": wl, "system": system, "task": task, "directive": a["directive"],\n'
        '                                     "validator_prefers": a["verdict"] == "prefer",\n'
        '                                     "judge_prefers": wins["candidate"] > wins["incumbent"], **wins})\n'
        '    # summaries\n'
        '    for src, payload in sources.items():\n'
        '        if not payload:\n'
        '            continue\n'
        '        groups: dict[tuple, list] = {}\n'
        '        for r in payload["rows"]:\n'
        '            if r.get("judge_acc") is not None:\n'
        '                groups.setdefault((r["workload"], r["system"]), []).append(r["judge_acc"])\n'
        '        for (wl, system), v in groups.items():\n'
        '            out["summary"].append({"source": src, "workload": wl, "system": system, "judge_acc": float(np.mean(v)),\n'
        '                                   "n_tasks": len(v)})\n'
        '    for wl in sorted({v["workload"] for v in out["validator"]}):\n'
        '        vs = [v for v in out["validator"] if v["workload"] == wl]\n'
        '        a_, b_ = [v["validator_prefers"] for v in vs], [v["judge_prefers"] for v in vs]\n'
        '        out["validator_summary"].append({"workload": wl, "n": len(vs),\n'
        '                                         "agreement": float(np.mean([x == y for x, y in zip(a_, b_)])),\n'
        '                                         "kappa": cohen_kappa(a_, b_),\n'
        '                                         "validator_prefer_rate": float(np.mean(a_)),\n'
        '                                         "judge_prefer_rate": float(np.mean(b_))})\n'
        '    allv = out["validator"]\n'
        '    if allv:\n'
        '        a_, b_ = [v["validator_prefers"] for v in allv], [v["judge_prefers"] for v in allv]\n'
        '        out["validator_summary"].append({"workload": "all", "n": len(allv),\n'
        '                                         "agreement": float(np.mean([x == y for x, y in zip(a_, b_)])),\n'
        '                                         "kappa": cohen_kappa(a_, b_),\n'
        '                                         "validator_prefer_rate": float(np.mean(a_)),\n'
        '                                         "judge_prefer_rate": float(np.mean(b_))})\n'
        '    out["judge_usage"] = judge.usage()\n'
        '    return out\n'
        '\n'
        '\n'
        'EXPERIMENTS = {\n'
        '    "baselines": exp_baselines, "pareto": exp_pareto, "gate": exp_gate, "memory": exp_memory,\n'
        '    "cost_model": exp_cost_model, "latency": exp_latency, "ablation": exp_ablation, "judge": exp_judge,\n'
        '}\n'
        '\n'
        '\n'
        'def run_experiment(name: str, settings: Settings, profile: str = "quick", workloads=None,\n'
        '                   progress: Progress = print) -> Path:\n'
        '    t0 = time.time()\n'
        '    if name == "transcripts":\n'
        '        payload = exp_transcripts(settings)\n'
        '    else:\n'
        '        payload = EXPERIMENTS[name](settings, profile=profile, workloads=workloads or WORKLOADS, progress=progress)\n'
        '    payload["elapsed_s"] = time.time() - t0\n'
        '    return save(settings, name, payload)\n'
        '\n'
        '\n'
        'ALL = ["baselines", "pareto", "gate", "memory", "cost_model", "latency", "ablation", "transcripts", "judge"]\n'
    ),
})
run_selected(['transcripts', 'judge'])

plannerllm/eval/indep_judge.py
plannerllm/eval/experiments.py
transcripts | seed 0 | already executed | reproduced_runs\recorded_groq\openai_gpt-oss-20b_seed0\transcripts.json
transcripts | seed 1 | already executed | reproduced_runs\recorded_groq\openai_gpt-oss-20b_seed1\transcripts.json
transcripts | seed 2 | already executed | reproduced_runs\recorded_groq\openai_gpt-oss-20b_seed2\transcripts.json
judge | seed 0 | already executed | reproduced_runs\recorded_groq\openai_gpt-oss-20b_seed0\judge.json
judge | seed 1 | already executed | reproduced_runs\recorded_groq\openai_gpt-oss-20b_seed1\judge.json
judge | seed 2 | 71 progress entries | reproduced_runs\recorded_groq\openai_gpt-oss-20b_seed2\judge.json
   judge compare | cuad | POST_FILTER
   judge compare | cuad | SWAP_INDEX
   judge compare | cuad | FILTERED_PROBE


"""Independent judge: a second LLM, from a different model family, used only for evaluation.

The optimizer's validation agent and the pipeline's answer generator are the same model
(gpt-oss-20b), so a verdict could share the generator's blind spots.  This module asks a
model from another family (Qwen, by default) two questions the optimizer never sees:

* ``grade``    - is a held-out answer correct with respect to the human reference answers?
                 This gives an accuracy that does not depend on word overlap.
* ``compare``  - given the same two outputs the validation agent compared, which one is better?
                 Agreement with the validation agent's verdict measures judge bias.

Both calls go through the normal client, so they are cached and rate limited, and their
tokens are kept in a separate ledger (they are evaluation cost, not optimisation cost).
"""
from __future__ import annotations

import hashlib
import json

from ..llm.base import TokenLedger
from ..llm.cache import ResponseCache
from ..llm.client import GroqClient

SYSTEM = ("You are an impartial grader of question-answering and information-extraction systems. "
          "You compare outputs carefully and reply only with the JSON object you are asked for.")

GRADE = """Task of the system: {task}

Question: {question}

Reference answer(s) written by people:
{refs}

System answer:
{answer}

Grade the system answer against the references.
- "correct": it states the information of at least one reference answer. Wording may differ and extra correct detail is fine.
- "partial": it contains only part of that information, or is too vague to be fully correct.
- "incorrect": it is wrong, unrelated, or says the information is missing when a reference exists.
A reference of NONE means the source has no such information; then an answer of NONE (or "not found") is correct and any other answer is incorrect.
Reply with JSON: {{"verdict": "correct" | "partial" | "incorrect"}}"""

COMPARE = """Task of the system: {task}

Question: {question}

Output A:
{a}

Output B:
{b}

Which output accomplishes the task better for this question? Judge correctness, completeness and
faithfulness to the task; do not prefer an output only because it is longer.
Reply with JSON: {{"better": "A" | "B" | "tie"}}"""

SCORES = {"correct": 1.0, "partial": 0.5, "incorrect": 0.0}


class IndependentJudge:
    def __init__(self, settings) -> None:
        cfg = settings.llm
        self.model = cfg.judge_model
        self.ledger = TokenLedger(cfg.price_in_per_m, cfg.price_out_per_m, None)
        self.client = GroqClient(self.ledger, ResponseCache(cfg.cache_path), model=self.model,
                                 base_url=cfg.base_url, temperature=0.0, rpm=cfg.requests_per_minute,
                                 tpm=cfg.tokens_per_minute, max_retries=cfg.max_retries,
                                 timeout_s=cfg.timeout_s, extra_body={"reasoning_effort": "none"},
                                 reasoning_tokens=0)

    def grade(self, task: str, question: str, refs: list[str], answer: str) -> float | None:
        refs_txt = "\n".join(f"- {r[:600]}" for r in refs[:4]) or "- NONE"
        msg = GRADE.format(task=task, question=question, refs=refs_txt, answer=(answer or "(empty)")[:800])
        r = self.client.complete(msg, purpose="judge", task="judge_grade", system=SYSTEM, max_tokens=60,
                                 json_mode=True)
        v = str((r.json({}) or {}).get("verdict", "")).strip().lower()
        return SCORES.get(v)

    def compare(self, task: str, question: str, candidate: str, incumbent: str, key: str) -> str | None:
        """Returns 'candidate', 'incumbent' or 'tie'.  The A/B position is decided by a hash of ``key``
        so that position bias averages out over items."""
        swap = int(hashlib.sha256(key.encode()).hexdigest(), 16) % 2 == 1
        a, b = (incumbent, candidate) if swap else (candidate, incumbent)
        msg = COMPARE.format(task=task, ques

"""Experiment suite for Section 10 (one function per subsection).

Every experiment writes ``runs/<backend>/<experiment>.json`` so the dashboard
and ``reporting.py`` can render it without re-running anything.  Runs are
resumable for free: every LLM response is cached on disk, so re-running an
interrupted experiment replays finished calls instantly.

    E1 baselines        10.1 / 10.2 table: accuracy, spend, latency per system
    E2 pareto           10.2 accuracy vs optimisation-token budget, theta sweep
    E3 gate             10.3 gate precision / recall / calibration (shadow mode)
    E4 memory           10.4 hit rate, reuse accuracy, decay, perturbed shapes
    E5 cost_model       10.5 calibration error: symbolic / neural / hybrid, zero-shot
    E6 latency          10.6 wall-clock decomposition, ANN family sweep
    E7 ablation         10.7 -gate, -memory, symbolic-only, neural-only
    E8 transcripts      10.8 validation transcripts for accepted rewrites
"""
from __future__ import annotations

import json
import math
import random
import time
from pathlib import Path
from typing import Callable

import numpy as np

from ..core.directives import DirectiveContext
from ..core.memory import PlanMemory
from ..core.optimizer import ABLATIONS, SYSTEMS, make_system
from ..core.session import Session, build_session, mean_observed
from ..settings import RUNS_DIR, Settings

PROFILES = {   # groups per workload, optimisation questions, held-out questions
    "smoke": {"n_tasks": 1, "n_opt": 5, "n_test": 5},
    "quick": {"n_tasks": 2, "n_opt": 6, "n_test": 6},
    "full": {"n_tasks": 4, "n_opt": 8, "n_test": 10},
}
WORKLOADS = ["msmarco", "cuad", "spider"]
Progress = Callable[[str], None]


def _noop(_: str) -> None:
    pass


def out_dir(settings: Settings) -> Path:
    tag = "sim" if settings.llm.backend == "sim" else settings.llm.model.replace("/", "_")
    d = RUNS_DIR / f"{tag}_seed{settings.sampling.task_seed}"
    d.mkdir(parents=True, exist_ok=True)
    return d


def save(settings: Settings, name: str, payload: dict) -> Path:
    payload = {"experiment": name, "backend": settings.llm.backend,
               "model": settings.llm.model if settings.llm.backend != "sim" else "simulated-extractive",
               "created": time.strftime("%Y-%m-%d %H:%M:%S"), "seed": settings.sampling.task_seed,
               "sampling_seed": settings.sampling.seed, **payload}
    path = out_dir(settings) / f"{name}.json"
    path.write_text(json.dumps(payload, indent=1, default=_json_default), encoding="utf-8")
    return path


def load(settings: Settings, name: str) -> dict | None:
    path = out_dir(settings) / f"{name}.json"
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None


def _json_default(o):
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    if hasattr(o, "to_dict"):
        return o.to_dict()
    return str(o)


class LogicalClock:
    """Days advance per optimised task, so decay is reproducible."""

    def __init__(self, days_per_task: float = 1.0) -> None:
        self.day, self.step = 0.0, days_per_task

    def __call__(self) -> float:
        return self.day

    def tick(self) -> None:
        self.day += self.step


def _session(wl: str, settings: Settings, clock: LogicalClock | None = None, **kw) -> Session:
    clock = clock or LogicalClock()
    s = build_session(wl, settings, clock=clock, **kw)
    s.clock = clock   # type: ignore[attr-defined]
    return s


def evaluate_plan(s: Session, plan, queries: list[dict], log_telemetry: bool = True) -> dict:
    results = [s.executor.run(plan, q, purpose="execute") for q in queries]
    sc = s.oracle.score(queries, results)
    usd, ms = mean_observed(s, results)
    br = [r.breakdown() for r in results]
    sc.update(exec_usd=usd, exec_ms=ms,
              db_ms=float(np.mean([b["db_ms"] for b in br])), ann_ms=float(np.mean([b["ann_ms"] for b in br])),
    

## Aggregate the freshly executed runs and check against the report artifacts

In [8]:
tag = settings_for(0).llm.model.replace("/", "_")
aggregate_path = build(tag)
print("Fresh analysis:", aggregate_path)
for seed in SEEDS:
    fresh = E.load(settings_for(seed), "baselines")
    original = json.loads((ROOT / "runs" / f"{tag}_seed{seed}" / "baselines.json").read_text(encoding="utf-8"))
    def outcomes(result):
        return {(r["workload"], r["system"], q["qid"]): (q["accuracy"], q.get("answer"))
                for r in result["rows"] for q in r["per_query"]}
    a, b = outcomes(fresh), outcomes(original)
    common = a.keys() & b.keys()
    print(f"Seed {seed}: {sum(a[k] == b[k] for k in common)}/{len(common)} matched held-out scores and recorded answers")

for seed in SEEDS:
    folder = E.out_dir(settings_for(seed))
    complete = [name for name in E.ALL if (folder / f"{name}.json").exists()]
    missing = [name for name in E.ALL if name not in complete]
    print(f"Seed {seed} | completed: {', '.join(complete)} | incomplete: {', '.join(missing) or 'none'}")

Fresh analysis: C:\Users\aayus\OneDrive\Desktop\LLMPlanner\Evaluation_Submission\reproduced_runs\recorded_groq\openai_gpt-oss-20b_aggregate\aggregate.json
Seed 0: 180/180 matched held-out scores and recorded answers
Seed 1: 180/180 matched held-out scores and recorded answers
Seed 2: 177/180 matched held-out scores and recorded answers
Seed 0 | completed: baselines, pareto, gate, memory, cost_model, latency, ablation, transcripts, judge | incomplete: none
Seed 1 | completed: baselines, pareto, gate, memory, cost_model, latency, ablation, transcripts, judge | incomplete: none
Seed 2 | completed: baselines, pareto, gate, memory, cost_model, latency, ablation, transcripts, judge | incomplete: none
